In [ ]:

"""
================================================================================
BEACON INDOOR LOCALIZATION — GRAPH ATTENTION + kNN HYBRID + KALMAN SMOOTHING
                            + SELF-SUPERVISED PRETRAINING (Temporal Branch)
Leakage-free, train-only statistics, session/fingerprint split.

MERGED FROM:
  • Code 1: GAT dual-branch (temporal transformer + beacon graph attention)
            + kNN hybrid + Kalman smoothing + real-time inference
  • Code 2: Self-supervised masked pretraining on unlabeled trajectory data
            (temporal branch only — graph branch trained on labeled data only)

PRETRAINING STRATEGY:
  • Only the TEMPORAL branch is pretrained (masked autoencoder on raw sequences)
  • The GRAPH branch is always trained from scratch on labeled data
  • Reason: trajectory data is smaller than labeled data → pretraining the graph
    branch on it would likely overfit; temporal branch is larger & benefits more

REAL-TIME UPDATE:
  • Causal forward-only Kalman filter for streaming inference
  • RealTimeLocalizer + StreamingLocalizer classes for live deployment
  • Per-trajectory MAD outlier rejection
  • Fast prediction path: TTA & kNN optional/disabled by default
================================================================================
"""

import os
import re
import json
import math
import shutil
import pickle
import warnings
from collections import defaultdict

import chardet
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model, optimizers, callbacks
from tensorflow.keras.utils import Sequence
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
tf.keras.backend.clear_session()
tf.keras.backend.set_floatx('float32')

from google.colab import drive
drive.mount('/content/drive')

# ==============================================================================
# CONFIG
# ==============================================================================
BASE_DIR         = r'/content/drive/MyDrive/beacons_chaos'
CACHE_MODELS_DIR = os.path.join(BASE_DIR, 'train_cache_gat')
EVAL_FILE_PATH   = os.path.join(BASE_DIR, 'eval_df_gat.csv')
NORM_FILE_PATH   = os.path.join(BASE_DIR, 'normalization_gat.json')

# --- Pretraining paths (NEW) ---
TRAJECTORY_DATA_DIR      = r'/content/drive/MyDrive/beacons_chaos/trajectory_data'
PRETRAINED_WEIGHTS_PATH  = os.path.join(CACHE_MODELS_DIR, 'temporal_pretrained.weights.h5')
PRETRAIN_EPOCHS          = 50
PRETRAIN_MASK_RATIO      = 0.15

os.makedirs(CACHE_MODELS_DIR, exist_ok=True)

BATCH_SIZE    = 128
VAL_SPLIT     = 0.15
TEST_SPLIT    = 0.15
WINDOW_SIZE   = 5
WINDOW_STRIDE = 1

N_FLOORS      = 2
FLOOR_CLASSES     = {3: 0, 4: 1}
FLOOR_CLASSES_INV = {0: 3, 1: 4}

W_Y     = 1.0
W_FLOOR = 0.05

Y_MIN, Y_MAX = 0.0, 93.0
MIN_BEACONS_PER_WINDOW = 3

# Beacon graph parameters
NUM_BEACONS    = 7      # 0 = padding/unknown, 1..6 = real beacons
EMBED_DIM      = 16
N_BEACON_FEATS = 6      # presence, mean_rssi_norm, std_rssi_norm,
                        # mean_rssi_distance, count_ratio, mean_motion_magnitude

# kNN / Kalman defaults
KNN_K       = 5
KALMAN_Q    = 0.1
KALMAN_R    = 2.0

# Optional: only if known from architectural drawings, NOT estimated from data
BEACON_Y_POS = {}


# ==============================================================================
# BEACON MAPPING
# ==============================================================================
BEACON_MAP = {
    'fda50693-a4e2-4fb1-afcf-c6eb07647825:1:1':         1,
    'fda50693-a4e2-4fb1-afcf-c6eb07647825:0:2':         2,
    'aaaaaaaa-aaaa-aaaa-aaaa-aaaaaaaaaaaa:0:3':          3,
    'fda50693-a4e2-4fb1-afcf-c6eb07647825:10065:26049': 4,
    'fda50693-a4e2-4fb1-afcf-c6eb07647825:1:5':         5,
    'fda50693-a4e2-4fb1-afcf-c6eb07647825:1:6':         6,
}
BEACON_FLOOR = {1: 3, 2: 3, 3: 3, 4: 4, 5: 4, 6: 4}


# ==============================================================================
# HELPERS
# ==============================================================================
def pkl_save(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)

def pkl_load(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

def json_save(obj, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=4)

def json_load(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def clear_stale_cache(expected_features):
    marker = os.path.join(CACHE_MODELS_DIR, '.feature_count')
    if os.path.exists(marker):
        with open(marker) as f:
            cached = int(f.read().strip())
        if cached != expected_features:
            print(f"  Cache stale ({cached} → {expected_features}). Clearing...")
            shutil.rmtree(CACHE_MODELS_DIR)
            os.makedirs(CACHE_MODELS_DIR, exist_ok=True)
    with open(marker, 'w') as f:
        f.write(str(expected_features))


# ==============================================================================
# DATA LOADING
# ==============================================================================
def inspect_raw_file(filepath, n_bytes=500):
    if not os.path.exists(filepath):
        return None, None
    with open(filepath, 'rb') as f:
        raw = f.read(n_bytes)
    enc = chardet.detect(raw)['encoding'] or 'utf-8'
    with open(filepath, 'r', encoding=enc, errors='replace') as f:
        first = f.readline()
    seps = {s: first.count(s) for s in ['\t', ',', ';', '|']}
    sep = max(seps, key=seps.get) if max(seps.values()) > 0 else '\t'
    return enc, sep

def robust_load_csv(filepath):
    encoding, sep = inspect_raw_file(filepath)
    df = None
    for kw in [
        {'encoding': encoding, 'sep': sep},
        {'encoding': 'utf-8-sig', 'sep': sep},
        {'encoding': 'utf-8-sig', 'sep': sep, 'engine': 'python'},
        {'encoding': 'latin-1', 'sep': sep},
    ]:
        try:
            df = pd.read_csv(filepath, **kw)
            print(f"  Loaded {len(df)} rows × {len(df.columns)} cols")
            break
        except Exception:
            continue
    if df is None:
        raise RuntimeError(f"Cannot load {filepath}")

    def clean(c):
        c = str(c).replace('\ufeff', '').strip()
        return ''.join(ch for ch in c if ch.isprintable() or ch.isspace()).strip()

    df.columns = [clean(c) for c in df.columns]

    seen, new_cols = {}, []
    for c in df.columns:
        seen[c] = seen.get(c, 0) + 1
        new_cols.append(c if seen[c] == 1 else f"{c}_dup{seen[c]}")
    df.columns = new_cols

        # Remove any beaconId column before alias remapping.
    # beaconId will be recomputed later from beaconUid.
    for col in list(df.columns):
        if col.lower().replace("_", "").replace(" ", "") == "beaconid":
            df.drop(columns=[col], inplace=True)
            print(f"  Dropped column: {col}")

    remap = {}
    for c in df.columns:
        cl = c.lower().replace(' ', '').replace('_', '')
        if cl in ('fingerprintid', 'fingerprint', 'fprintid'):
            remap[c] = 'fingerprintId'
        elif cl in ('sessionid', 'session'):
            remap[c] = 'sessionId'
        elif cl in ('floorlevel', 'floor', 'floorlvl'):
            remap[c] = 'floorLevel'
        elif cl in ('windowindex', 'windowidx', 'winindex'):
            remap[c] = 'windowIndex'
        elif cl in ('beaconuid', 'beaconid', 'beaconuuid'):
            remap[c] = 'beaconUid'
        elif cl in ('capturedat', 'timestamp', 'time', 'datetime'):
            remap[c] = 'capturedAt'
    if remap:
        df.rename(columns=remap, inplace=True)
    return df

def load_all_datasets(data_dir='./data'):
    file_mapping = {
        'adhamFloor3.csv': {'person': 'adham', 'floor': 3},
        'adhamFloor4.csv': {'person': 'adham', 'floor': 4},
        'baselFloor3.csv': {'person': 'basel', 'floor': 3},
        'baselFloor4.csv': {'person': 'basel', 'floor': 4},
        'remonFloor3.csv': {'person': 'remon', 'floor': 3},
        'remonFloor4.csv': {'person': 'remon', 'floor': 4},
    }
    all_dfs = []
    for fname, info in file_mapping.items():
        fpath = os.path.join(data_dir, fname)
        if not os.path.exists(fpath):
            print(f"  WARNING: {fname} not found")
            continue
        print(f"\n  {fname}")
        df = robust_load_csv(fpath)
        df['person'] = info['person']
        df['floor'] = info['floor']
        df['session_name'] = f"{info['person']}_floor{info['floor']}"
        all_dfs.append(df)

    if not all_dfs:
        raise RuntimeError("No input files found.")

    combined = pd.concat(all_dfs, ignore_index=True)
    print(f"\nCOMBINED: {len(combined)} rows")
    return combined


# ==============================================================================
# TRAJECTORY DATA LOADING (NEW — from Code 2)
# ==============================================================================
def load_trajectory_dataset(traj_dir=TRAJECTORY_DATA_DIR):
    """
    Load unlabeled / weakly-labeled trajectory CSVs for self-supervised pretraining.
    The trajectory data has the same columns as labeled data (including x, y, floor)
    but we deliberately IGNORE x and y during pretraining — the model only learns
    to reconstruct the sensor/RSSI sequence (masked autoencoder objective).
    """
    if not os.path.isdir(traj_dir):
        print(f"  WARNING: trajectory directory not found: {traj_dir}")
        return None

    all_dfs = []
    for fname in os.listdir(traj_dir):
        if not fname.endswith('.csv'):
            continue
        fpath = os.path.join(traj_dir, fname)
        print(f"\n  [trajectory] {fname}")
        try:
            df = robust_load_csv(fpath)
        except Exception as e:
            print(f"    SKIP — {e}")
            continue

        if 'session_name' not in df.columns:
            df['session_name'] = os.path.splitext(fname)[0]

        all_dfs.append(df)

    if not all_dfs:
        print("  WARNING: no trajectory CSVs loaded.")
        return None

    combined = pd.concat(all_dfs, ignore_index=True)
    print(f"\n  TRAJECTORY COMBINED: {len(combined)} rows")
    return combined


# ==============================================================================
# CLEANING
# ==============================================================================
def minimal_clean(df):
    df = df.copy()

    for col in ['pressure', 'relativeAltitude']:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)

    df['beaconId'] = df['beaconUid'].map(BEACON_MAP)
    unknown = df['beaconId'].isna()
    n_unknown = unknown.sum()
    if n_unknown:
        print(f"  WARNING: dropping {n_unknown} rows with unknown beacons")
        df = df[~unknown].copy()

    df['beaconId'] = df['beaconId'].astype(int)

    df['capturedAt'] = pd.to_datetime(df['capturedAt'], errors='coerce')
    df = df.dropna(subset=['capturedAt'])

    df.sort_values(['session_name', 'fingerprintId', 'windowIndex', 'capturedAt'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df


def minimal_clean_trajectory(df):
    """
    Same as minimal_clean but tolerates missing x/y/floor — trajectory files
    may have them but we never use them as labels in pretraining.
    Unknown beacons are mapped dynamically (not dropped) because trajectory
    sessions may visit areas not covered by the labeled set.
    """
    df = df.copy()

    for col in ['pressure', 'relativeAltitude']:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)

    # Map known beacons; dynamically assign IDs for unknowns
    df['beaconId'] = df['beaconUid'].map(BEACON_MAP)
    unknown_uids = df.loc[df['beaconId'].isna(), 'beaconUid'].dropna().unique()
    if len(unknown_uids):
        nxt = max(BEACON_MAP.values()) + 1
        for uid in unknown_uids:
            BEACON_MAP[uid] = nxt
            nxt += 1
        df['beaconId'] = df['beaconUid'].map(BEACON_MAP)

    df['capturedAt'] = pd.to_datetime(df['capturedAt'], errors='coerce')
    df = df.dropna(subset=['capturedAt'])

    if 'floorLevel' not in df.columns:
        df['floorLevel'] = 3   # default floor if missing

    if 'session_name' not in df.columns:
        df['session_name'] = 'traj'
    if 'fingerprintId' not in df.columns:
        df['fingerprintId'] = 0
    if 'windowIndex' not in df.columns:
        df['windowIndex'] = 0

    sort_cols = [c for c in ['session_name', 'fingerprintId', 'windowIndex', 'capturedAt']
                 if c in df.columns]
    df.sort_values(sort_cols, inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df


# ==============================================================================
# PREPROCESSING — TRAIN-ONLY IMPUTATION
# ==============================================================================
def preprocess_data(df, sensor_medians=None):
    df = df.copy()

    df['rel_time'] = df.groupby(['session_name', 'fingerprintId', 'windowIndex'])['capturedAt'].transform(
        lambda x: (x - x.min()).dt.total_seconds()
    )
    df['window_row_idx'] = df.groupby(['session_name', 'fingerprintId', 'windowIndex']).cumcount()

    df.sort_values(['session_name', 'fingerprintId', 'windowIndex', 'capturedAt'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    impute_cols = [
        'gyroX', 'gyroY', 'gyroZ', 'accelX', 'accelY', 'accelZ',
        'userAccelX', 'userAccelY', 'userAccelZ',
        'magX', 'magY', 'magZ', 'pitch', 'roll', 'yaw'
    ]

    for col in impute_cols:
        if col not in df.columns:
            continue
        if df[col].isna().any():
            df[col] = df.groupby(['session_name', 'fingerprintId', 'windowIndex'])[col].transform(
                lambda x: x.ffill().bfill()
            )
            if sensor_medians is not None and col in sensor_medians:
                df[col] = df[col].fillna(sensor_medians[col])
            else:
                df[col] = df[col].fillna(0)

    df['floor_class'] = df['floorLevel'].map(FLOOR_CLASSES).fillna(0).astype(int)
    return df


# ==============================================================================
# FEATURE ENGINEERING — NO FINGERPRINT MEMORIZATION
# ==============================================================================
BASE_SENSOR_COLS = [
    'beaconId', 'rssi', 'rel_time',
    'gyroX', 'gyroY', 'gyroZ', 'accelX', 'accelY', 'accelZ',
    'userAccelX', 'userAccelY', 'userAccelZ',
    'magX', 'magY', 'magZ', 'pitch', 'roll', 'yaw',
    'rssi_norm', 'rssi_inv', 'rssi_distance',
    'beacon_floor_match', 'window_progress',
    'rssi_diff_from_beacon_median',
    'beacon_signal_rank',
    'window_rssi_mean', 'window_rssi_std',
    'min_beacon_distance', 'beacon_weight',
    'rssi_vs_floor_median',
    'beacon_on_this_floor',
    'dominant_beacon_id',
    'rssi_diff_to_max', 'rssi_diff_to_2nd_max',
    'beacon_count_in_window',
    'motion_magnitude', 'motion_yaw_rate',
    'beacon_y_pos',
    'unique_beacon_count',
    'weighted_beacon_y',
    'strongest_beacon_y',
]

FEATURE_COLS = None
N_FEATURES = None

def engineer_features(df, beacon_median_map=None, floor_median_map=None):
    df = df.copy()

    df['rssi_norm'] = (df['rssi'] + 100) / 60
    df['rssi_inv'] = -df['rssi']

    TX_POWER, PATH_LOSS = -59, 2.0
    df['rssi_distance'] = 10 ** ((TX_POWER - df['rssi']) / (10 * PATH_LOSS))
    df['rssi_distance'] = df['rssi_distance'].clip(upper=500)

    df['beacon_floor'] = df['beaconId'].map(BEACON_FLOOR).fillna(0).astype(int)
    df['beacon_on_this_floor'] = (df['beacon_floor'] == df['floorLevel']).astype(float)
    df['beacon_floor_match'] = df['beacon_on_this_floor']

    df['window_progress'] = df.groupby(['session_name', 'fingerprintId', 'windowIndex'])['window_row_idx'].transform(
        lambda x: x / max(int(x.max()), 1)
    )

    group_cols = ['session_name', 'fingerprintId', 'windowIndex']

    if beacon_median_map is not None:
        df['beacon_median_rssi'] = df['beaconId'].map(beacon_median_map).fillna(df['rssi'])
        df['rssi_diff_from_beacon_median'] = df['rssi'] - df['beacon_median_rssi']
        df.drop(columns=['beacon_median_rssi'], inplace=True)
    else:
        df['rssi_diff_from_beacon_median'] = 0.0

    df['beacon_signal_rank'] = df.groupby(
        ['session_name', 'fingerprintId', 'windowIndex', 'capturedAt']
    )['rssi'].rank(ascending=False, method='dense')

    df['window_rssi_mean'] = df.groupby(group_cols)['rssi'].transform('mean')
    df['window_rssi_std'] = df.groupby(group_cols)['rssi'].transform('std').fillna(0)

    df['min_beacon_distance'] = df.groupby(
        ['session_name', 'fingerprintId', 'windowIndex', 'capturedAt']
    )['rssi_distance'].transform('min')

    df['beacon_weight'] = 1.0 / (df['rssi_distance'] + 0.1)

    if floor_median_map is not None:
        df['floor_beacon_median_rssi'] = df.apply(
            lambda row: floor_median_map.get((row['beaconId'], row['floorLevel']), row['rssi']),
            axis=1
        )
        df['rssi_vs_floor_median'] = df['rssi'] - df['floor_beacon_median_rssi']
        df.drop(columns=['floor_beacon_median_rssi'], inplace=True)
    else:
        df['rssi_vs_floor_median'] = 0.0

    group_time = ['session_name', 'fingerprintId', 'windowIndex', 'capturedAt']
    idxmax = df.groupby(group_time)['rssi'].idxmax()
    dominant_map = df.loc[idxmax].set_index(group_time)['beaconId']
    df['dominant_beacon_id'] = df.set_index(group_time).index.map(dominant_map).fillna(0).astype(int)

    max_rssi_per_group = df.groupby(group_time)['rssi'].transform('max')
    df['rssi_diff_to_max'] = df['rssi'] - max_rssi_per_group

    df_sorted = df.sort_values(group_time + ['rssi'], ascending=[True, True, True, True, False])
    df_sorted['rssi_rank'] = df_sorted.groupby(group_time).cumcount()
    second_max_map = df_sorted[df_sorted['rssi_rank'] == 1].set_index(group_time)['rssi']
    df['second_max_rssi'] = df.set_index(group_time).index.map(second_max_map)
    df['second_max_rssi'] = df['second_max_rssi'].fillna(max_rssi_per_group)
    df['rssi_diff_to_2nd_max'] = df['rssi'] - df['second_max_rssi']
    df.drop(columns=['second_max_rssi'], inplace=True)

    df['beacon_count_in_window'] = df.groupby(group_cols)['beaconId'].transform('count')
    df['motion_magnitude'] = np.sqrt(df['accelX']**2 + df['accelY']**2 + df['accelZ']**2)
    df['motion_yaw_rate'] = df['gyroZ'].abs()

    if BEACON_Y_POS:
        df['beacon_y_pos'] = df['beaconId'].map(BEACON_Y_POS).fillna(-1)
    else:
        df['beacon_y_pos'] = -1

    df['unique_beacon_count'] = df.groupby(group_cols)['beaconId'].transform('nunique')

    if BEACON_Y_POS:
        df['rssi_lin'] = 10 ** (df['rssi'] / 10.0)
        df['weighted_y_num'] = df['beacon_y_pos'] * df['rssi_lin']

        wy_num = df.groupby(group_cols)['weighted_y_num'].transform('sum')
        wy_den = df.groupby(group_cols)['rssi_lin'].transform('sum')
        df['weighted_beacon_y'] = wy_num / (wy_den + 1e-6)

        idxmax_rssi = df.groupby(group_time)['rssi'].idxmax()
        strongest_y_map = df.loc[idxmax_rssi].set_index(group_time)['beacon_y_pos']
        df['strongest_beacon_y'] = df.set_index(group_time).index.map(strongest_y_map).fillna(-1)

        df.drop(columns=['rssi_lin', 'weighted_y_num'], inplace=True)
    else:
        df['weighted_beacon_y'] = -1
        df['strongest_beacon_y'] = -1

    return df

def finalise_feature_cols(df):
    global FEATURE_COLS, N_FEATURES
    FEATURE_COLS = [c for c in BASE_SENSOR_COLS if c in df.columns]
    N_FEATURES = len(FEATURE_COLS)
    print(f"\n  Final feature set ({N_FEATURES}): {FEATURE_COLS[:6]} ... {FEATURE_COLS[-4:]}")
    return FEATURE_COLS


# ==============================================================================
# BEACON GRAPH FEATURE EXTRACTOR
# ==============================================================================
def extract_beacon_graph_features(X, beacon_ids, num_beacons=NUM_BEACONS):
    feats = np.zeros((num_beacons, N_BEACON_FEATS), dtype=np.float32)

    rssi_norm_idx = FEATURE_COLS.index('rssi_norm')
    rssi_dist_idx = FEATURE_COLS.index('rssi_distance')
    motion_idx    = FEATURE_COLS.index('motion_magnitude') if 'motion_magnitude' in FEATURE_COLS else None

    b_ids = np.clip(beacon_ids.astype(int), 0, num_beacons - 1)

    for b in range(num_beacons):
        mask = b_ids == b
        if not mask.any():
            continue
        sub = X[mask]
        feats[b, 0] = 1.0
        feats[b, 1] = sub[:, rssi_norm_idx].mean()
        feats[b, 2] = sub[:, rssi_norm_idx].std() if len(sub) > 1 else 0.0
        feats[b, 3] = sub[:, rssi_dist_idx].mean()
        feats[b, 4] = len(sub) / len(X)
        if motion_idx is not None:
            feats[b, 5] = sub[:, motion_idx].mean()

    return feats

def extract_rssi_signature(X, beacon_ids, num_beacons=NUM_BEACONS, fill=-2.0):
    sig = np.full(num_beacons, fill, dtype=np.float32)
    rssi_norm_idx = FEATURE_COLS.index('rssi_norm')
    b_ids = np.clip(beacon_ids.astype(int), 0, num_beacons - 1)
    for b in range(num_beacons):
        mask = b_ids == b
        if mask.any():
            sig[b] = X[mask, rssi_norm_idx].mean()
    return sig


# ==============================================================================
# SEQUENCE CREATION
# ==============================================================================
def create_sequences(df, window_size=WINDOW_SIZE, stride=WINDOW_STRIDE,
                     require_labels=True):
    """
    require_labels=True  → labeled data path (includes y, x, floor_class)
    require_labels=False → trajectory/pretraining path (only sensor sequences)
    """
    feature_cols = FEATURE_COLS
    sequences = []
    grouped = df.groupby(['session_name', 'fingerprintId', 'windowIndex'])
    print(f"\n  Creating sequences from {len(grouped)} windows (labels={require_labels})...")

    rejected = 0
    for (session, fp_id, w_idx), wdf in grouped:
        wdf = wdf.sort_values('capturedAt').reset_index(drop=True)
        n = len(wdf)

        if wdf['beaconId'].nunique() < MIN_BEACONS_PER_WINDOW:
            rejected += 1
            continue

        base = {
            'session':       session,
            'fingerprintId': fp_id,
            'windowIndex':   w_idx,
            'timestamp':     wdf['capturedAt'].min().timestamp() if 'capturedAt' in wdf.columns else 0.0,
        }

        if require_labels:
            base.update({
                'y':           float(wdf['y'].iloc[0]),
                'x':           float(wdf['x'].iloc[0]) if 'x' in wdf.columns else 0.0,
                'floor_class': int(wdf['floor_class'].iloc[0]),
            })

        feat  = wdf[feature_cols].values.astype(np.float32)
        b_ids = wdf['beaconId'].values.astype(np.int32)

        if n < window_size:
            feat_pad  = np.pad(feat,  ((0, window_size - n), (0, 0)), mode='edge')
            b_ids_pad = np.pad(b_ids, (0, window_size - n), mode='edge')
            X_graph   = extract_beacon_graph_features(feat_pad, b_ids_pad)
            sequences.append({**base, 'X': feat_pad, 'X_graph': X_graph, 'beacon_ids': b_ids_pad})
        else:
            for start in range(0, n - window_size + 1, stride):
                sub_feat = feat[start:start + window_size]
                sub_bids = b_ids[start:start + window_size]
                if len(np.unique(sub_bids)) < MIN_BEACONS_PER_WINDOW:
                    rejected += 1
                    continue
                X_graph = extract_beacon_graph_features(sub_feat, sub_bids)
                sequences.append({
                    **base,
                    'X': sub_feat,
                    'X_graph': X_graph,
                    'beacon_ids': sub_bids,
                })

    print(f"  Total sequences: {len(sequences)} (rejected {rejected} under-determined)")
    return sequences

def create_single_sequence(window_df):
    """Lightweight sequence extraction for a single live window (real-time)."""
    global FEATURE_COLS, N_FEATURES
    if FEATURE_COLS is None:
        finalise_feature_cols(window_df)

    feature_cols = FEATURE_COLS
    wdf = window_df.sort_values('capturedAt').reset_index(drop=True)
    n = len(wdf)

    if wdf['beaconId'].nunique() < MIN_BEACONS_PER_WINDOW:
        return None

    base = {
        'y':           float(wdf['y'].iloc[0]) if 'y' in wdf.columns else 0.0,
        'x':           float(wdf['x'].iloc[0]) if 'x' in wdf.columns else 0.0,
        'floor_class': int(wdf['floor_class'].iloc[0]) if 'floor_class' in wdf.columns else 0,
        'session':     str(wdf['session_name'].iloc[0]) if 'session_name' in wdf.columns else 'live',
        'fingerprintId': int(wdf['fingerprintId'].iloc[0]) if 'fingerprintId' in wdf.columns else 0,
        'windowIndex': int(wdf['windowIndex'].iloc[0]) if 'windowIndex' in wdf.columns else 0,
        'timestamp':   wdf['capturedAt'].min().timestamp() if 'capturedAt' in wdf.columns else 0.0,
    }

    feat  = wdf[feature_cols].values.astype(np.float32)
    b_ids = wdf['beaconId'].values.astype(np.int32)

    if n < WINDOW_SIZE:
        feat  = np.pad(feat,  ((0, WINDOW_SIZE - n), (0, 0)), mode='edge')
        b_ids = np.pad(b_ids, (0, WINDOW_SIZE - n), mode='edge')

    X_graph = extract_beacon_graph_features(feat, b_ids)
    return {**base, 'X': feat, 'X_graph': X_graph, 'beacon_ids': b_ids}


# ==============================================================================
# SPLIT — BY SESSION + FINGERPRINT ONLY
# ==============================================================================
def split_by_session(sequences, val_ratio=0.15, test_ratio=0.15, seed=42):
    np.random.seed(seed)
    groups = defaultdict(list)

    for s in sequences:
        groups[(s['session'], s['fingerprintId'])].append(s)

    keys = list(groups.keys())
    np.random.shuffle(keys)

    n = len(keys)
    n_test = int(n * test_ratio)
    n_val  = int(n * val_ratio)

    test_keys  = keys[:n_test]
    val_keys   = keys[n_test:n_test + n_val]
    train_keys = keys[n_test + n_val:]

    train_seq = [s for k in train_keys for s in groups[k]]
    val_seq   = [s for k in val_keys   for s in groups[k]]
    test_seq  = [s for k in test_keys  for s in groups[k]]

    print(f"\n  Split: train={len(train_seq)}, val={len(val_seq)}, test={len(test_seq)}")
    return train_seq, val_seq, test_seq


# ==============================================================================
# NORMALIZATION — TRAIN ONLY
# ==============================================================================
def normalize_sequences(train_seq, val_seq, test_seq):
    n_feat = N_FEATURES

    # --- temporal branch ---
    train_X = np.array([s['X'] for s in train_seq], dtype=np.float32)
    flat = train_X.reshape(-1, n_feat)
    scaler = StandardScaler()
    scaler.fit(flat)

    # Keep beaconId as integer index for Embedding layer
    bidx = FEATURE_COLS.index('beaconId')
    scaler.mean_[bidx]  = 0.0
    scaler.scale_[bidx] = 1.0

    # --- graph branch ---
    train_Xg = np.array([s['X_graph'] for s in train_seq], dtype=np.float32)
    flat_g = train_Xg.reshape(-1, N_BEACON_FEATS)
    scaler_g = StandardScaler()
    scaler_g.fit(flat_g)

    # --- y ---
    train_y = np.array([s['y'] for s in train_seq], dtype=np.float32)
    y_mean = float(train_y.mean())
    y_std  = float(train_y.std())
    y_std  = max(y_std, 1e-6)

    for seq_list in [train_seq, val_seq, test_seq]:
        for s in seq_list:
            s['X']       = scaler.transform(s['X'].reshape(-1, n_feat)).reshape(WINDOW_SIZE, n_feat).astype(np.float32)
            s['X_graph'] = scaler_g.transform(s['X_graph'].reshape(-1, N_BEACON_FEATS)).reshape(NUM_BEACONS, N_BEACON_FEATS).astype(np.float32)
            s['y_norm']  = (s['y'] - y_mean) / y_std

    json_save({
        'feature_mean':  scaler.mean_.tolist(),
        'feature_scale': scaler.scale_.tolist(),
        'feature_cols':  FEATURE_COLS,
        'graph_mean':    scaler_g.mean_.tolist(),
        'graph_scale':   scaler_g.scale_.tolist(),
        'y_mean': y_mean,
        'y_std':  y_std,
        'y_min':  Y_MIN,
        'y_max':  Y_MAX,
    }, NORM_FILE_PATH)

    print(f"  y∈[{train_y.min():.1f},{train_y.max():.1f}] μ={y_mean:.2f} σ={y_std:.2f}")
    return train_seq, val_seq, test_seq, (y_mean, y_std), scaler, scaler_g

def normalize_trajectory_sequences(traj_seq, scaler):
    """
    Normalize trajectory sequences using the SUPERVISED scaler (train-only stats).
    Only the temporal branch (X) is normalized — no graph branch needed for pretraining.
    """
    n_feat = N_FEATURES
    for s in traj_seq:
        s['X'] = scaler.transform(s['X'].reshape(-1, n_feat)).reshape(WINDOW_SIZE, n_feat).astype(np.float32)
    return traj_seq

def normalize_single_sequence(seq, scaler, scaler_g, y_mean, y_std):
    """Normalize a single live-window sequence in-place."""
    n_feat = N_FEATURES
    seq['X']       = scaler.transform(seq['X'].reshape(-1, n_feat)).reshape(WINDOW_SIZE, n_feat).astype(np.float32)
    seq['X_graph'] = scaler_g.transform(seq['X_graph'].reshape(-1, N_BEACON_FEATS)).reshape(NUM_BEACONS, N_BEACON_FEATS).astype(np.float32)
    seq['y_norm']  = (seq['y'] - y_mean) / max(y_std, 1e-6)
    return seq


# ==============================================================================
# GENERATORS
# ==============================================================================

# ── NEW: Masked pretraining generator (from Code 2) ──────────────────────────
class MaskedPretrainingGenerator(Sequence):
    """
    Self-supervised generator for temporal-branch pretraining.
    Input:  masked sensor sequence (some time steps zeroed out)
    Target: original sensor sequence (reconstruction objective)
    Only uses 'X' (temporal features) — no graph branch needed for pretraining.
    """
    def __init__(self, sequences, batch_size=BATCH_SIZE,
                 shuffle=True, mask_ratio=PRETRAIN_MASK_RATIO):
        self.sequences  = sequences
        self.batch_size = batch_size
        self.shuffle    = shuffle
        self.mask_ratio = mask_ratio
        self.indices    = np.arange(len(sequences))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.sequences) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx  = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        original_X = np.array([self.sequences[i]['X'] for i in batch_idx], dtype=np.float32)

        # Build mask: True = position to mask out
        mask     = np.random.rand(*original_X.shape) < self.mask_ratio
        masked_X = original_X.copy()

        # Zero out masked positions; protect beaconId (col 0) from masking
        masked_X[:, :, 1:][mask[:, :, 1:]] = 0.0

        # sample_weight = mask so loss only applies to masked positions
        return masked_X, original_X, mask.astype(np.float32)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


# ── Existing: Dual-input supervised generator ────────────────────────────────
class MultiTaskGenerator(Sequence):
    def __init__(self, sequences, batch_size=BATCH_SIZE, shuffle=True, augment=False):
        self.sequences  = sequences
        self.batch_size = batch_size
        self.shuffle    = shuffle
        self.augment    = augment
        self.indices    = np.arange(len(sequences))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.sequences) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        X_seq   = np.array([self.sequences[i]['X']          for i in batch_idx], dtype=np.float32)
        X_graph = np.array([self.sequences[i]['X_graph']    for i in batch_idx], dtype=np.float32)
        y  = np.array([[self.sequences[i]['y_norm']]        for i in batch_idx], dtype=np.float32)
        fl = np.array([self.sequences[i]['floor_class']     for i in batch_idx], dtype=np.float32)

        if self.augment:
            rssi_idx = FEATURE_COLS.index('rssi') if 'rssi' in FEATURE_COLS else 1
            X_seq[:, :, rssi_idx] += np.random.normal(0, 0.10, X_seq[:, :, rssi_idx].shape)

            mask = np.random.random((X_seq.shape[0], X_seq.shape[1])) > 0.12
            X_seq[:, :, 1:] = X_seq[:, :, 1:] * mask[:, :, np.newaxis]

            noise = np.random.normal(0, 0.015, X_seq.shape)
            noise[:, :, 0] = 0.0
            X_seq += noise
            X_seq[:, :, 0] = np.round(np.clip(X_seq[:, :, 0], 0, NUM_BEACONS - 1))

            X_graph[:, :, 1] += np.random.normal(0, 0.08, X_graph[:, :, 1].shape)
            X_graph[:, :, 3] *= np.random.uniform(0.9, 1.1, X_graph[:, :, 3].shape)
            drop_mask = np.random.random((X_graph.shape[0], NUM_BEACONS)) > 0.05
            X_graph[:, :, 0] *= drop_mask.astype(np.float32)
            X_graph += np.random.normal(0, 0.01, X_graph.shape)

        return (X_seq, X_graph), {'y_output': y, 'floor_output': fl}

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


# ==============================================================================
# LOSSES
# ==============================================================================
def logcosh_euclidean(y_true, y_pred):
    dist = tf.norm(y_true - y_pred, axis=-1)
    dist = tf.clip_by_value(dist, 0.0, 50.0)
    return tf.reduce_mean(tf.math.log(tf.cosh(dist) + 1e-8))

def mae_euclidean(y_true, y_pred):
    return tf.reduce_mean(tf.norm(y_true - y_pred, axis=-1))

def masked_mse_loss(y_true, y_pred, sample_weight=None):
    """MSE only over masked positions (used in pretraining)."""
    squared_error = tf.square(y_true - y_pred)
    if sample_weight is not None:
        mask      = tf.cast(sample_weight, tf.float32)
        masked_se = squared_error * mask
        n_masked  = tf.reduce_sum(mask) + 1e-8
        return tf.reduce_sum(masked_se) / n_masked
    return tf.reduce_mean(squared_error)


# ==============================================================================
# MODEL — DUAL-BRANCH: TEMPORAL TRANSFORMER + BEACON GRAPH ATTENTION
#
# KEY DESIGN: the TEMPORAL BRANCH backbone is named consistently so that
# pretrained weights can be transferred layer-by-layer after pretraining.
# Naming convention for transferable layers: prefix 'tb_' (temporal branch).
# ==============================================================================

# Layer name prefixes belonging to the temporal backbone (used for transfer)
TEMPORAL_BACKBONE_PREFIXES = (
    'tb_beacon_emb', 'tb_concat', 'tb_proj',
    'tb_spatial_drop',
    'tb_mha_', 'tb_ln_attn_', 'tb_ffn_a_', 'tb_ffn_drop_', 'tb_ffn_b_', 'tb_ln_ffn_',
    'tb_gap', 'tb_gmp', 'tb_pool_concat',
)


def _build_temporal_backbone(inp):
    """
    Shared temporal transformer backbone.
    All layers are named with 'tb_' prefix so they can be matched
    during weight transfer from the pretrainer to the localizer.
    """
    beacon_ids = layers.Lambda(lambda x: tf.cast(x[:, :, 0], tf.int32))(inp)
    beacon_emb = layers.Embedding(NUM_BEACONS, EMBED_DIM, name='tb_beacon_emb')(beacon_ids)

    rest = layers.Lambda(lambda x: x[:, :, 1:])(inp)
    x    = layers.Concatenate(axis=-1, name='tb_concat')([beacon_emb, rest])

    x = layers.Dense(128, activation='gelu', name='tb_proj')(x)
    x = layers.SpatialDropout1D(0.15, name='tb_spatial_drop')(x)

    for i in range(4):
        attn = layers.MultiHeadAttention(
            num_heads=8, key_dim=16, dropout=0.15, name=f'tb_mha_{i}'
        )(x, x)
        x = layers.LayerNormalization(name=f'tb_ln_attn_{i}')(x + attn)

        ffn = layers.Dense(256, activation='gelu', name=f'tb_ffn_a_{i}')(x)
        ffn = layers.Dropout(0.15, name=f'tb_ffn_drop_{i}')(ffn)
        ffn = layers.Dense(128, name=f'tb_ffn_b_{i}')(ffn)
        x   = layers.LayerNormalization(name=f'tb_ln_ffn_{i}')(x + ffn)

    avg_t = layers.GlobalAveragePooling1D(name='tb_gap')(x)
    max_t = layers.GlobalMaxPooling1D(name='tb_gmp')(x)
    pooled = layers.Concatenate(name='tb_pool_concat')([avg_t, max_t])   # 256 dim

    return pooled, x   # (pooled embedding, full sequence output)


def build_temporal_pretrainer(name='temporal_pretrainer'):
    """
    Self-supervised pretrainer: masked autoencoder on the temporal branch only.
    Input  → temporal backbone → reconstruct original sequence.
    Graph branch is NOT included here (it will be trained from scratch on labeled data).
    """
    inp = layers.Input(shape=(WINDOW_SIZE, N_FEATURES), name='sequence_input')

    _, x_seq = _build_temporal_backbone(inp)

    recon = layers.Dense(256, activation='gelu', name='recon_dense')(x_seq)
    recon = layers.Dense(N_FEATURES, activation='linear', name='reconstruction')(recon)

    model = Model(inputs=inp, outputs=recon, name=name)
    model.compile(
        optimizer=optimizers.Adam(1e-4, clipvalue=1.0),
        loss=masked_mse_loss,
    )
    return model


def build_hybrid_localizer(name='hybrid_gat'):
    """
    Full dual-branch localizer:
      • Temporal branch  — uses shared _build_temporal_backbone (pretrain-compatible)
      • Graph branch     — always trained from scratch on labeled data
    """
    seq_inp   = layers.Input(shape=(WINDOW_SIZE, N_FEATURES),    name='sequence_input')
    graph_inp = layers.Input(shape=(NUM_BEACONS, N_BEACON_FEATS), name='beacon_graph_input')

    # ── TEMPORAL BRANCH (pretrain-compatible) ────────────────────────────────
    temporal_emb, _ = _build_temporal_backbone(seq_inp)   # 256 dim

    # ── BEACON GRAPH BRANCH ──────────────────────────────────────────────────
    def tile_beacon_indices(x):
        return tf.tile(tf.expand_dims(tf.range(NUM_BEACONS, dtype=tf.int32), 0),
                       [tf.shape(x)[0], 1])

    beacon_idx_layer = layers.Lambda(tile_beacon_indices)(graph_inp)
    graph_beacon_emb = layers.Embedding(NUM_BEACONS, EMBED_DIM)(beacon_idx_layer)

    x_g = layers.Concatenate(axis=-1)([graph_beacon_emb, graph_inp])   # (B, 7, 16+6)
    x_g = layers.Dense(64, activation='gelu')(x_g)

    for _ in range(2):
        attn_g = layers.MultiHeadAttention(num_heads=4, key_dim=16, dropout=0.1)(x_g, x_g)
        x_g = layers.LayerNormalization()(x_g + attn_g)

        ffn_g = layers.Dense(128, activation='gelu')(x_g)
        ffn_g = layers.Dropout(0.1)(ffn_g)
        ffn_g = layers.Dense(64)(ffn_g)
        x_g = layers.LayerNormalization()(x_g + ffn_g)

    avg_g = layers.GlobalAveragePooling1D()(x_g)
    max_g = layers.GlobalMaxPooling1D()(x_g)
    graph_emb = layers.Concatenate()([avg_g, max_g])                   # 128 dim

    # ── FUSION ───────────────────────────────────────────────────────────────
    fused  = layers.Concatenate()([temporal_emb, graph_emb])            # 384 dim
    shared = layers.Dense(256, activation='gelu')(fused)
    shared = layers.BatchNormalization(epsilon=1e-4)(shared)
    shared = layers.Dropout(0.35)(shared)
    shared = layers.Dense(128, activation='gelu')(shared)
    shared = layers.BatchNormalization(epsilon=1e-4)(shared)
    shared = layers.Dropout(0.25)(shared)

    # --- Y branch ---
    y_branch = layers.Dense(256, activation='gelu')(shared)
    y_branch = layers.BatchNormalization(epsilon=1e-4)(y_branch)
    y_branch = layers.Dropout(0.25)(y_branch)
    y_branch = layers.Dense(128, activation='gelu')(y_branch)
    y_branch = layers.BatchNormalization(epsilon=1e-4)(y_branch)
    y_branch = layers.Dropout(0.15)(y_branch)
    y_branch = layers.Dense(64, activation='gelu')(y_branch)
    y_out    = layers.Dense(1, activation='linear', name='y_output')(y_branch)

    # --- Floor branch ---
    fl     = layers.Dense(64, activation='gelu')(shared)
    fl     = layers.Dropout(0.1)(fl)
    fl_out = layers.Dense(1, activation='sigmoid', name='floor_output')(fl)

    model = Model(inputs=[seq_inp, graph_inp],
                  outputs={'y_output': y_out, 'floor_output': fl_out},
                  name=name)

    model.compile(
        optimizer=optimizers.Adam(1.5e-4, clipnorm=1.0),
        loss={
            'y_output':     logcosh_euclidean,
            'floor_output': 'binary_crossentropy',
        },
        loss_weights={
            'y_output':     W_Y,
            'floor_output': W_FLOOR,
        },
        metrics={
            'y_output':     [mae_euclidean],
            'floor_output': ['accuracy'],
        }
    )
    return model


def transfer_temporal_weights(pretrain_model, localizer_model):
    """
    Copy temporal backbone weights from pretrainer → localizer.
    Only layers whose names start with a 'tb_' prefix are transferred;
    everything else (graph branch, heads) is left untouched.
    """
    pretrain_layer_map = {l.name: l for l in pretrain_model.layers}
    transferred, skipped = 0, 0

    for layer in localizer_model.layers:
        is_temporal = any(layer.name.startswith(p) for p in TEMPORAL_BACKBONE_PREFIXES)
        if not is_temporal:
            continue
        if layer.name in pretrain_layer_map:
            src = pretrain_layer_map[layer.name]
            src_w = src.get_weights()
            if src_w:
                layer.set_weights(src_w)
                transferred += 1
            else:
                skipped += 1
        else:
            skipped += 1

    print(f"  Temporal weight transfer: {transferred} layers transferred, {skipped} skipped.")


# ==============================================================================
# CALLBACKS
# ==============================================================================
class HistoryCheckpoint(callbacks.Callback):
    def __init__(self, name):
        super().__init__()
        self.path    = os.path.join(CACHE_MODELS_DIR, f'{name}.history.json')
        self.history = json_load(self.path) if os.path.exists(self.path) else {}

    def on_epoch_end(self, epoch, logs=None):
        for k, v in (logs or {}).items():
            self.history.setdefault(k, []).append(float(v))
        json_save(self.history, self.path)

    def get(self):
        return self.history

def cosine_warmup(epoch, lr, warmup=10, total=400, base_lr=1.5e-4, min_lr=1e-6):
    if epoch < warmup:
        return base_lr * (epoch + 1) / warmup
    progress = (epoch - warmup) / max(total - warmup, 1)
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * progress))


# ── NEW: Pretrainer wrapper ───────────────────────────────────────────────────
class PretrainModel:
    """
    Manages self-supervised pretraining of the temporal backbone.
    Skips if pretrained weights already exist on disk.
    """
    def __init__(self, model):
        self.model   = model
        self.trained = os.path.exists(PRETRAINED_WEIGHTS_PATH)
        if self.trained:
            self.model.load_weights(PRETRAINED_WEIGHTS_PATH)
            print(f'  [pretrainer] weights loaded from {PRETRAINED_WEIGHTS_PATH}')

    def fit(self, traj_seq, epochs=PRETRAIN_EPOCHS, force=False):
        if self.trained and not force:
            print('  [pretrainer] already pretrained — skipping')
            return

        n_val = max(1, int(len(traj_seq) * 0.10))
        np.random.shuffle(traj_seq)
        val_seq_pt   = traj_seq[:n_val]
        train_seq_pt = traj_seq[n_val:]

        train_gen_pt = MaskedPretrainingGenerator(train_seq_pt, shuffle=True)
        val_gen_pt   = MaskedPretrainingGenerator(val_seq_pt,   shuffle=False)

        cb = [
            callbacks.TerminateOnNaN(),
            callbacks.EarlyStopping(monitor='val_loss', patience=10,
                                    restore_best_weights=True),
            callbacks.ModelCheckpoint(PRETRAINED_WEIGHTS_PATH, monitor='val_loss',
                                      save_best_only=True, save_weights_only=True),
            callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                        patience=5, min_lr=1e-7),
        ]

        print(f"\n  Pretraining on {len(train_seq_pt)} sequences "
              f"({len(val_seq_pt)} val) for up to {epochs} epochs …")
        self.model.fit(train_gen_pt, validation_data=val_gen_pt,
                       epochs=epochs, callbacks=cb, verbose=1)
        self.trained = True


# ── Existing: Supervised fine-tuning wrapper ──────────────────────────────────
class CustomModel:
    def __init__(self, name, model):
        self.name    = re.sub(r'[^A-Za-z0-9]', '_', name).lower()
        self.weights = os.path.join(CACHE_MODELS_DIR, f'{self.name}.weights.h5')
        self.model   = model
        self.history = {}
        self._load()

    def _load(self):
        if os.path.exists(self.weights):
            self.model.load_weights(self.weights)
            self.trained = True
            print(f'  [{self.name}] weights loaded')
        else:
            self.trained = False
        hcp = HistoryCheckpoint(self.name)
        self.history = hcp.get()

    def fit(self, train_gen, val_gen, epochs=400, force=False):
        if self.trained and not force:
            print(f'  [{self.name}] already trained, skipping')
            return
        hcp = HistoryCheckpoint(self.name)
        cb = [
            callbacks.TerminateOnNaN(),
            callbacks.EarlyStopping(monitor='val_loss', patience=35,
                                    restore_best_weights=True),
            callbacks.ModelCheckpoint(self.weights, monitor='val_loss',
                                      save_best_only=True, save_weights_only=True),
            callbacks.LearningRateScheduler(
                lambda e, lr: cosine_warmup(e, lr, total=epochs)
            ),
            hcp,
        ]
        self.model.fit(train_gen, validation_data=val_gen,
                       epochs=epochs, callbacks=cb, verbose=1)
        self.trained = True
        self.history = hcp.get()


# ==============================================================================
# kNN FINGERPRINT RETRIEVAL
# ==============================================================================
class FingerprintKNN:
    def __init__(self, k=KNN_K):
        self.k = k
        self.nn = None
        self.X_sigs = None
        self.y_vals = None

    def fit(self, train_seq):
        fp_data = defaultdict(lambda: {'sigs': [], 'y': None, 'floor': None})
        for s in train_seq:
            key = (s['session'], s['fingerprintId'])
            sig = extract_rssi_signature(s['X'], s['beacon_ids'])
            fp_data[key]['sigs'].append(sig)
            fp_data[key]['y']     = s['y']
            fp_data[key]['floor'] = s['floor_class']

        keys, X_sigs, y_vals = [], [], []
        for key, data in fp_data.items():
            keys.append(key)
            X_sigs.append(np.mean(data['sigs'], axis=0))
            y_vals.append(data['y'])

        self.keys   = keys
        self.X_sigs = np.array(X_sigs)
        self.y_vals = np.array(y_vals)

        k_eff = min(self.k, len(X_sigs))
        self.nn = NearestNeighbors(n_neighbors=k_eff, metric='euclidean')
        self.nn.fit(self.X_sigs)
        print(f"  kNN DB: {len(X_sigs)} fingerprints, k={k_eff}")

    def predict(self, test_seq):
        sigs  = np.array([extract_rssi_signature(s['X'], s['beacon_ids']) for s in test_seq])
        k_eff = min(self.k, len(self.X_sigs))
        dist, idx = self.nn.kneighbors(sigs, n_neighbors=k_eff)
        w = 1.0 / (dist + 1e-6)
        w = w / w.sum(axis=1, keepdims=True)
        return np.sum(self.y_vals[idx] * w, axis=1)


# ==============================================================================
# REAL-TIME KALMAN FILTER (Causal, Forward-Only)
# ==============================================================================
class OneDKalmanFilter:
    def __init__(self, Q=None, R=None, process_noise=None, measurement_noise=None):
        if Q is not None: process_noise = Q
        if R is not None: measurement_noise = R
        self.Q = float(process_noise)     if process_noise     is not None else float(KALMAN_Q)
        self.R = float(measurement_noise) if measurement_noise is not None else float(KALMAN_R)
        self.x = 0.0
        self.P = 1.0
        self.initialized = False

    def reset(self):
        self.initialized = False
        self.x = 0.0
        self.P = 1.0

    def update(self, measurement):
        if not self.initialized:
            self.x = float(measurement)
            self.P = 1.0
            self.initialized = True
            return self.x
        x_pred = self.x
        P_pred = self.P + self.Q
        K      = P_pred / (P_pred + self.R)
        self.x = x_pred + K * (measurement - x_pred)
        self.P = (1.0 - K) * P_pred
        return self.x


# ==============================================================================
# BATCH KALMAN SMOOTHER (Forward-Backward RTS)
# ==============================================================================
class OneDKalmanSmoother:
    def __init__(self, process_noise=None, measurement_noise=None, Q=None, R=None):
        if Q is not None: process_noise = Q
        if R is not None: measurement_noise = R
        self.Q = float(process_noise)     if process_noise     is not None else float(KALMAN_Q)
        self.R = float(measurement_noise) if measurement_noise is not None else float(KALMAN_R)

    def smooth(self, measurements):
        n = len(measurements)
        if n == 0: return []
        x    = np.zeros(n)
        P    = np.zeros(n)
        x[0] = float(measurements[0])
        P[0] = 1.0
        for t in range(1, n):
            x_pred = x[t-1]
            P_pred = P[t-1] + self.Q
            K      = P_pred / (P_pred + self.R)
            x[t]   = x_pred + K * (measurements[t] - x_pred)
            P[t]   = (1.0 - K) * P_pred
        x_s = x.copy()
        for t in range(n - 2, -1, -1):
            denom = P[t] + self.Q
            C     = P[t] / denom if denom > 1e-9 else 0.0
            x_s[t] = x[t] + C * (x_s[t+1] - x[t])
        return x_s.tolist()


# ==============================================================================
# PER-TRAJECTORY MAD OUTLIER REJECTION
# ==============================================================================
class MADOutlierRejector:
    def __init__(self, window=10, z_thresh=3.0):
        self.window   = window
        self.z_thresh = z_thresh
        self.history  = []

    def reset(self):
        self.history = []

    def update(self, value):
        self.history.append(float(value))
        if len(self.history) > self.window:
            self.history.pop(0)
        if len(self.history) < 3:
            return value
        arr = np.array(self.history)
        med = np.median(arr)
        mad = np.median(np.abs(arr - med))
        if mad < 1e-6:
            return value
        z = np.abs(value - med) / (1.4826 * mad)
        if z > self.z_thresh:
            return self.history[-2] if len(self.history) >= 2 else med
        return value

def apply_mad_per_trajectory(y_pred, test_seq, z_thresh=3.0):
    fp_data = defaultdict(list)
    for i, s in enumerate(test_seq):
        key = (s['session'], s['fingerprintId'])
        fp_data[key].append(i)
    y_clean = y_pred.copy()
    for indices in fp_data.values():
        if len(indices) < 5: continue
        vals = y_pred[indices]
        med  = np.median(vals)
        mad  = np.median(np.abs(vals - med))
        if mad < 1e-6: continue
        for idx in indices:
            z = np.abs(y_pred[idx] - med) / (1.4826 * mad)
            if z > z_thresh:
                y_clean[idx] = med
    return y_clean


# ==============================================================================
# REAL-TIME INFERENCE ENGINE
# ==============================================================================
class RealTimeLocalizer:
    def __init__(self, model, scaler, scaler_g, y_stats,
                 beacon_median_map, floor_median_map, sensor_medians,
                 use_tta=False, use_knn=False, knn_db=None,
                 kalman_q=KALMAN_Q, kalman_r=KALMAN_R,
                 outlier_z=3.0, outlier_window=10):
        self.model            = model
        self.scaler           = scaler
        self.scaler_g         = scaler_g
        self.y_mean, self.y_std = y_stats
        self.y_stats          = y_stats
        self.beacon_median_map = beacon_median_map
        self.floor_median_map  = floor_median_map
        self.sensor_medians    = sensor_medians
        self.use_tta           = use_tta
        self.use_knn           = use_knn
        self.knn_db            = knn_db
        self.kalman            = OneDKalmanFilter(Q=kalman_q, R=kalman_r)
        self.outlier_rejector  = MADOutlierRejector(window=outlier_window, z_thresh=outlier_z)
        self.last_floor        = None
        self.last_y            = None

    def reset(self):
        self.kalman.reset()
        self.outlier_rejector.reset()
        self.last_floor = None
        self.last_y     = None

    def _infer_floor_from_beacons(self, beacon_ids):
        floors = [BEACON_FLOOR.get(int(b), 0) for b in beacon_ids if int(b) in BEACON_FLOOR]
        if not floors:
            return self.last_floor if self.last_floor is not None else 3
        return int(max(set(floors), key=floors.count))

    def _prepare_window_df(self, window_df, assumed_floor=None):
        df = window_df.copy()
        if 'beaconId' not in df.columns and 'beaconUid' in df.columns:
            df['beaconId'] = df['beaconUid'].map(BEACON_MAP)
        df = df.dropna(subset=['beaconId'])
        if len(df) == 0: return df
        df['beaconId'] = df['beaconId'].astype(int)

        if 'floorLevel' not in df.columns or df['floorLevel'].isna().all():
            if assumed_floor is None:
                assumed_floor = self._infer_floor_from_beacons(df['beaconId'].values)
            df['floorLevel'] = assumed_floor

        for col, default in [('session_name','live'), ('fingerprintId',0),
                              ('windowIndex',0), ('y',0.0), ('x',0.0)]:
            if col not in df.columns:
                df[col] = default
        if 'capturedAt' not in df.columns:
            df['capturedAt'] = pd.Timestamp.now()

        df = preprocess_data(df, sensor_medians=self.sensor_medians)
        df = engineer_features(df, beacon_median_map=self.beacon_median_map,
                               floor_median_map=self.floor_median_map)
        return df

    def predict_window(self, window_df, assumed_floor=None, return_floor=False):
        df = self._prepare_window_df(window_df, assumed_floor)
        if len(df) == 0: return (None, None) if return_floor else None

        seq = create_single_sequence(df)
        if seq is None: return (None, None) if return_floor else None

        normalize_single_sequence(seq, self.scaler, self.scaler_g, self.y_mean, self.y_std)

        X_seq   = np.expand_dims(seq['X'], 0)
        X_graph = np.expand_dims(seq['X_graph'], 0)

        y_knn = None
        if self.use_knn and self.knn_db is not None:
            sig   = extract_rssi_signature(seq['X'], seq['beacon_ids'])
            k_eff = min(self.knn_db.k, len(self.knn_db.X_sigs))
            if k_eff > 0:
                dist, idx = self.knn_db.nn.kneighbors(np.expand_dims(sig, 0), n_neighbors=k_eff)
                w = 1.0 / (dist + 1e-6)
                w = w / w.sum(axis=1, keepdims=True)
                y_knn = np.sum(self.knn_db.y_vals[idx] * w, axis=1)[0]

        if self.use_tta:
            y_pred, fl_prob = predict_with_tta(self.model, X_seq, X_graph, self.y_stats, n_tta=10)
            y_pred  = float(y_pred[0])
            fl_prob = float(fl_prob[0])
        else:
            out     = self.model.predict([X_seq, X_graph], batch_size=1, verbose=0)
            y_pred  = float(out['y_output'][0, 0] * self.y_std + self.y_mean)
            fl_prob = float(out['floor_output'][0, 0])

        y_pred  = float(np.clip(y_pred, Y_MIN, Y_MAX))
        fl_pred = 1 if fl_prob >= 0.5 else 0
        fl_label = FLOOR_CLASSES_INV[fl_pred]

        if y_knn is not None and abs(y_pred - y_knn) > 10.0:
            y_pred = y_knn

        self.last_floor = fl_label
        self.last_y     = y_pred
        return (y_pred, fl_label) if return_floor else y_pred

    def update(self, window_df, assumed_floor=None):
        y_raw, fl = self.predict_window(window_df, assumed_floor, return_floor=True)
        if y_raw is None: return None, None
        y_clean  = self.outlier_rejector.update(y_raw)
        y_smooth = self.kalman.update(y_clean)
        self.last_y = y_smooth
        return float(y_smooth), int(fl)


# ==============================================================================
# STREAMING LOCALIZER
# ==============================================================================
class StreamingLocalizer:
    def __init__(self, localizer, window_size=WINDOW_SIZE, min_rows=4):
        self.localizer    = localizer
        self.window_size  = window_size
        self.min_rows     = min_rows
        self.buffer       = []
        self.obs_count    = 0
        self.session_name = 'live_stream'
        self.fingerprint_id = 0
        self.window_index = 0

    def reset(self):
        self.buffer = []
        self.obs_count  = 0
        self.window_index = 0
        self.localizer.reset()

    def add_observation(self, beacon_uid, rssi, timestamp=None,
                        gyro=None, accel=None, user_accel=None,
                        mag=None, pitch=None, roll=None, yaw=None,
                        floor_level=None):
        row = {
            'beaconUid':     beacon_uid,
            'rssi':          rssi,
            'capturedAt':    pd.Timestamp(timestamp) if timestamp else pd.Timestamp.now(),
            'session_name':  self.session_name,
            'fingerprintId': self.fingerprint_id,
            'windowIndex':   self.window_index,
        }
        if floor_level  is not None: row['floorLevel'] = floor_level
        if gyro         is not None: row['gyroX'], row['gyroY'], row['gyroZ'] = gyro
        if accel        is not None: row['accelX'], row['accelY'], row['accelZ'] = accel
        if user_accel   is not None: row['userAccelX'], row['userAccelY'], row['userAccelZ'] = user_accel
        if mag          is not None: row['magX'], row['magY'], row['magZ'] = mag
        if pitch        is not None: row['pitch'] = pitch
        if roll         is not None: row['roll']  = roll
        if yaw          is not None: row['yaw']   = yaw

        self.buffer.append(row)
        self.obs_count += 1

        if len(self.buffer) > self.window_size * 2:
            self.advance_window()

    def advance_window(self):
        self.window_index   += 1
        self.buffer          = []
        self.fingerprint_id += 1

    def predict(self, force=False):
        if len(self.buffer) < self.min_rows and not force:
            return None, None
        if len(self.buffer) < self.min_rows:
            return None, None
        df = pd.DataFrame(self.buffer)
        return self.localizer.update(df)


# ==============================================================================
# TEST-TIME AUGMENTATION
# ==============================================================================
def predict_with_tta(model, X_seq, X_graph, y_stats, n_tta=20):
    y_mean, y_std = y_stats
    preds_y, preds_fl = [], []
    rssi_idx = FEATURE_COLS.index('rssi') if 'rssi' in FEATURE_COLS else 1

    for i in range(n_tta):
        Xs_aug = X_seq.copy()
        Xg_aug = X_graph.copy()
        if i > 0:
            Xs_aug[:, :, rssi_idx] += np.random.normal(0, 0.12, Xs_aug[:, :, rssi_idx].shape)
            mask = np.random.random((Xs_aug.shape[0], Xs_aug.shape[1])) > 0.15
            Xs_aug[:, :, 1:] = Xs_aug[:, :, 1:] * mask[:, :, np.newaxis]
            noise = np.random.normal(0, 0.02, Xs_aug.shape)
            noise[:, :, 0] = 0.0
            Xs_aug += noise
            Xs_aug[:, :, 0] = np.round(np.clip(Xs_aug[:, :, 0], 0, NUM_BEACONS - 1))
            Xg_aug[:, :, 1] += np.random.normal(0, 0.08, Xg_aug[:, :, 1].shape)
            Xg_aug[:, :, 3] *= np.random.uniform(0.9, 1.1, Xg_aug[:, :, 3].shape)
            drop_mask = np.random.random((Xg_aug.shape[0], NUM_BEACONS)) > 0.08
            Xg_aug[:, :, 0] *= drop_mask.astype(np.float32)
            Xg_aug += np.random.normal(0, 0.015, Xg_aug.shape)

        out = model.predict([Xs_aug, Xg_aug], batch_size=BATCH_SIZE, verbose=0)
        preds_y.append(out['y_output'])
        preds_fl.append(out['floor_output'])

    y_med  = np.median(preds_y,  axis=0)
    fl_med = np.median(preds_fl, axis=0)
    y_pred = y_med[:, 0] * y_std + y_mean
    y_pred = np.clip(y_pred, Y_MIN, Y_MAX)
    return y_pred, fl_med


# ==============================================================================
# POST-PROCESSING & EVALUATION
# ==============================================================================
def geometric_postprocess(y_pred, test_seq, train_seq, y_stats):
    train_ys  = np.array(sorted(set(s['y'] for s in train_seq)))
    train_min, train_max = train_ys.min(), train_ys.max()
    snapped   = y_pred.copy()
    n_clamped = 0
    for i, y in enumerate(y_pred):
        if y < train_min - 3.0 or y > train_max + 3.0:
            snapped[i] = np.clip(y, train_min - 2.0, train_max + 2.0)
            n_clamped += 1
    if n_clamped > 0:
        print(f"    Corridor clamp: fixed {n_clamped} out-of-bounds predictions")
    return snapped

def fingerprint_kalman_smooth(y_pred, test_seq,
                               process_noise=KALMAN_Q, measurement_noise=KALMAN_R):
    fp_data = defaultdict(lambda: {'indices': [], 'y_preds': [], 'timestamp': None})
    for i, s in enumerate(test_seq):
        key = (s['session'], s['fingerprintId'])
        fp_data[key]['indices'].append(i)
        fp_data[key]['y_preds'].append(y_pred[i])
        if fp_data[key]['timestamp'] is None:
            fp_data[key]['timestamp'] = s.get('timestamp', 0)

    fp_y = {k: np.median(v['y_preds']) for k, v in fp_data.items()}
    sessions = defaultdict(list)
    for key, data in fp_data.items():
        sessions[key[0]].append({'key': key, 'ts': data['timestamp'], 'indices': data['indices']})

    y_smooth = y_pred.copy()
    smoother = OneDKalmanSmoother(process_noise=process_noise, measurement_noise=measurement_noise)

    for session, items in sessions.items():
        if len(items) < 3: continue
        items.sort(key=lambda x: x['ts'])
        y_seq       = [fp_y[it['key']] for it in items]
        y_smooth_seq = smoother.smooth(y_seq)
        for it, ys in zip(items, y_smooth_seq):
            for idx in it['indices']:
                y_smooth[idx] = ys

    return y_smooth

def evaluate_model(model_wrapper, test_seq, y_stats, train_seq, val_seq=None):
    y_mean, y_std = y_stats

    X_seq_test   = np.array([s['X']          for s in test_seq], dtype=np.float32)
    X_graph_test = np.array([s['X_graph']    for s in test_seq], dtype=np.float32)
    y_test       = np.array([s['y']          for s in test_seq], dtype=np.float32)
    fl_test      = np.array([s['floor_class'] for s in test_seq], dtype=np.float32)

    knn_db = FingerprintKNN(k=KNN_K)
    knn_db.fit(train_seq)

    out_raw  = model_wrapper.model.predict([X_seq_test, X_graph_test], batch_size=BATCH_SIZE, verbose=0)
    y_nn_raw = out_raw['y_output'][:, 0] * y_std + y_mean
    y_nn_raw = np.clip(y_nn_raw, Y_MIN, Y_MAX)

    y_nn_tta, fl_tta = predict_with_tta(model_wrapper.model, X_seq_test, X_graph_test, y_stats, n_tta=20)
    y_knn = knn_db.predict(test_seq)

    if val_seq is not None and len(val_seq) > 0:
        X_seq_val   = np.array([s['X']       for s in val_seq], dtype=np.float32)
        X_graph_val = np.array([s['X_graph'] for s in val_seq], dtype=np.float32)
        y_val       = np.array([s['y']       for s in val_seq], dtype=np.float32)

        out_val  = model_wrapper.model.predict([X_seq_val, X_graph_val], batch_size=BATCH_SIZE, verbose=0)
        y_nn_val = out_val['y_output'][:, 0] * y_std + y_mean
        y_knn_val = knn_db.predict(val_seq)

        best_w, best_mae = 0.5, float('inf')
        for w in np.arange(0.0, 1.01, 0.05):
            mae = np.abs(y_val - (w * y_nn_val + (1 - w) * y_knn_val)).mean()
            if mae < best_mae:
                best_mae, best_w = mae, w
        print(f"  Tuned NN/kNN blend weight: {best_w:.2f} (val MAE={best_mae:.3f}m)")
    else:
        best_w = 0.70

    y_hybrid = best_w * y_nn_tta + (1.0 - best_w) * y_knn
    y_geom   = geometric_postprocess(y_hybrid, test_seq, train_seq, y_stats)

    if val_seq is not None and len(val_seq) > 0:
        y_hybrid_val = best_w * y_nn_val + (1.0 - best_w) * y_knn_val
        y_geom_val   = geometric_postprocess(y_hybrid_val, val_seq, train_seq, y_stats)
        best_q, best_r, best_mae = KALMAN_Q, KALMAN_R, float('inf')
        for Q in [0.01, 0.05, 0.1, 0.2, 0.5]:
            for R in [0.5, 1.0, 2.0, 4.0]:
                y_sm = fingerprint_kalman_smooth(y_geom_val, val_seq, process_noise=Q, measurement_noise=R)
                mae  = np.abs(y_val - y_sm).mean()
                if mae < best_mae:
                    best_mae, best_q, best_r = mae, Q, R
        print(f"  Tuned Kalman Q={best_q}, R={best_r} (val MAE={best_mae:.3f}m)")
    else:
        best_q, best_r = KALMAN_Q, KALMAN_R

    y_final = fingerprint_kalman_smooth(y_geom, test_seq, process_noise=best_q, measurement_noise=best_r)

    def report(y_pred, label):
        err   = np.abs(y_test - y_pred)
        mae   = err.mean()
        acc_1m = (err <= 1.0).mean() * 100
        acc_2m = (err <= 2.0).mean() * 100
        acc_5m = (err <= 5.0).mean() * 100
        fl_cls = (fl_tta.flatten() >= 0.5).astype(int)
        fl_acc = (fl_cls == fl_test).mean() * 100
        print(f"\n  [{label}]")
        print(f"    Y MAE : {mae:.3f}m")
        print(f"    Acc@1m: {acc_1m:.1f}%  |  Acc@2m: {acc_2m:.1f}%  |  Acc@5m: {acc_5m:.1f}%")
        print(f"    Floor Accuracy : {fl_acc:.1f}%")
        return mae, acc_1m, acc_2m, acc_5m, fl_acc, err

    report(y_nn_raw,  'NN raw')
    report(y_nn_tta, 'NN + TTA')
    report(y_hybrid,  'NN + TTA + kNN')
    mae_final, acc1, acc2, acc5, fl_acc, err_final = report(y_final, 'NN + TTA + kNN + Kalman')

    print(f"\n  ── Error Analysis ──")
    print(f"    Median Y Error : {np.median(err_final):.3f}m")
    print(f"    90th percentile: {np.percentile(err_final, 90):.3f}m")
    print(f"    95th percentile: {np.percentile(err_final, 95):.3f}m")
    print(f"    Max Y Error    : {err_final.max():.3f}m")
    for floor in [3, 4]:
        mask = np.array([s['floor_class'] for s in test_seq]) == FLOOR_CLASSES[floor]
        if mask.sum() > 0:
            print(f"    Floor {floor} MAE  : {err_final[mask].mean():.3f}m (n={mask.sum()})")

    pd.DataFrame([{
        'model': 'hybrid_gat_kalman', 'mae_final': mae_final,
        'acc_1m': acc1, 'acc_2m': acc2, 'acc_5m': acc5,
        'floor_acc': fl_acc, 'kalman_Q': best_q, 'kalman_R': best_r, 'nn_weight': best_w,
    }]).to_csv(EVAL_FILE_PATH, index=False)

    return {
        'model': 'hybrid_gat_kalman', 'mae_final': mae_final,
        'acc_1m': acc1, 'acc_2m': acc2, 'acc_5m': acc5, 'floor_acc': fl_acc,
        'y_pred': y_final, 'y_err': err_final,
    }


# ==============================================================================
# REAL-TIME (CAUSAL) EVALUATION
# ==============================================================================
def evaluate_realtime(model_wrapper, test_seq, y_stats, train_seq, val_seq=None,
                      kalman_q=KALMAN_Q, kalman_r=KALMAN_R,
                      outlier_z=3.0, outlier_window=10):
    y_mean, y_std = y_stats
    X_seq_test   = np.array([s['X']          for s in test_seq], dtype=np.float32)
    X_graph_test = np.array([s['X_graph']    for s in test_seq], dtype=np.float32)
    y_test       = np.array([s['y']          for s in test_seq], dtype=np.float32)
    fl_test      = np.array([s['floor_class'] for s in test_seq], dtype=np.float32)

    out     = model_wrapper.model.predict([X_seq_test, X_graph_test], batch_size=BATCH_SIZE, verbose=0)
    y_nn    = out['y_output'][:, 0] * y_std + y_mean
    y_nn    = np.clip(y_nn, Y_MIN, Y_MAX)
    fl_prob = out['floor_output'][:, 0]
    fl_cls  = (fl_prob >= 0.5).astype(int)

    y_mad = apply_mad_per_trajectory(y_nn, test_seq, z_thresh=outlier_z)

    fp_data  = defaultdict(list)
    for i, s in enumerate(test_seq):
        fp_data[(s['session'], s['fingerprintId'])].append(i)
    sessions = defaultdict(list)
    for key, indices in fp_data.items():
        sessions[key[0]].append({'indices': indices, 'ts': test_seq[indices[0]]['timestamp']})

    y_causal = y_mad.copy()
    for session, items in sessions.items():
        if len(items) < 2: continue
        items.sort(key=lambda x: x['ts'])
        kf = OneDKalmanFilter(Q=kalman_q, R=kalman_r)
        for it in items:
            for idx in it['indices']:
                y_causal[idx] = kf.update(y_mad[idx])

    y_final = geometric_postprocess(y_causal, test_seq, train_seq, y_stats)

    def report(y_pred, label):
        err   = np.abs(y_test - y_pred)
        mae   = err.mean()
        acc_1m = (err <= 1.0).mean() * 100
        acc_2m = (err <= 2.0).mean() * 100
        acc_5m = (err <= 5.0).mean() * 100
        fl_acc = (fl_cls == fl_test).mean() * 100
        print(f"\n  [{label}]")
        print(f"    Y MAE : {mae:.3f}m")
        print(f"    Acc@1m: {acc_1m:.1f}%  |  Acc@2m: {acc_2m:.1f}%  |  Acc@5m: {acc_5m:.1f}%")
        print(f"    Floor Accuracy : {fl_acc:.1f}%")
        return mae, acc_1m, acc_2m, acc_5m, fl_acc, err

    mae_final, acc1, acc2, acc5, fl_acc, err_final = report(y_final, 'REAL-TIME (Causal + MAD + KF)')

    print(f"\n  ── Real-Time Error Analysis ──")
    print(f"    Median Y Error : {np.median(err_final):.3f}m")
    print(f"    90th percentile: {np.percentile(err_final, 90):.3f}m")
    print(f"    95th percentile: {np.percentile(err_final, 95):.3f}m")
    print(f"    Max Y Error    : {err_final.max():.3f}m")
    for floor in [3, 4]:
        mask = np.array([s['floor_class'] for s in test_seq]) == FLOOR_CLASSES[floor]
        if mask.sum() > 0:
            print(f"    Floor {floor} MAE  : {err_final[mask].mean():.3f}m (n={mask.sum()})")

    return {
        'model': 'realtime_causal', 'mae_final': mae_final,
        'acc_1m': acc1, 'acc_2m': acc2, 'acc_5m': acc5, 'floor_acc': fl_acc,
        'y_pred': y_final, 'y_err': err_final,
    }


# ==============================================================================
# MAIN
# ==============================================================================
def main(realtime_mode=False):
    print("\n" + "=" * 70)
    print("BEACON INDOOR LOCALIZATION — GAT + kNN + KALMAN + PRETRAINING")
    print("=" * 70)
    if realtime_mode:
        print("  [REAL-TIME MODE: causal Kalman + MAD outlier rejection]")

    # ── STEP 1: LOAD ───────────────────────────────────────────────────────────
    print("\n── STEP 1: LOAD ──────────────────────────────────────────────────")
    df = load_all_datasets(data_dir='/content/drive/MyDrive/beacons_chaos/data')

    # ── STEP 2: MINIMAL CLEAN ──────────────────────────────────────────────────
    print("\n── STEP 2: MINIMAL CLEAN ─────────────────────────────────────────")
    df = minimal_clean(df)
    print(f"  NUM_BEACONS = {NUM_BEACONS} (fixed)")

    # ── STEP 3: SPLIT ──────────────────────────────────────────────────────────
    print("\n── STEP 3: SPLIT ─────────────────────────────────────────────────")
    unique_sessions = df[['session_name', 'fingerprintId']].drop_duplicates()
    shuffled = unique_sessions.sample(frac=1, random_state=42).reset_index(drop=True)

    n      = len(shuffled)
    n_test = int(n * TEST_SPLIT)
    n_val  = int(n * VAL_SPLIT)

    test_keys_df  = shuffled.iloc[:n_test]
    val_keys_df   = shuffled.iloc[n_test:n_test + n_val]
    train_keys_df = shuffled.iloc[n_test + n_val:]

    train_keys_set = set(zip(train_keys_df['session_name'], train_keys_df['fingerprintId']))
    val_keys_set   = set(zip(val_keys_df['session_name'],   val_keys_df['fingerprintId']))
    test_keys_set  = set(zip(test_keys_df['session_name'],  test_keys_df['fingerprintId']))

    train_df = df[df.apply(lambda r: (r['session_name'], r['fingerprintId']) in train_keys_set, axis=1)].copy()
    val_df   = df[df.apply(lambda r: (r['session_name'], r['fingerprintId']) in val_keys_set,   axis=1)].copy()
    test_df  = df[df.apply(lambda r: (r['session_name'], r['fingerprintId']) in test_keys_set,  axis=1)].copy()

    print(f"  Rows: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

    # ── STEP 4: TRAIN-ONLY STATISTICS ──────────────────────────────────────────
    print("\n── STEP 4: TRAIN-ONLY STATISTICS ─────────────────────────────────")
    beacon_median_train = train_df.groupby('beaconId')['rssi'].median().to_dict()

    floor_median_train = {}
    for (bid, flvl), grp in train_df.groupby(['beaconId', 'floorLevel'])['rssi']:
        floor_median_train[(bid, flvl)] = grp.median()

    sensor_cols = [
        'gyroX', 'gyroY', 'gyroZ', 'accelX', 'accelY', 'accelZ',
        'userAccelX', 'userAccelY', 'userAccelZ',
        'magX', 'magY', 'magZ', 'pitch', 'roll', 'yaw'
    ]
    sensor_medians = {
        col: float(train_df[col].median())
        for col in sensor_cols if col in train_df.columns
    }
    print(f"  Beacon medians: {len(beacon_median_train)}, "
          f"Floor-beacon medians: {len(floor_median_train)}")

    # ── STEP 5: PREPROCESS ─────────────────────────────────────────────────────
    print("\n── STEP 5: PREPROCESS ────────────────────────────────────────────")
    train_df = preprocess_data(train_df, sensor_medians=sensor_medians)
    val_df   = preprocess_data(val_df,   sensor_medians=sensor_medians)
    test_df  = preprocess_data(test_df,  sensor_medians=sensor_medians)

    # ── STEP 6: ENGINEER FEATURES ──────────────────────────────────────────────
    print("\n── STEP 6: ENGINEER FEATURES ─────────────────────────────────────")
    train_df = engineer_features(train_df, beacon_median_map=beacon_median_train,
                                           floor_median_map=floor_median_train)
    val_df   = engineer_features(val_df,   beacon_median_map=beacon_median_train,
                                           floor_median_map=floor_median_train)
    test_df  = engineer_features(test_df,  beacon_median_map=beacon_median_train,
                                           floor_median_map=floor_median_train)

    df_final = pd.concat([train_df, val_df, test_df], ignore_index=True)
    finalise_feature_cols(df_final)

    assert not df_final[FEATURE_COLS].isna().any().any(), "NaN in engineered features!"
    assert not np.isinf(df_final[FEATURE_COLS].values).any(),  "Inf in engineered features!"

    all_sequences = create_sequences(df_final, window_size=WINDOW_SIZE,
                                     stride=WINDOW_STRIDE, require_labels=True)

    train_seq, val_seq, test_seq = [], [], []
    for seq in all_sequences:
        key = (seq['session'], seq['fingerprintId'])
        if key in test_keys_set:
            test_seq.append(seq)
        elif key in val_keys_set:
            val_seq.append(seq)
        elif key in train_keys_set:
            train_seq.append(seq)

    print(f"\n  Sequences: train={len(train_seq)}, val={len(val_seq)}, test={len(test_seq)}")

    # ── STEP 7: NORMALIZE ──────────────────────────────────────────────────────
    print("\n── STEP 7: NORMALIZE ─────────────────────────────────────────────")
    train_seq, val_seq, test_seq, y_stats, scaler, scaler_g = \
        normalize_sequences(train_seq, val_seq, test_seq)

    # ── STEP 8: CLEAR STALE CACHE & BUILD MODELS ───────────────────────────────
    print("\n── STEP 8: BUILD MODELS ──────────────────────────────────────────")
    clear_stale_cache(N_FEATURES)
    tf.keras.backend.clear_session()

    # ── STEP 9: SELF-SUPERVISED PRETRAINING (NEW) ──────────────────────────────
    print("\n── STEP 9: SELF-SUPERVISED PRETRAINING (Temporal Branch) ─────────")

    traj_df_raw = load_trajectory_dataset(TRAJECTORY_DATA_DIR)
    pretrain_wrapper = None

    if traj_df_raw is not None and len(traj_df_raw) > 0:
        print("  Cleaning trajectory data …")
        traj_df = minimal_clean_trajectory(traj_df_raw)

        print("  Preprocessing trajectory data …")
        traj_df = preprocess_data(traj_df, sensor_medians=sensor_medians)

        print("  Engineering features on trajectory data …")
        traj_df = engineer_features(traj_df,
                                    beacon_median_map=beacon_median_train,
                                    floor_median_map=floor_median_train)

        # Keep only feature columns present in both labeled and trajectory data
        keep_cols = FEATURE_COLS + [c for c in
                    ['session_name', 'fingerprintId', 'windowIndex', 'capturedAt',
                     'beaconId', 'floorLevel']
                    if c in traj_df.columns]
        traj_df = traj_df[[c for c in traj_df.columns if c in keep_cols]]

        # Fill any missing FEATURE_COLS with 0 (trajectory may lack some labeled columns)
        for col in FEATURE_COLS:
            if col not in traj_df.columns:
                traj_df[col] = 0.0

        print("  Creating trajectory sequences (no labels required) …")
        traj_seq = create_sequences(traj_df, window_size=WINDOW_SIZE,
                                    stride=WINDOW_STRIDE, require_labels=False)

        if len(traj_seq) > 0:
            print(f"  Normalizing {len(traj_seq)} trajectory sequences …")
            # Use supervised scaler (train-only statistics — no leakage)
            traj_seq = normalize_trajectory_sequences(traj_seq, scaler)

            pretrainer = build_temporal_pretrainer(name='temporal_pretrainer')
            pretrainer.summary()
            pretrain_wrapper = PretrainModel(pretrainer)
            pretrain_wrapper.fit(traj_seq, epochs=PRETRAIN_EPOCHS)
        else:
            print("  WARNING: no valid trajectory sequences — skipping pretraining.")
    else:
        print("  No trajectory data found — skipping pretraining.")

    # ── STEP 10: BUILD & FINE-TUNE LOCALIZER ───────────────────────────────────
    print("\n── STEP 10: SUPERVISED FINE-TUNING ───────────────────────────────")
    localizer    = build_hybrid_localizer(name='hybrid_gat')
    localizer.summary()
    custom_model = CustomModel(name='hybrid_gat', model=localizer)

    # Transfer temporal backbone weights if pretraining succeeded
    if pretrain_wrapper is not None and pretrain_wrapper.trained and not custom_model.trained:
        print("\n  Transferring temporal backbone weights: pretrainer → localizer …")
        transfer_temporal_weights(pretrain_wrapper.model, localizer)
    elif custom_model.trained:
        print("\n  Localizer already trained — skipping weight transfer.")
    else:
        print("\n  No pretrained weights — training localizer from scratch.")

    # ── STEP 11: GENERATORS ────────────────────────────────────────────────────
    print("\n── STEP 11: GENERATORS ───────────────────────────────────────────")
    train_gen = MultiTaskGenerator(train_seq, batch_size=BATCH_SIZE, shuffle=True,  augment=True)
    val_gen   = MultiTaskGenerator(val_seq,   batch_size=BATCH_SIZE, shuffle=False, augment=False)

    (Bx_seq, Bx_graph), By = train_gen[0]
    print(f"  Batch: seq{Bx_seq.shape} | graph{Bx_graph.shape} "
          f"| y{By['y_output'].shape} | fl{By['floor_output'].shape}")

    # ── STEP 12: TRAIN ─────────────────────────────────────────────────────────
    print("\n── STEP 12: TRAIN ────────────────────────────────────────────────")
    custom_model.fit(train_gen, val_gen, epochs=400)

    # ── STEP 13: EVALUATE ──────────────────────────────────────────────────────
    print("\n── STEP 13: EVALUATE ─────────────────────────────────────────────")
    if realtime_mode:
        result = evaluate_realtime(custom_model, test_seq, y_stats, train_seq, val_seq=val_seq)
    else:
        result = evaluate_model(custom_model, test_seq, y_stats, train_seq, val_seq=val_seq)

    print("\n" + "=" * 70)
    print("DONE")
    print(f"  Final Y MAE : {result['mae_final']:.3f} m")
    print(f"  Acc@1m      : {result['acc_1m']:.1f} %")
    print(f"  Acc@2m      : {result['acc_2m']:.1f} %")
    print(f"  Floor       : {result['floor_acc']:.1f} %")
    print("=" * 70)

    return {
        'model':             custom_model,
        'scaler':            scaler,
        'scaler_g':          scaler_g,
        'y_stats':           y_stats,
        'beacon_median_map': beacon_median_train,
        'floor_median_map':  floor_median_train,
        'sensor_medians':    sensor_medians,
        'result':            result,
    }


# ==============================================================================
# ENTRY POINT
# ==============================================================================
if __name__ == '__main__':
    # Option A: Standard batch evaluation (RTS smoother, TTA, kNN)
    # artifacts = main(realtime_mode=False)

    # Option B: Real-time causal evaluation (fast, no future data)
    artifacts = main(realtime_mode=True)

    # Option C: Deploy RealTimeLocalizer for live streaming
    #
    # localizer = RealTimeLocalizer(
    #     model=artifacts['model'].model,
    #     scaler=artifacts['scaler'],
    #     scaler_g=aratifacts['scaler_g'],
    #     y_stats=artifacts['y_stats'],
    #     beacon_median_map=artifacts['beacon_median_map'],
    #     floor_median_map=artifacts['floor_median_map'],
    #     sensor_medians=artifacts['sensor_medians'],
    #     use_tta=False,
    #     use_knn=False,
    #     kalman_q=0.5,
    #     kalman_r=4.0,
    # )
    # y_smooth, floor = localizer.update(live_window_df)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

BEACON INDOOR LOCALIZATION — GAT + kNN + KALMAN + PRETRAINING
  [REAL-TIME MODE: causal Kalman + MAD outlier rejection]

── STEP 1: LOAD ──────────────────────────────────────────────────

  adhamFloor3.csv
  Loaded 18166 rows × 26 cols

  adhamFloor4.csv
  Loaded 12180 rows × 26 cols

  baselFloor3.csv
  Loaded 18082 rows × 26 cols

  baselFloor4.csv
  Loaded 15730 rows × 26 cols

  remonFloor3.csv
  Loaded 22020 rows × 26 cols

  remonFloor4.csv
  Loaded 21038 rows × 26 cols

COMBINED: 107216 rows

── STEP 2: MINIMAL CLEAN ─────────────────────────────────────────
  NUM_BEACONS = 7 (fixed)

── STEP 3: SPLIT ─────────────────────────────────────────────────
  Rows: train=74862, val=15793, test=16561

── STEP 4: TRAIN-ONLY STATISTICS ─────────────────────────────────
  Beacon medians: 6, Floor-beacon medians: 12

── STEP 5: PREPROCESS ───────────────────────

Model: "hybrid_gat"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 5, 41)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 5)         │          0 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tb_beacon_emb       │ (None, 5, 16)     │        112 │ lambda[0][0]      │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 5, 40)     │          0 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tb_concat           │ (None, 5, 56)     │          0 │ tb_beacon_emb[0]… │
│ (Concatenate)       │                   │            │ lambda_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tb_proj (Dense)     │ (None, 5, 128)    │      7,296 │ tb_concat[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tb_spatial_drop     │ (None, 5, 128)    │          0 │ tb_proj[0][0]     │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tb_mha_0            │ (None, 5, 128)    │     66,048 │ tb_spatial_drop[… │
│ (MultiHeadAttentio… │                   │            │ tb_spatial_drop[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 5, 128)    │          0 │ tb_spatial_drop[… │
│                     │                   │            │ tb_mha_0[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tb_ln_attn_0        │ (None, 5, 128)    │        256 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tb_ffn_a_0 (Dense)  │ (None, 5, 256)    │     33,024 │ tb_ln_attn_0[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tb_ffn_drop_0       │ (None, 5, 256)    │          0 │ tb_ffn_a_0[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tb_ffn_b_0 (Dense)  │ (None, 5, 128)    │     32,896 │ tb_ffn_drop_0[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 5, 128)    │          0 │ tb_ln_attn_0[0][… │
│                     │                   │            │ tb_ffn_b_0[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tb_ln_ffn_0         │ (None, 5, 128)    │        256 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tb_mha_1            │ (None, 5, 128)    │     66,048 │ tb_ln_ffn_0[0][0… │
│ (MultiHeadAttentio… │                   │            │ tb_ln_ffn_0[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 5, 128)    │          0 │ tb_ln_ffn_0[0][0… │
│                     │                   │            │ tb_mha_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tb_ln_attn_1        │ (None, 5, 128)    │        256 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 822,946 (3.14 MB)

 Trainable params: 821,410 (3.13 MB)

 Non-trainable params: 1,536 (6.00 KB)


  No pretrained weights — training localizer from scratch.

── STEP 11: GENERATORS ───────────────────────────────────────────
  Batch: seq(128, 5, 41) | graph(128, 7, 6) | y(128, 1) | fl(128,)

── STEP 12: TRAIN ────────────────────────────────────────────────
Epoch 1/400
409/409 ━━━━━━━━━━━━━━━━━━━━ 93s 104ms/step - floor_output_accuracy: 0.5218 - floor_output_loss: 0.7834 - loss: 0.6830 - y_output_loss: 0.6437 - y_output_mae_euclidean: 1.1332 - val_floor_output_accuracy: 0.6173 - val_floor_output_loss: 0.6531 - val_loss: 0.3567 - val_y_output_loss: 0.3243 - val_y_output_mae_euclidean: 0.7476 - learning_rate: 1.5000e-05
Epoch 2/400
409/409 ━━━━━━━━━━━━━━━━━━━━ 9s 21ms/step - floor_output_accuracy: 0.6027 - floor_output_loss: 0.6814 - loss: 0.4901 - y_output_loss: 0.4560 - y_output_mae_euclidean: 0.9053 - val_floor_output_accuracy: 0.7823 - val_floor_output_loss: 0.5039 - val_loss: 0.3428 - val_y_output_loss: 0.3175 - val_y_output_mae_euclidean: 0.7171 - learning_rate: 3.0000e-05
Epo

In [ ]:
"""
================================================================================
 Trajectory JSON -> time-aligned CSV for indoor-localization pretraining (Colab)
================================================================================
 Reads every *.json file in JSON_FOLDER (a Google Drive folder) and produces a
 single CSV where EACH ROW pairs one BLE/RSSI reading with the IMU motion state
 (and interpolated position) sampled at the SAME moment in time -- then segments
 each walk into time windows so the downstream masked-autoencoder pretrainer can
 group the rows into sequences.

 Row model
 ---------
 One row per BLE/RSSI reading. The IMU sample and interpolated (x, y) closest in
 time (shared per-walk `tMs` clock) are attached via merge_asof(nearest).

 Windowing (IMPORTANT for the pretraining model)
 -----------------------------------------------
 The localization model groups rows by (session_name, fingerprintId, windowIndex)
 to build training windows. If those columns are empty, pandas' groupby drops
 every row and pretraining silently produces zero sequences. So this script
 fills them:
     * fingerprintId : a unique integer per walk (a contiguous trajectory).
     * windowIndex   : a time bucket within the walk,
                       int((tMs - walk_start_tMs) / PRETRAIN_WINDOW_MS).
 PRETRAIN_WINDOW_MS = 1000 keeps each window ~1 s (a handful of beacons, small
 rel_time), which matches the scale of the labeled fingerprint windows.

 Output columns (fixed order):
     session_name fingerprintId windowIndex capturedAt beaconUid beaconId rssi
     x y floorLevel gyroX gyroY gyroZ accelX accelY accelZ magX magY magZ
     pitch roll yaw

 Note on beaconId: the localization model recomputes beaconId from beaconUid
 (BEACON_MAP) and ignores this column's value. It is emitted only to satisfy the
 fixed schema. See the delivery notes about a one-line loader tweak so the
 model's CSV reader does not alias beaconId onto beaconUid.

 Guarantees
 ----------
   * No BLE record is dropped; duplicates are kept.
   * Numbers/timestamps are preserved EXACTLY as text (parse_float=str /
     parse_int=str); only a throwaway numeric tMs copy is used for the time
     join and windowing.
   * Missing values are empty cells; corrupted files are skipped with a warning
     and execution continues.
================================================================================
"""

# ------------------------------------------------------------------ settings
# The ONLY path you need to change:
JSON_FOLDER = "/content/drive/MyDrive/trajectoryData"

# Window length (ms) used to bucket each walk into windowIndex groups for
# pretraining. 1000 ms is a good default; larger = fewer, longer windows.
PRETRAIN_WINDOW_MS = 1000
# ----------------------------------------------------------------------------

import glob
import json
import os

import pandas as pd
from google.colab import drive

# ---------------------------------------------------------------------------
# 1. Mount Google Drive
# ---------------------------------------------------------------------------
drive.mount("/content/drive", force_remount=False)

# ---------------------------------------------------------------------------
# 2. Constants
# ---------------------------------------------------------------------------
COLUMNS = [
    "session_name", "fingerprintId", "windowIndex", "capturedAt",
    "beaconUid", "beaconId", "rssi", "x", "y", "floorLevel",
    "gyroX", "gyroY", "gyroZ",
    "accelX", "accelY", "accelZ",
    "magX", "magY", "magZ",
    "pitch", "roll", "yaw",
]

IMU_COLS = [
    "gyroX", "gyroY", "gyroZ",
    "accelX", "accelY", "accelZ",
    "magX", "magY", "magZ",
    "pitch", "roll", "yaw",
]


# ---------------------------------------------------------------------------
# 3. Helpers
# ---------------------------------------------------------------------------
def to_cell(value):
    """JSON value -> exact CSV cell text (numbers stay verbatim; None -> '')."""
    if value is None:
        return ""
    if value is True:
        return "true"
    if value is False:
        return "false"
    return str(value)


def derive_beacon_id(beacon_uid):
    """beaconUid 'uuid:major:minor' -> 'major:minor' (fallback: whole uid)."""
    if not beacon_uid:
        return ""
    uid = str(beacon_uid)
    return uid.split(":", 1)[1] if ":" in uid else uid


def build_walk_rows(walk, session_name, session_floor, fingerprint_id,
                    window_ms=PRETRAIN_WINDOW_MS):
    """Time-aligned rows for one walk, anchored on BLE readings.

    fingerprint_id : unique integer identifying this walk.
    windowIndex    : time bucket within the walk (int((tMs - t0) / window_ms)).
    """
    ble = walk.get("bleReadings") or []
    if not ble:
        return []  # no rssi -> no aligned pretraining rows

    floor_level = to_cell(walk.get("floorLevel", session_floor))
    imu = walk.get("imuSamples") or []
    steps = walk.get("steps") or []

    # BLE anchor. _t is a numeric copy of tMs, used only for join + windowing.
    ble_df = pd.DataFrame({
        "_t": [float(r["tMs"]) for r in ble],
        "capturedAt": [to_cell(r.get("capturedAt")) for r in ble],
        "beaconUid": [to_cell(r.get("beaconUid")) for r in ble],
        "rssi": [to_cell(r.get("rssi")) for r in ble],
    }).sort_values("_t", kind="mergesort").reset_index(drop=True)

    # Nearest-in-time IMU sample per BLE reading.
    if imu:
        imu_df = pd.DataFrame({
            "_t": [float(r["tMs"]) for r in imu],
            **{c: [to_cell(r.get(c)) for r in imu] for c in IMU_COLS},
        }).sort_values("_t", kind="mergesort").reset_index(drop=True)
        merged = pd.merge_asof(ble_df, imu_df, on="_t", direction="nearest")
    else:
        merged = ble_df.copy()
        for c in IMU_COLS:
            merged[c] = ""

    # Nearest-in-time interpolated position (x, y) from steps.
    if steps:
        steps_df = pd.DataFrame({
            "_t": [float(r["tMs"]) for r in steps],
            "x": [to_cell(r.get("interpolatedX")) for r in steps],
            "y": [to_cell(r.get("interpolatedY")) for r in steps],
        }).sort_values("_t", kind="mergesort").reset_index(drop=True)
        merged = pd.merge_asof(
            merged.sort_values("_t", kind="mergesort"),
            steps_df, on="_t", direction="nearest",
        )
    else:
        merged["x"] = ""
        merged["y"] = ""

    # windowIndex = time bucket within this walk (integer, starts at 0).
    # Note: column name must NOT start with "_" (itertuples renames those).
    t0 = merged["_t"].min()
    merged["winidx"] = ((merged["_t"] - t0) // float(window_ms)).astype(int)
    merged = merged.drop(columns=["_t"])

    rows = []
    for r in merged.itertuples(index=False):
        d = r._asdict()
        rows.append({
            "session_name": session_name,
            "fingerprintId": fingerprint_id,        # unique per walk
            "windowIndex": int(d["winidx"]),        # time bucket within walk
            "capturedAt": d["capturedAt"],
            "beaconUid": d["beaconUid"],
            "beaconId": derive_beacon_id(d["beaconUid"]),
            "rssi": d["rssi"],
            "x": d.get("x", ""),
            "y": d.get("y", ""),
            "floorLevel": floor_level,
            **{c: d.get(c, "") for c in IMU_COLS},
        })
    return rows


def extract_rows_from_file(path, walk_id_start):
    """Parse one file -> (rows, n_walks). walk_id_start seeds fingerprintId.

    Raises on malformed JSON so the caller can skip the file.
    """
    with open(path, "r", encoding="utf-8") as fh:
        data = json.load(fh, parse_float=str, parse_int=str)

    session = data.get("session") or {}
    session_name = to_cell(session.get("name"))
    session_floor = session.get("floorLevel")

    rows = []
    walks = data.get("walks") or []
    for w_idx, walk in enumerate(walks):
        fingerprint_id = walk_id_start + w_idx
        rows.extend(build_walk_rows(walk, session_name, session_floor, fingerprint_id))
    return rows, len(walks)


# ---------------------------------------------------------------------------
# 4. Process every .json file in the folder
# ---------------------------------------------------------------------------
json_paths = sorted(glob.glob(os.path.join(JSON_FOLDER, "*.json")))

if not json_paths:
    raise FileNotFoundError(
        f"No .json files found in '{JSON_FOLDER}'. "
        "Check the JSON_FOLDER path at the top of the script."
    )

all_rows = []
files_processed = 0
files_skipped = 0
next_walk_id = 0          # global counter -> fingerprintId unique across files

for path in json_paths:
    filename = os.path.basename(path)
    print(f"Processing: {filename} ...", flush=True)
    try:
        rows, n_walks = extract_rows_from_file(path, next_walk_id)
    except Exception as exc:  # corrupted / unreadable -> skip, keep going
        files_skipped += 1
        print(f"  [SKIPPED] Could not process '{filename}': {exc}")
        continue

    files_processed += 1
    next_walk_id += n_walks
    all_rows.extend(rows)
    print(f"  -> {len(rows)} aligned records extracted ({n_walks} walk(s))")

# ---------------------------------------------------------------------------
# 5. Build the final DataFrame (fixed column order, duplicates preserved)
# ---------------------------------------------------------------------------
df = pd.DataFrame(all_rows, columns=COLUMNS, dtype=object)
df = df.fillna("")

# ---------------------------------------------------------------------------
# 6. Summary
# ---------------------------------------------------------------------------
print("\n================ SUMMARY ================")
print(f"Total files processed : {files_processed}")
print(f"Total files skipped   : {files_skipped}")
print(f"Total walks (windows) : {next_walk_id}")
print(f"Total records         : {len(df)}")
print(f"Final DataFrame shape : {df.shape}")
print("==========================================\n")

# ---------------------------------------------------------------------------
# 7. Save the CSV (locally and back into the Drive folder)
# ---------------------------------------------------------------------------
local_csv = "/content/output.csv"
drive_csv = os.path.join(JSON_FOLDER, "merged_trajectory_data.csv")

df.to_csv(local_csv, index=False)
print(f"Saved: {local_csv}")

df.to_csv(drive_csv, index=False)
print(f"Saved: {drive_csv}")

Mounted at /content/drive
Processing: trajectory-adham-cmrajgl9-floor-4-export.json ...
  -> 3330 aligned records extracted (2 walk(s))
Processing: trajectory-adham-cmrajsl9-floor-4-export.json ...
  -> 2838 aligned records extracted (2 walk(s))
Processing: trajectory-adham-cmrajz3z-floor-4-export.json ...
  -> 2823 aligned records extracted (2 walk(s))
Processing: trajectory-adham-session-cmrai7mm-floor-4-export.json ...
  -> 3444 aligned records extracted (2 walk(s))
Processing: trajectory-adham-third-floor-correct-2-cmralhlh-floor-3-export.json ...
  -> 2598 aligned records extracted (2 walk(s))
Processing: trajectory-adham-third-floor-correct-cmralbez-floor-3-export.json ...
  -> 2270 aligned records extracted (2 walk(s))
Processing: trajectory-remoniphone-cmraky0u-floor-3-export.json ...
  -> 5498 aligned records extracted (2 walk(s))
Processing: trajectory-remoniphone2-cmralbk2-floor-3-export.json ...
  -> 2450 aligned records extracted (2 walk(s))
Processing: trajectory-yousef-c

/tmp/ipykernel_399/3665407929.py:242: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna("")



================ SUMMARY ================
Total files processed : 15
Total files skipped   : 0
Total walks (windows) : 30
Total records         : 48396
Final DataFrame shape : (48396, 22)

Saved: /content/output.csv
Saved: /content/drive/MyDrive/trajectoryData/merged_trajectory_data.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
================================================================================
BEACON INDOOR LOCALIZATION — GRAPH ATTENTION + kNN HYBRID + KALMAN SMOOTHING
Leakage-free, train-only statistics, session/fingerprint split.

REAL-TIME UPDATE (v2):
  • Causal forward-only Kalman filter for streaming inference
  • RealTimeLocalizer + StreamingLocalizer classes for live deployment
  • Per-trajectory MAD outlier rejection (fixes 29 m jumps)
  • Fast prediction path: TTA & kNN optional/disabled by default
  • Batch vs. Real-Time evaluation modes in main()
================================================================================
"""

import os
import re
import sys
import json
import math
import shutil
import pickle
import warnings
from collections import defaultdict

# Windows consoles default to cp1252 and crash on the Unicode box-drawing chars used in the
# step banners. Force UTF-8 on stdout/stderr so the script runs without PYTHONUTF8=1.
for _stream in (sys.stdout, sys.stderr):
    try:
        _stream.reconfigure(encoding='utf-8')
    except Exception:
        pass

import chardet
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model, optimizers, callbacks
from tensorflow.keras.utils import Sequence
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
tf.keras.backend.clear_session()
tf.keras.backend.set_floatx('float32')


# ==============================================================================
# CONFIG
# ==============================================================================
BASE_DIR         = r'/content/drive/MyDrive/beacons_chaos'
CACHE_MODELS_DIR = os.path.join(BASE_DIR, 'train_cache_gat')
EVAL_FILE_PATH   = os.path.join(BASE_DIR, 'eval_df_gat.csv')
NORM_FILE_PATH   = os.path.join(BASE_DIR, 'normalization_gat.json')

os.makedirs(CACHE_MODELS_DIR, exist_ok=True)

BATCH_SIZE    = 128
VAL_SPLIT     = 0.15
TEST_SPLIT    = 0.15
WINDOW_SIZE   = 5
WINDOW_STRIDE = 1

N_FLOORS      = 2
FLOOR_CLASSES = {3: 0, 4: 1}
FLOOR_CLASSES_INV = {0: 3, 1: 4}

W_Y     = 1.0
W_FLOOR = 0.05

Y_MIN, Y_MAX = 0.0, 93.0
MIN_BEACONS_PER_WINDOW = 2   # lowered 4->2: heuristic informativeness filter, not a
                             # trilateration requirement (1-D regressor). >=4 rejects
                             # 5-28% of walking windows; >=2 holds ~99-100% (measured).

# Beacon graph parameters
NUM_BEACONS    = 7      # 0 = padding/unknown, 1..6 = real beacons
EMBED_DIM      = 16
N_BEACON_FEATS = 6      # presence, mean_rssi_norm, std_rssi_norm,
                        # mean_rssi_distance, count_ratio, mean_motion_magnitude

# kNN / Kalman defaults (tuned automatically on val set)
KNN_K       = 5
KALMAN_Q    = 0.1
KALMAN_R    = 2.0

# ------------------------------------------------------------------------------
# SELF-SUPERVISED PRETRAINING (masked reconstruction on trajectory walks)
# ------------------------------------------------------------------------------
# PRETRAIN=False -> supervised-only path (from-scratch baseline / ablation control):
#   no trajectory or SSL code runs, pipeline behaves exactly as the labeled-only version.
# PRETRAIN=True  -> pretrain the shared encoder on unlabeled trajectory JSON via masked
#   signal reconstruction, then fine-tune the full supervised model on the labeled CSVs.
PRETRAIN                = True
TRAJ_DATA_DIR           = r'/content/drive/MyDrive/beacons_chaos/trajectory_data'   # folder with the 15 trajectory JSON files
PRETRAIN_EPOCHS         = 100
PRETRAIN_WINDOW_SECONDS = 1.5     # time-bucket size for synthetic walk windows (~WINDOW_SIZE readings)
SSL_MASK_RATE           = 0.5     # fraction of (timestep x channel) entries masked for reconstruction
IMU_JOIN_TOLERANCE_MS   = 40      # as-of merge tolerance BLE<-nearest IMU sample
ENCODER_WEIGHTS_PATH    = os.path.join(CACHE_MODELS_DIR, 'ssl_encoder.weights.h5')

# Optional: only if known from architectural drawings, NOT estimated from data
BEACON_Y_POS = {
    # 1: 0.0,
    # 2: 18.0,
    # 3: 37.0,
    # 4: 56.0,
    # 5: 74.0,
    # 6: 93.0,
}

# ==============================================================================
# BEACON MAPPING
# ==============================================================================
BEACON_MAP = {
    'fda50693-a4e2-4fb1-afcf-c6eb07647825:1:1':         1,
    'fda50693-a4e2-4fb1-afcf-c6eb07647825:0:2':         2,
    'aaaaaaaa-aaaa-aaaa-aaaa-aaaaaaaaaaaa:0:3':          3,
    'fda50693-a4e2-4fb1-afcf-c6eb07647825:10065:26049': 4,
    'fda50693-a4e2-4fb1-afcf-c6eb07647825:1:5':         5,
    'fda50693-a4e2-4fb1-afcf-c6eb07647825:1:6':         6,
}
BEACON_FLOOR = {1: 3, 2: 3, 3: 3, 4: 4, 5: 4, 6: 4}


# ==============================================================================
# HELPERS
# ==============================================================================
def pkl_save(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)

def pkl_load(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

def json_save(obj, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=4)

def json_load(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def clear_stale_cache(expected_features):
    marker = os.path.join(CACHE_MODELS_DIR, '.feature_count')
    if os.path.exists(marker):
        with open(marker) as f:
            cached = int(f.read().strip())
        if cached != expected_features:
            print(f"  Cache stale ({cached} → {expected_features}). Clearing...")
            shutil.rmtree(CACHE_MODELS_DIR)
            os.makedirs(CACHE_MODELS_DIR, exist_ok=True)
    with open(marker, 'w') as f:
        f.write(str(expected_features))


# ==============================================================================
# DATA LOADING
# ==============================================================================
def inspect_raw_file(filepath, n_bytes=500):
    if not os.path.exists(filepath):
        return None, None
    with open(filepath, 'rb') as f:
        raw = f.read(n_bytes)
    enc = chardet.detect(raw)['encoding'] or 'utf-8'
    with open(filepath, 'r', encoding=enc, errors='replace') as f:
        first = f.readline()
    seps = {s: first.count(s) for s in ['\t', ',', ';', '|']}
    sep = max(seps, key=seps.get) if max(seps.values()) > 0 else '\t'
    return enc, sep

def robust_load_csv(filepath):
    encoding, sep = inspect_raw_file(filepath)
    df = None
    for kw in [
        {'encoding': encoding, 'sep': sep},
        {'encoding': 'utf-8-sig', 'sep': sep},
        {'encoding': 'utf-8-sig', 'sep': sep, 'engine': 'python'},
        {'encoding': 'latin-1', 'sep': sep},
    ]:
        try:
            df = pd.read_csv(filepath, **kw)
            print(f"  Loaded {len(df)} rows × {len(df.columns)} cols")
            break
        except Exception:
            continue
    if df is None:
        raise RuntimeError(f"Cannot load {filepath}")

    def clean(c):
        c = str(c).replace('\ufeff', '').strip()
        return ''.join(ch for ch in c if ch.isprintable() or ch.isspace()).strip()

    df.columns = [clean(c) for c in df.columns]

    seen, new_cols = {}, []
    for c in df.columns:
        seen[c] = seen.get(c, 0) + 1
        new_cols.append(c if seen[c] == 1 else f"{c}_dup{seen[c]}")
    df.columns = new_cols

    remap = {}
    for c in df.columns:
        cl = c.lower().replace(' ', '').replace('_', '')
        if cl in ('fingerprintid', 'fingerprint', 'fprintid'):
            remap[c] = 'fingerprintId'
        elif cl in ('sessionid', 'session'):
            remap[c] = 'sessionId'
        elif cl in ('floorlevel', 'floor', 'floorlvl'):
            remap[c] = 'floorLevel'
        elif cl in ('windowindex', 'windowidx', 'winindex'):
            remap[c] = 'windowIndex'
        elif cl in ('beaconuid', 'beaconid', 'beaconuuid'):
            remap[c] = 'beaconUid'
        elif cl in ('capturedat', 'timestamp', 'time', 'datetime'):
            remap[c] = 'capturedAt'
    if remap:
        df.rename(columns=remap, inplace=True)
    return df

def load_all_datasets(data_dir='labelled_data'):
    # Filenames match the actual files in labelled_data/ (note the source typos
    # "remonFloro4" and the " (2)" suffix — kept as-is to avoid renaming raw data).
    file_mapping = {
        'adhamFloor3.csv':     {'person': 'adham', 'floor': 3},
        'adhamFloor4.csv': {'person': 'adham', 'floor': 4},
        'baselFloor3.csv':     {'person': 'basel', 'floor': 3},
        'baselFloor4.csv':     {'person': 'basel', 'floor': 4},
        'remonFloor3.csv':     {'person': 'remon', 'floor': 3},
        'remonFloor4.csv':     {'person': 'remon', 'floor': 4},
    }
    all_dfs = []
    for fname, info in file_mapping.items():
        fpath = os.path.join(data_dir, fname)
        if not os.path.exists(fpath):
            print(f"  WARNING: {fname} not found")
            continue
        print(f"\n  {fname}")
        df = robust_load_csv(fpath)
        df['person'] = info['person']
        df['floor'] = info['floor']
        df['session_name'] = f"{info['person']}_floor{info['floor']}"
        all_dfs.append(df)

    if not all_dfs:
        raise RuntimeError("No input files found.")

    combined = pd.concat(all_dfs, ignore_index=True)
    print(f"\nCOMBINED: {len(combined)} rows")
    return combined


# ==============================================================================
# TRAJECTORY LOADING  (unlabeled walks -> CSV-schema rows for SSL pretraining)
# ==============================================================================
# The trajectory JSON is structured completely differently from the labeled CSV:
#   session -> walks[] -> {steps[], imuSamples[], bleReadings[], checkpoints[]}
# bleReadings and imuSamples are SEPARATE asynchronous streams and there is no
# fingerprintId / windowIndex / sessionId on the readings. This loader flattens each
# walk into the SAME flat schema the labeled CSV uses, so every downstream function
# (minimal_clean / preprocess_data / engineer_features / create_sequences) is reused
# verbatim. Position labels are NOT used (SSL is label-free); y is a dummy.
IMU_COLS_JSON = [
    'gyroX', 'gyroY', 'gyroZ', 'accelX', 'accelY', 'accelZ',
    'userAccelX', 'userAccelY', 'userAccelZ',
    'magX', 'magY', 'magZ', 'pitch', 'roll', 'yaw',
]

def load_trajectory_json(filepath):
    """Flatten one trajectory JSON file into a CSV-schema DataFrame (one row per BLE reading)."""
    data = json_load(filepath)
    stem = os.path.splitext(os.path.basename(filepath))[0]
    session = data.get('session', {}) or {}
    floor_level = session.get('floorLevel')

    walk_frames = []
    for w_ord, walk in enumerate(data.get('walks', [])):
        ble = walk.get('bleReadings', []) or []
        imu = walk.get('imuSamples', []) or []
        if not ble:
            continue

        ble_df = pd.DataFrame(ble)[['capturedAt', 'beaconUid', 'rssi']].copy()
        ble_df['capturedAt'] = pd.to_datetime(ble_df['capturedAt'], errors='coerce', utc=True)
        ble_df = ble_df.dropna(subset=['capturedAt']).sort_values('capturedAt').reset_index(drop=True)
        if ble_df.empty:
            continue

        # As-of merge: attach the nearest-in-time IMU sample to each BLE reading.
        if imu:
            imu_df = pd.DataFrame(imu)
            keep = ['capturedAt'] + [c for c in IMU_COLS_JSON if c in imu_df.columns]
            imu_df = imu_df[keep].copy()
            imu_df['capturedAt'] = pd.to_datetime(imu_df['capturedAt'], errors='coerce', utc=True)
            imu_df = imu_df.dropna(subset=['capturedAt']).sort_values('capturedAt').reset_index(drop=True)
            merged = pd.merge_asof(
                ble_df, imu_df, on='capturedAt', direction='nearest',
                tolerance=pd.Timedelta(milliseconds=IMU_JOIN_TOLERANCE_MS),
            )
        else:
            merged = ble_df.copy()

        # Ensure all IMU columns exist (missing -> NaN; existing imputation handles it).
        for col in IMU_COLS_JSON:
            if col not in merged.columns:
                merged[col] = np.nan

        # Reconstruct window boundaries from timestamps (no windowIndex exists in the JSON).
        t0 = merged['capturedAt'].min()
        rel_s = (merged['capturedAt'] - t0).dt.total_seconds()
        merged['windowIndex'] = np.floor(rel_s / PRETRAIN_WINDOW_SECONDS).astype(int)

        # Synthetic metadata mirroring the labeled CSV schema.
        merged['fingerprintId'] = f"{stem}_w{w_ord}"        # walk ordinal within the file
        merged['sessionId'] = stem
        merged['session_name'] = f"traj_{stem}"             # never collides with labeled sessions
        merged['floorLevel'] = floor_level
        merged['x'] = 1.0                                   # constant corridor coordinate
        merged['y'] = 0.0                                   # dummy label (unused by SSL)
        walk_frames.append(merged)

    if not walk_frames:
        return pd.DataFrame()
    return pd.concat(walk_frames, ignore_index=True)

def load_all_trajectories(data_dir=TRAJ_DATA_DIR):
    if not os.path.isdir(data_dir):
        raise RuntimeError(f"Trajectory dir not found: {data_dir}")
    files = sorted(f for f in os.listdir(data_dir) if f.lower().endswith('.json'))
    if not files:
        raise RuntimeError(f"No trajectory JSON files in {data_dir}")
    all_dfs = []
    for fname in files:
        df = load_trajectory_json(os.path.join(data_dir, fname))
        if not df.empty:
            print(f"  {fname}: {len(df)} BLE rows, "
                  f"{df['windowIndex'].nunique()} windows/walk-avg")
            all_dfs.append(df)
    combined = pd.concat(all_dfs, ignore_index=True)
    print(f"\nTRAJECTORY COMBINED: {len(combined)} rows from {len(all_dfs)} files")
    return combined


# ==============================================================================
# CLEANING  —  drop unknown beacons so NUM_BEACONS stays fixed
# ==============================================================================
def minimal_clean(df):
    df = df.copy()

    for col in ['pressure', 'relativeAltitude']:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)

    df['beaconId'] = df['beaconUid'].map(BEACON_MAP)
    unknown = df['beaconId'].isna()
    n_unknown = unknown.sum()
    if n_unknown:
        print(f"  WARNING: dropping {n_unknown} rows with unknown beacons")
        df = df[~unknown].copy()

    df['beaconId'] = df['beaconId'].astype(int)

    df['capturedAt'] = pd.to_datetime(df['capturedAt'], errors='coerce')
    df = df.dropna(subset=['capturedAt'])

    df.sort_values(['session_name', 'fingerprintId', 'windowIndex', 'capturedAt'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df


# ==============================================================================
# PREPROCESSING — TRAIN-ONLY IMPUTATION
# ==============================================================================
def preprocess_data(df, sensor_medians=None):
    df = df.copy()

    df['rel_time'] = df.groupby(['session_name', 'fingerprintId', 'windowIndex'])['capturedAt'].transform(
        lambda x: (x - x.min()).dt.total_seconds()
    )
    df['window_row_idx'] = df.groupby(['session_name', 'fingerprintId', 'windowIndex']).cumcount()

    df.sort_values(['session_name', 'fingerprintId', 'windowIndex', 'capturedAt'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    impute_cols = [
        'gyroX', 'gyroY', 'gyroZ', 'accelX', 'accelY', 'accelZ',
        'userAccelX', 'userAccelY', 'userAccelZ',
        'magX', 'magY', 'magZ', 'pitch', 'roll', 'yaw'
    ]

    for col in impute_cols:
        if col not in df.columns:
            continue
        if df[col].isna().any():
            df[col] = df.groupby(['session_name', 'fingerprintId', 'windowIndex'])[col].transform(
                lambda x: x.ffill().bfill()
            )
            if sensor_medians is not None and col in sensor_medians:
                df[col] = df[col].fillna(sensor_medians[col])
            else:
                df[col] = df[col].fillna(0)

    df['floor_class'] = df['floorLevel'].map(FLOOR_CLASSES).fillna(0).astype(int)
    return df


# ==============================================================================
# FEATURE ENGINEERING — NO FINGERPRINT MEMORIZATION
# ==============================================================================
BASE_SENSOR_COLS = [
    'beaconId', 'rssi', 'rel_time',
    'gyroX', 'gyroY', 'gyroZ', 'accelX', 'accelY', 'accelZ',
    'userAccelX', 'userAccelY', 'userAccelZ',
    'magX', 'magY', 'magZ', 'pitch', 'roll', 'yaw',
    'rssi_norm', 'rssi_inv', 'rssi_distance',
    'beacon_floor_match', 'window_progress',
    'rssi_diff_from_beacon_median',
    'beacon_signal_rank',
    'window_rssi_mean', 'window_rssi_std',
    'min_beacon_distance', 'beacon_weight',
    'rssi_vs_floor_median',
    'beacon_on_this_floor',
    'dominant_beacon_id',
    'rssi_diff_to_max', 'rssi_diff_to_2nd_max',
    'beacon_count_in_window',
    'motion_magnitude', 'motion_yaw_rate',
    'beacon_y_pos',
    'unique_beacon_count',
    'weighted_beacon_y',
    'strongest_beacon_y',
]

FEATURE_COLS = None
N_FEATURES = None

def engineer_features(df, beacon_median_map=None, floor_median_map=None):
    df = df.copy()

    df['rssi_norm'] = (df['rssi'] + 100) / 60
    df['rssi_inv'] = -df['rssi']

    TX_POWER, PATH_LOSS = -59, 2.0
    df['rssi_distance'] = 10 ** ((TX_POWER - df['rssi']) / (10 * PATH_LOSS))
    # FIX: Cap rssi_distance to prevent gradient explosions from anomalous RSSI
    df['rssi_distance'] = df['rssi_distance'].clip(upper=500)

    df['beacon_floor'] = df['beaconId'].map(BEACON_FLOOR).fillna(0).astype(int)
    df['beacon_on_this_floor'] = (df['beacon_floor'] == df['floorLevel']).astype(float)
    df['beacon_floor_match'] = df['beacon_on_this_floor']

    df['window_progress'] = df.groupby(['session_name', 'fingerprintId', 'windowIndex'])['window_row_idx'].transform(
        lambda x: x / max(int(x.max()), 1)
    )

    group_cols = ['session_name', 'fingerprintId', 'windowIndex']

    # Train-only beacon median reference
    if beacon_median_map is not None:
        df['beacon_median_rssi'] = df['beaconId'].map(beacon_median_map).fillna(df['rssi'])
        df['rssi_diff_from_beacon_median'] = df['rssi'] - df['beacon_median_rssi']
        df.drop(columns=['beacon_median_rssi'], inplace=True)
    else:
        df['rssi_diff_from_beacon_median'] = 0.0

    # Per-observation ranks within a timestamp group
    df['beacon_signal_rank'] = df.groupby(
        ['session_name', 'fingerprintId', 'windowIndex', 'capturedAt']
    )['rssi'].rank(ascending=False, method='dense')

    df['window_rssi_mean'] = df.groupby(group_cols)['rssi'].transform('mean')
    df['window_rssi_std'] = df.groupby(group_cols)['rssi'].transform('std').fillna(0)

    df['min_beacon_distance'] = df.groupby(
        ['session_name', 'fingerprintId', 'windowIndex', 'capturedAt']
    )['rssi_distance'].transform('min')

    df['beacon_weight'] = 1.0 / (df['rssi_distance'] + 0.1)

    # Train-only floor-beacon median reference
    if floor_median_map is not None:
        df['floor_beacon_median_rssi'] = df.apply(
            lambda row: floor_median_map.get((row['beaconId'], row['floorLevel']), row['rssi']),
            axis=1
        )
        df['rssi_vs_floor_median'] = df['rssi'] - df['floor_beacon_median_rssi']
        df.drop(columns=['floor_beacon_median_rssi'], inplace=True)
    else:
        df['rssi_vs_floor_median'] = 0.0

    # Dominant / second strongest beacons inside each timestamp group
    group_time = ['session_name', 'fingerprintId', 'windowIndex', 'capturedAt']
    idxmax = df.groupby(group_time)['rssi'].idxmax()
    dominant_map = df.loc[idxmax].set_index(group_time)['beaconId']
    df['dominant_beacon_id'] = df.set_index(group_time).index.map(dominant_map).fillna(0).astype(int)

    max_rssi_per_group = df.groupby(group_time)['rssi'].transform('max')
    df['rssi_diff_to_max'] = df['rssi'] - max_rssi_per_group

    df_sorted = df.sort_values(group_time + ['rssi'], ascending=[True, True, True, True, False])
    df_sorted['rssi_rank'] = df_sorted.groupby(group_time).cumcount()
    second_max_map = df_sorted[df_sorted['rssi_rank'] == 1].set_index(group_time)['rssi']
    df['second_max_rssi'] = df.set_index(group_time).index.map(second_max_map)
    df['second_max_rssi'] = df['second_max_rssi'].fillna(max_rssi_per_group)
    df['rssi_diff_to_2nd_max'] = df['rssi'] - df['second_max_rssi']
    df.drop(columns=['second_max_rssi'], inplace=True)

    df['beacon_count_in_window'] = df.groupby(group_cols)['beaconId'].transform('count')
    df['motion_magnitude'] = np.sqrt(df['accelX']**2 + df['accelY']**2 + df['accelZ']**2)
    df['motion_yaw_rate'] = df['gyroZ'].abs()

    if BEACON_Y_POS:
        df['beacon_y_pos'] = df['beaconId'].map(BEACON_Y_POS).fillna(-1)
    else:
        df['beacon_y_pos'] = -1

    df['unique_beacon_count'] = df.groupby(group_cols)['beaconId'].transform('nunique')

    # Optional geometric features from external beacon map only
    if BEACON_Y_POS:
        df['rssi_lin'] = 10 ** (df['rssi'] / 10.0)
        df['weighted_y_num'] = df['beacon_y_pos'] * df['rssi_lin']

        wy_num = df.groupby(group_cols)['weighted_y_num'].transform('sum')
        wy_den = df.groupby(group_cols)['rssi_lin'].transform('sum')
        df['weighted_beacon_y'] = wy_num / (wy_den + 1e-6)

        idxmax_rssi = df.groupby(group_time)['rssi'].idxmax()
        strongest_y_map = df.loc[idxmax_rssi].set_index(group_time)['beacon_y_pos']
        df['strongest_beacon_y'] = df.set_index(group_time).index.map(strongest_y_map).fillna(-1)

        df.drop(columns=['rssi_lin', 'weighted_y_num'], inplace=True)
    else:
        df['weighted_beacon_y'] = -1
        df['strongest_beacon_y'] = -1

    return df

def finalise_feature_cols(df):
    global FEATURE_COLS, N_FEATURES
    FEATURE_COLS = [c for c in BASE_SENSOR_COLS if c in df.columns]
    N_FEATURES = len(FEATURE_COLS)
    print(f"\n  Final feature set ({N_FEATURES}): {FEATURE_COLS[:6]} ... {FEATURE_COLS[-4:]}")
    return FEATURE_COLS


# ==============================================================================
# BEACON GRAPH FEATURE EXTRACTOR  (per window → fixed-size beacon nodes)
# ==============================================================================
def extract_beacon_graph_features(X, beacon_ids, num_beacons=NUM_BEACONS):
    """
    X          : (window_size, n_features)  — pre-normalisation sequence matrix
    beacon_ids : (window_size,)              — raw int beacon IDs (1..6)
    Returns    : (num_beacons, N_BEACON_FEATS)
    """
    feats = np.zeros((num_beacons, N_BEACON_FEATS), dtype=np.float32)

    rssi_norm_idx = FEATURE_COLS.index('rssi_norm')
    rssi_dist_idx = FEATURE_COLS.index('rssi_distance')
    motion_idx    = FEATURE_COLS.index('motion_magnitude') if 'motion_magnitude' in FEATURE_COLS else None

    b_ids = np.clip(beacon_ids.astype(int), 0, num_beacons - 1)

    for b in range(num_beacons):
        mask = b_ids == b
        if not mask.any():
            continue
        sub = X[mask]
        feats[b, 0] = 1.0                                           # presence
        feats[b, 1] = sub[:, rssi_norm_idx].mean()                  # mean rssi_norm
        feats[b, 2] = sub[:, rssi_norm_idx].std() if len(sub) > 1 else 0.0  # std
        feats[b, 3] = sub[:, rssi_dist_idx].mean()                  # mean distance
        feats[b, 4] = len(sub) / len(X)                           # count ratio
        if motion_idx is not None:
            feats[b, 5] = sub[:, motion_idx].mean()                 # mean motion

    return feats

def extract_rssi_signature(X, beacon_ids, num_beacons=NUM_BEACONS, fill=-2.0):
    """Single-vector RSSI signature for kNN lookup."""
    sig = np.full(num_beacons, fill, dtype=np.float32)
    rssi_norm_idx = FEATURE_COLS.index('rssi_norm')
    b_ids = np.clip(beacon_ids.astype(int), 0, num_beacons - 1)
    for b in range(num_beacons):
        mask = b_ids == b
        if mask.any():
            sig[b] = X[mask, rssi_norm_idx].mean()
    return sig

# ==============================================================================
# SEQUENCE CREATION  (batch + single-window for real-time)
# ==============================================================================
def create_sequences(df, window_size=WINDOW_SIZE, stride=WINDOW_STRIDE):
    feature_cols = FEATURE_COLS
    sequences = []
    grouped = df.groupby(['session_name', 'fingerprintId', 'windowIndex'])
    print(f"\n  Creating sequences from {len(grouped)} windows...")

    rejected = 0
    for (session, fp_id, w_idx), wdf in grouped:
        wdf = wdf.sort_values('capturedAt').reset_index(drop=True)
        n = len(wdf)

        if wdf['beaconId'].nunique() < MIN_BEACONS_PER_WINDOW:
            rejected += 1
            continue

        base = {
            'y': float(wdf['y'].iloc[0]),
            'x': float(wdf['x'].iloc[0]) if 'x' in wdf.columns else 0.0,
            'floor_class': int(wdf['floor_class'].iloc[0]),
            'session': session,
            'fingerprintId': fp_id,
            'windowIndex': w_idx,
            'timestamp': wdf['capturedAt'].min().timestamp() if 'capturedAt' in wdf.columns else 0.0,
        }

        feat = wdf[feature_cols].values.astype(np.float32)
        b_ids = wdf['beaconId'].values.astype(np.int32)

        if n < window_size:
            feat_pad = np.pad(feat, ((0, window_size - n), (0, 0)), mode='edge')
            b_ids_pad = np.pad(b_ids, (0, window_size - n), mode='edge')
            X_graph = extract_beacon_graph_features(feat_pad, b_ids_pad)
            sequences.append({**base, 'X': feat_pad, 'X_graph': X_graph, 'beacon_ids': b_ids_pad})
        else:
            for start in range(0, n - window_size + 1, stride):
                sub_feat = feat[start:start + window_size]
                sub_bids = b_ids[start:start + window_size]
                if len(np.unique(sub_bids)) < MIN_BEACONS_PER_WINDOW:
                    rejected += 1
                    continue
                X_graph = extract_beacon_graph_features(sub_feat, sub_bids)
                sequences.append({
                    **base,
                    'X': sub_feat,
                    'X_graph': X_graph,
                    'beacon_ids': sub_bids,
                })

    print(f"  Total sequences: {len(sequences)} (rejected {rejected} under-determined)")
    return sequences

def create_single_sequence(window_df):
    """Lightweight sequence extraction for a single live window (real-time)."""
    global FEATURE_COLS, N_FEATURES
    if FEATURE_COLS is None:
        finalise_feature_cols(window_df)

    feature_cols = FEATURE_COLS
    wdf = window_df.sort_values('capturedAt').reset_index(drop=True)
    n = len(wdf)

    if wdf['beaconId'].nunique() < MIN_BEACONS_PER_WINDOW:
        return None

    base = {
        'y': float(wdf['y'].iloc[0]) if 'y' in wdf.columns else 0.0,
        'x': float(wdf['x'].iloc[0]) if 'x' in wdf.columns else 0.0,
        'floor_class': int(wdf['floor_class'].iloc[0]) if 'floor_class' in wdf.columns else 0,
        'session': str(wdf['session_name'].iloc[0]) if 'session_name' in wdf.columns else 'live',
        'fingerprintId': int(wdf['fingerprintId'].iloc[0]) if 'fingerprintId' in wdf.columns else 0,
        'windowIndex': int(wdf['windowIndex'].iloc[0]) if 'windowIndex' in wdf.columns else 0,
        'timestamp': wdf['capturedAt'].min().timestamp() if 'capturedAt' in wdf.columns else 0.0,
    }

    feat = wdf[feature_cols].values.astype(np.float32)
    b_ids = wdf['beaconId'].values.astype(np.int32)

    if n < WINDOW_SIZE:
        feat = np.pad(feat, ((0, WINDOW_SIZE - n), (0, 0)), mode='edge')
        b_ids = np.pad(b_ids, (0, WINDOW_SIZE - n), mode='edge')

    X_graph = extract_beacon_graph_features(feat, b_ids)
    return {**base, 'X': feat, 'X_graph': X_graph, 'beacon_ids': b_ids}


# ==============================================================================
# SPLIT — BY SESSION + FINGERPRINT ONLY
# ==============================================================================
def split_by_session(sequences, val_ratio=0.15, test_ratio=0.15, seed=42):
    np.random.seed(seed)
    groups = defaultdict(list)

    for s in sequences:
        groups[(s['session'], s['fingerprintId'])].append(s)

    keys = list(groups.keys())
    np.random.shuffle(keys)

    n = len(keys)
    n_test = int(n * test_ratio)
    n_val = int(n * val_ratio)

    test_keys = keys[:n_test]
    val_keys = keys[n_test:n_test + n_val]
    train_keys = keys[n_test + n_val:]

    train_seq = [s for k in train_keys for s in groups[k]]
    val_seq   = [s for k in val_keys   for s in groups[k]]
    test_seq  = [s for k in test_keys  for s in groups[k]]

    print(f"\n  Split: train={len(train_seq)}, val={len(val_seq)}, test={len(test_seq)}")
    return train_seq, val_seq, test_seq


# ==============================================================================
# NORMALIZATION — TRAIN ONLY  (protect beaconId from scaling!)
# ==============================================================================
def normalize_sequences(train_seq, val_seq, test_seq):
    n_feat = N_FEATURES

    # --- temporal branch ---
    train_X = np.array([s['X'] for s in train_seq], dtype=np.float32)
    flat = train_X.reshape(-1, n_feat)
    scaler = StandardScaler()
    scaler.fit(flat)

    # CRITICAL: keep beaconId as integer index for Embedding layer
    bidx = FEATURE_COLS.index('beaconId')
    scaler.mean_[bidx] = 0.0
    scaler.scale_[bidx] = 1.0

    # --- graph branch ---
    train_Xg = np.array([s['X_graph'] for s in train_seq], dtype=np.float32)
    flat_g = train_Xg.reshape(-1, N_BEACON_FEATS)
    scaler_g = StandardScaler()
    scaler_g.fit(flat_g)

    # --- y ---
    train_y = np.array([s['y'] for s in train_seq], dtype=np.float32)
    y_mean = float(train_y.mean())
    y_std  = float(train_y.std())
    y_std  = max(y_std, 1e-6)

    for seq_list in [train_seq, val_seq, test_seq]:
        for s in seq_list:
            s['X'] = scaler.transform(s['X'].reshape(-1, n_feat)).reshape(WINDOW_SIZE, n_feat).astype(np.float32)
            s['X_graph'] = scaler_g.transform(s['X_graph'].reshape(-1, N_BEACON_FEATS)).reshape(NUM_BEACONS, N_BEACON_FEATS).astype(np.float32)
            s['y_norm'] = (s['y'] - y_mean) / y_std

    json_save({
        'feature_mean': scaler.mean_.tolist(),
        'feature_scale': scaler.scale_.tolist(),
        'feature_cols': FEATURE_COLS,
        'graph_mean': scaler_g.mean_.tolist(),
        'graph_scale': scaler_g.scale_.tolist(),
        'y_mean': y_mean,
        'y_std': y_std,
        'y_min': Y_MIN,
        'y_max': Y_MAX,
    }, NORM_FILE_PATH)

    print(f"  y∈[{train_y.min():.1f},{train_y.max():.1f}] μ={y_mean:.2f} σ={y_std:.2f}")
    return train_seq, val_seq, test_seq, (y_mean, y_std), scaler, scaler_g

def normalize_single_sequence(seq, scaler, scaler_g, y_mean, y_std):
    """Normalize a single live-window sequence in-place."""
    n_feat = N_FEATURES
    seq['X'] = scaler.transform(seq['X'].reshape(-1, n_feat)).reshape(WINDOW_SIZE, n_feat).astype(np.float32)
    seq['X_graph'] = scaler_g.transform(seq['X_graph'].reshape(-1, N_BEACON_FEATS)).reshape(NUM_BEACONS, N_BEACON_FEATS).astype(np.float32)
    seq['y_norm'] = (seq['y'] - y_mean) / max(y_std, 1e-6)
    return seq


# ==============================================================================
# GENERATOR  (dual input — tuple return required by TF)
# ==============================================================================
class MultiTaskGenerator(Sequence):
    def __init__(self, sequences, batch_size=BATCH_SIZE, shuffle=True, augment=False):
        self.sequences = sequences
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment
        self.indices = np.arange(len(sequences))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.sequences) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        X_seq   = np.array([self.sequences[i]['X'] for i in batch_idx], dtype=np.float32)
        X_graph = np.array([self.sequences[i]['X_graph'] for i in batch_idx], dtype=np.float32)
        y  = np.array([[self.sequences[i]['y_norm']] for i in batch_idx], dtype=np.float32)
        fl = np.array([self.sequences[i]['floor_class'] for i in batch_idx], dtype=np.float32)

        if self.augment:
            rssi_idx = FEATURE_COLS.index('rssi') if 'rssi' in FEATURE_COLS else 1
            # FIX: Protect beaconId (column 0) from corruption during augmentation
            X_seq[:, :, rssi_idx] += np.random.normal(0, 0.10, X_seq[:, :, rssi_idx].shape)

            # Dropout mask: shape (B,T,1) so it broadcasts to all feature channels
            mask = np.random.random((X_seq.shape[0], X_seq.shape[1])) > 0.12
            mask = mask[:, :, np.newaxis]          # (B, T, 1)

            # Apply mask only to non-beaconId columns (broadcasts correctly)
            X_seq[:, :, 1:] = X_seq[:, :, 1:] * mask

            # Gaussian noise: zero out channel 0 (beaconId)
            noise = np.random.normal(0, 0.015, X_seq.shape)
            noise[:, :, 0] = 0.0
            X_seq += noise
            # Ensure beaconId stays valid integer indices after any float drift
            X_seq[:, :, 0] = np.round(np.clip(X_seq[:, :, 0], 0, NUM_BEACONS - 1))

            # graph augment
            X_graph[:, :, 1] += np.random.normal(0, 0.08, X_graph[:, :, 1].shape)   # rssi_norm
            X_graph[:, :, 3] *= np.random.uniform(0.9, 1.1, X_graph[:, :, 3].shape)  # rssi_distance
            drop_mask = np.random.random((X_graph.shape[0], NUM_BEACONS)) > 0.05
            X_graph[:, :, 0] *= drop_mask.astype(np.float32)
            g_noise = np.random.normal(0, 0.01, X_graph.shape)
            X_graph += g_noise

        # TUPLE return is required for multi-input models in TF 2.x
        return (X_seq, X_graph), {'y_output': y, 'floor_output': fl}

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


# ==============================================================================
# LOSSES
# ==============================================================================
def logcosh_euclidean(y_true, y_pred):
    # FIX: Clip distance to prevent cosh() overflow → NaN weights
    dist = tf.norm(y_true - y_pred, axis=-1)
    dist = tf.clip_by_value(dist, 0.0, 50.0)
    return tf.reduce_mean(tf.math.log(tf.cosh(dist)))

def mae_euclidean(y_true, y_pred):
    return tf.reduce_mean(tf.norm(y_true - y_pred, axis=-1))


# ==============================================================================
# MODEL  —  Dual-branch: Temporal Transformer + Beacon Graph Attention
# ==============================================================================
def build_encoder(seq_inp, graph_inp):
    """
    Shared dual-branch encoder (temporal Transformer + beacon graph attention -> 128-dim
    `shared` embedding). Extracted so the SAME encoder can be reused by both the supervised
    localizer and the self-supervised pretrainer. Every weighted layer has an EXPLICIT name
    so pretrained weights transfer via `load_weights(by_name=True)`. The layer topology is
    byte-identical to the original inline version (order-based weight loading still works).
    """
    # -------------------------------------------------------------------------
    # TEMPORAL BRANCH  (window_size time steps)
    # -------------------------------------------------------------------------
    beacon_ids = layers.Lambda(lambda x: tf.cast(x[:, :, 0], tf.int32))(seq_inp)
    beacon_emb = layers.Embedding(NUM_BEACONS, EMBED_DIM, name='enc_t_beacon_emb')(beacon_ids)

    rest = layers.Lambda(lambda x: x[:, :, 1:])(seq_inp)
    x_t = layers.Concatenate(axis=-1)([beacon_emb, rest])

    x_t = layers.Dense(128, activation='gelu', name='enc_t_proj')(x_t)
    x_t = layers.SpatialDropout1D(0.15)(x_t)

    for i in range(4):
        attn = layers.MultiHeadAttention(num_heads=8, key_dim=16, dropout=0.15,
                                         name=f'enc_t_mha_{i}')(x_t, x_t)
        x_t = layers.LayerNormalization(name=f'enc_t_ln_a_{i}')(x_t + attn)

        ffn = layers.Dense(256, activation='gelu', name=f'enc_t_ffn0_{i}')(x_t)
        ffn = layers.Dropout(0.15)(ffn)
        ffn = layers.Dense(128, name=f'enc_t_ffn1_{i}')(ffn)
        x_t = layers.LayerNormalization(name=f'enc_t_ln_b_{i}')(x_t + ffn)

    avg_t = layers.GlobalAveragePooling1D()(x_t)
    max_t = layers.GlobalMaxPooling1D()(x_t)
    temporal_emb = layers.Concatenate()([avg_t, max_t])          # 256 dim

    # -------------------------------------------------------------------------
    # BEACON GRAPH BRANCH  (NUM_BEACONS nodes)
    # -------------------------------------------------------------------------
    def tile_beacon_indices(x):
        return tf.tile(tf.expand_dims(tf.range(NUM_BEACONS, dtype=tf.int32), 0),
                       [tf.shape(x)[0], 1])

    beacon_idx_layer = layers.Lambda(tile_beacon_indices)(graph_inp)
    graph_beacon_emb = layers.Embedding(NUM_BEACONS, EMBED_DIM, name='enc_g_beacon_emb')(beacon_idx_layer)

    x_g = layers.Concatenate(axis=-1)([graph_beacon_emb, graph_inp])   # (B, 7, 16+6)
    x_g = layers.Dense(64, activation='gelu', name='enc_g_proj')(x_g)

    for i in range(2):
        attn_g = layers.MultiHeadAttention(num_heads=4, key_dim=16, dropout=0.1,
                                           name=f'enc_g_mha_{i}')(x_g, x_g)
        x_g = layers.LayerNormalization(name=f'enc_g_ln_a_{i}')(x_g + attn_g)

        ffn_g = layers.Dense(128, activation='gelu', name=f'enc_g_ffn0_{i}')(x_g)
        ffn_g = layers.Dropout(0.1)(ffn_g)
        ffn_g = layers.Dense(64, name=f'enc_g_ffn1_{i}')(ffn_g)
        x_g = layers.LayerNormalization(name=f'enc_g_ln_b_{i}')(x_g + ffn_g)

    avg_g = layers.GlobalAveragePooling1D()(x_g)
    max_g = layers.GlobalMaxPooling1D()(x_g)
    graph_emb = layers.Concatenate()([avg_g, max_g])                   # 128 dim

    # -------------------------------------------------------------------------
    # FUSION
    # -------------------------------------------------------------------------
    fused = layers.Concatenate()([temporal_emb, graph_emb])            # 384 dim
    shared = layers.Dense(256, activation='gelu', name='enc_shared0')(fused)
    shared = layers.BatchNormalization(epsilon=1e-4, name='enc_shared_bn0')(shared)
    shared = layers.Dropout(0.35)(shared)
    shared = layers.Dense(128, activation='gelu', name='enc_shared1')(shared)
    shared = layers.BatchNormalization(epsilon=1e-4, name='enc_shared_bn1')(shared)
    shared = layers.Dropout(0.25)(shared)
    return shared


def build_hybrid_localizer(name='hybrid_gat'):
    # -------------------------------------------------------------------------
    # Inputs
    # -------------------------------------------------------------------------
    seq_inp   = layers.Input(shape=(WINDOW_SIZE, N_FEATURES), name='sequence_input')
    graph_inp = layers.Input(shape=(NUM_BEACONS, N_BEACON_FEATS), name='beacon_graph_input')

    # Shared encoder (pretrainable) -> 128-dim embedding
    shared = build_encoder(seq_inp, graph_inp)

    # --- Y branch ---
    y_branch = layers.Dense(256, activation='gelu')(shared)
    y_branch = layers.BatchNormalization(epsilon=1e-4)(y_branch)
    y_branch = layers.Dropout(0.25)(y_branch)
    y_branch = layers.Dense(128, activation='gelu')(y_branch)
    y_branch = layers.BatchNormalization(epsilon=1e-4)(y_branch)
    y_branch = layers.Dropout(0.15)(y_branch)
    y_branch = layers.Dense(64, activation='gelu')(y_branch)
    y_out = layers.Dense(1, activation='linear', name='y_output')(y_branch)

    # --- Floor branch ---
    fl = layers.Dense(64, activation='gelu')(shared)
    fl = layers.Dropout(0.1)(fl)
    fl_out = layers.Dense(1, activation='sigmoid', name='floor_output')(fl)

    model = Model(inputs=[seq_inp, graph_inp],
                  outputs={'y_output': y_out, 'floor_output': fl_out},
                  name=name)

    # FIX: Use clipnorm only (Keras forbids combining clipnorm + clipvalue)
    model.compile(
        optimizer=optimizers.Adam(1.5e-4, clipnorm=1.0),
        loss={
            'y_output': logcosh_euclidean,
            'floor_output': 'binary_crossentropy',
        },
        loss_weights={
            'y_output': W_Y,
            'floor_output': W_FLOOR,
        },
        metrics={
            'y_output': [mae_euclidean],
            'floor_output': ['accuracy'],
        }
    )
    return model


# ==============================================================================
# SELF-SUPERVISED PRETRAINING  (masked reconstruction on trajectory walks)
# ==============================================================================
def build_ssl_model(name='ssl_pretrainer'):
    """
    Same shared encoder as build_hybrid_localizer, with a lightweight reconstruction
    decoder instead of the y/floor heads. The encoder layer NAMES match, so after
    pretraining the encoder weights transfer to the supervised model via
    `load_weights(by_name=True, skip_mismatch=True)`. Pretext: reconstruct the (clean)
    temporal `X` and beacon-graph `X_graph` from their masked versions -> the encoder
    must learn RSSI / IMU / beacon structure without any position labels.
    """
    seq_inp   = layers.Input(shape=(WINDOW_SIZE, N_FEATURES), name='sequence_input')
    graph_inp = layers.Input(shape=(NUM_BEACONS, N_BEACON_FEATS), name='beacon_graph_input')

    shared = build_encoder(seq_inp, graph_inp)                  # 128-dim (encoder, shared names)

    # --- decoder (distinct names -> skipped by by_name transfer) ---
    h = layers.Dense(256, activation='gelu', name='dec_hidden')(shared)
    h = layers.Dropout(0.1)(h)

    seq_rec = layers.Dense(WINDOW_SIZE * N_FEATURES, name='dec_seq_dense')(h)
    seq_rec = layers.Reshape((WINDOW_SIZE, N_FEATURES), name='recon_seq')(seq_rec)

    graph_rec = layers.Dense(NUM_BEACONS * N_BEACON_FEATS, name='dec_graph_dense')(h)
    graph_rec = layers.Reshape((NUM_BEACONS, N_BEACON_FEATS), name='recon_graph')(graph_rec)

    model = Model(inputs=[seq_inp, graph_inp],
                  outputs={'recon_seq': seq_rec, 'recon_graph': graph_rec},
                  name=name)
    model.compile(
        optimizer=optimizers.Adam(1.5e-4, clipnorm=1.0),
        loss={'recon_seq': 'mse', 'recon_graph': 'mse'},
        loss_weights={'recon_seq': 1.0, 'recon_graph': 1.0},
    )
    return model


class SSLMaskGenerator(Sequence):
    """
    Yields ((X_masked, Xg_masked), {'recon_seq': X, 'recon_graph': Xg}).
    A random SSL_MASK_RATE fraction of (timestep x channel) entries is zeroed in the
    inputs; the target is the clean (unmasked) tensor. Column 0 of X (beaconId) is
    NEVER masked and is re-quantised to a valid embedding index, mirroring the existing
    augmenters so the Embedding lookup stays valid.
    """
    def __init__(self, sequences, batch_size=BATCH_SIZE, mask_rate=SSL_MASK_RATE, shuffle=True):
        self.sequences = sequences
        self.batch_size = batch_size
        self.mask_rate = mask_rate
        self.shuffle = shuffle
        self.indices = np.arange(len(sequences))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.sequences) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        X  = np.array([self.sequences[i]['X'] for i in batch_idx], dtype=np.float32)
        Xg = np.array([self.sequences[i]['X_graph'] for i in batch_idx], dtype=np.float32)

        Xm  = X.copy()
        Xgm = Xg.copy()

        # Temporal masking (protect beaconId at channel 0).
        seq_mask = (np.random.random(Xm.shape) > self.mask_rate).astype(np.float32)
        seq_mask[:, :, 0] = 1.0
        Xm = Xm * seq_mask
        Xm[:, :, 0] = np.round(np.clip(Xm[:, :, 0], 0, NUM_BEACONS - 1))

        # Graph masking (all channels of a node may be dropped).
        g_mask = (np.random.random(Xgm.shape) > self.mask_rate).astype(np.float32)
        Xgm = Xgm * g_mask

        return (Xm, Xgm), {'recon_seq': X, 'recon_graph': Xg}

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


def transfer_encoder_weights(dst_model, weights_path):
    """
    Copy pretrained ENCODER weights (layers named 'enc_*') from a saved SSL checkpoint into
    `dst_model` (the supervised localizer). Version-agnostic: works on both Keras 2 and Keras 3
    (Keras 3 removed `load_weights(by_name=...)`, so we rebuild the SSL model, full-load its
    weights, and copy matching encoder layers by name via set_weights).
    """
    src = build_ssl_model()
    src.load_weights(weights_path)
    src_layers = {l.name: l for l in src.layers}
    n = 0
    for l in dst_model.layers:
        if l.name.startswith('enc_') and l.name in src_layers:
            sw = src_layers[l.name].get_weights()
            if sw:
                l.set_weights(sw)
                n += 1
    print(f"  Transferred {n} pretrained encoder layers (by name) into the supervised model")
    return n


def transform_sequences(seqs, scaler, scaler_g, y_mean, y_std):
    """
    Apply the LABELED-TRAIN scaler to a list of raw sequences in place (batched version of
    normalize_single_sequence). The trajectory scaler is never fitted -> fine-tuning stays
    leakage-free. Reuses the exact per-sequence transform used by the supervised path.
    """
    for s in seqs:
        normalize_single_sequence(s, scaler, scaler_g, y_mean, y_std)
    return seqs


def pretrain_encoder(scaler, scaler_g, y_stats,
                     beacon_median_map, floor_median_map, sensor_medians):
    """
    Full SSL pretraining stage: trajectory JSON -> CSV-schema rows -> shared preprocessing
    -> masked-reconstruction training -> save encoder weights to ENCODER_WEIGHTS_PATH.
    Uses the same train-only statistics/scaler as the labeled pipeline (no leakage).
    """
    print("\n  Loading trajectory data for SSL pretraining...")
    traj_df = load_all_trajectories(TRAJ_DATA_DIR)
    traj_df = minimal_clean(traj_df)
    traj_df = preprocess_data(traj_df, sensor_medians=sensor_medians)
    traj_df = engineer_features(traj_df, beacon_median_map=beacon_median_map,
                                floor_median_map=floor_median_map)

    # Guard against NaN/Inf leaking into the encoder (same check as the supervised path).
    assert not traj_df[FEATURE_COLS].isna().any().any(), "NaN in trajectory features!"
    assert not np.isinf(traj_df[FEATURE_COLS].values).any(), "Inf in trajectory features!"

    traj_seq = create_sequences(traj_df, window_size=WINDOW_SIZE, stride=WINDOW_STRIDE)
    if len(traj_seq) == 0:
        print("  WARNING: no trajectory sequences produced; skipping pretraining.")
        return False

    transform_sequences(traj_seq, scaler, scaler_g, *y_stats)

    print(f"  Pretraining encoder on {len(traj_seq)} trajectory windows "
          f"({PRETRAIN_EPOCHS} epochs, mask_rate={SSL_MASK_RATE})...")
    ssl_gen = SSLMaskGenerator(traj_seq, batch_size=BATCH_SIZE, shuffle=True)
    ssl_model = build_ssl_model()
    ssl_cb = [
        callbacks.TerminateOnNaN(),
        callbacks.EarlyStopping(monitor='loss', patience=15, restore_best_weights=True),
    ]
    ssl_model.fit(ssl_gen, epochs=PRETRAIN_EPOCHS, callbacks=ssl_cb, verbose=1)
    ssl_model.save_weights(ENCODER_WEIGHTS_PATH)
    print(f"  Saved pretrained encoder -> {ENCODER_WEIGHTS_PATH}")
    return True


# ==============================================================================
# CALLBACKS
# ==============================================================================
class HistoryCheckpoint(callbacks.Callback):
    def __init__(self, name):
        super().__init__()
        self.path = os.path.join(CACHE_MODELS_DIR, f'{name}.history.json')
        self.history = json_load(self.path) if os.path.exists(self.path) else {}

    def on_epoch_end(self, epoch, logs=None):
        for k, v in (logs or {}).items():
            self.history.setdefault(k, []).append(float(v))
        json_save(self.history, self.path)

    def get(self):
        return self.history

def cosine_warmup(epoch, lr, warmup=10, total=400, base_lr=1.5e-4, min_lr=1e-6):
    if epoch < warmup:
        return base_lr * (epoch + 1) / warmup
    progress = (epoch - warmup) / max(total - warmup, 1)
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * progress))

class CustomModel:
    def __init__(self, name, model):
        self.name = re.sub(r'[^A-Za-z0-9]', '_', name).lower()
        self.weights = os.path.join(CACHE_MODELS_DIR, f'{self.name}.weights.h5')
        self.model = model
        self.history = {}
        self._load()

    def _load(self):
        if os.path.exists(self.weights):
            self.model.load_weights(self.weights)
            self.trained = True
            print(f'  [{self.name}] weights loaded')
        else:
            self.trained = False
        hcp = HistoryCheckpoint(self.name)
        self.history = hcp.get()

    def fit(self, train_gen, val_gen, epochs=400, force=False):
        if self.trained and not force:
            print(f'  [{self.name}] already trained, skipping')
            return
        hcp = HistoryCheckpoint(self.name)
        # FIX: Added TerminateOnNaN to stop immediately on first corrupted batch
        cb = [
            callbacks.TerminateOnNaN(),
            callbacks.EarlyStopping(monitor='val_loss', patience=35, restore_best_weights=True),
            callbacks.ModelCheckpoint(self.weights, monitor='val_loss', save_best_only=True, save_weights_only=True),
            callbacks.LearningRateScheduler(lambda e, lr: cosine_warmup(e, lr, total=epochs)),
            hcp,
        ]
        self.model.fit(train_gen, validation_data=val_gen, epochs=epochs, callbacks=cb, verbose=1)
        self.trained = True
        self.history = hcp.get()


# ==============================================================================
# kNN FINGERPRINT RETRIEVAL  (train-only database)
# ==============================================================================
class FingerprintKNN:
    def __init__(self, k=KNN_K):
        self.k = k
        self.nn = None
        self.X_sigs = None
        self.y_vals = None

    def fit(self, train_seq):
        fp_data = defaultdict(lambda: {'sigs': [], 'y': None, 'floor': None})
        for s in train_seq:
            key = (s['session'], s['fingerprintId'])
            sig = extract_rssi_signature(s['X'], s['beacon_ids'])
            fp_data[key]['sigs'].append(sig)
            fp_data[key]['y'] = s['y']
            fp_data[key]['floor'] = s['floor_class']

        keys = []
        X_sigs = []
        y_vals = []
        for key, data in fp_data.items():
            keys.append(key)
            X_sigs.append(np.mean(data['sigs'], axis=0))
            y_vals.append(data['y'])

        self.keys = keys
        self.X_sigs = np.array(X_sigs)
        self.y_vals = np.array(y_vals)

        k_eff = min(self.k, len(X_sigs))
        self.nn = NearestNeighbors(n_neighbors=k_eff, metric='euclidean')
        self.nn.fit(self.X_sigs)
        print(f"  kNN DB: {len(X_sigs)} fingerprints, k={k_eff}")

    def predict(self, test_seq):
        sigs = np.array([extract_rssi_signature(s['X'], s['beacon_ids']) for s in test_seq])
        k_eff = min(self.k, len(self.X_sigs))
        dist, idx = self.nn.kneighbors(sigs, n_neighbors=k_eff)

        w = 1.0 / (dist + 1e-6)
        w = w / w.sum(axis=1, keepdims=True)
        y_pred = np.sum(self.y_vals[idx] * w, axis=1)
        return y_pred


# ==============================================================================
# REAL-TIME KALMAN FILTER (Causal, Forward-Only)
# ==============================================================================
class OneDKalmanFilter:
    """Forward-only Kalman filter suitable for streaming / real-time use."""
    # FIX: Accept both Q/R and process_noise/measurement_noise to match all call sites
    def __init__(self, Q=None, R=None, process_noise=None, measurement_noise=None):
        if Q is not None:
            process_noise = Q
        if R is not None:
            measurement_noise = R
        self.Q = float(process_noise) if process_noise is not None else float(KALMAN_Q)
        self.R = float(measurement_noise) if measurement_noise is not None else float(KALMAN_R)
        self.x = 0.0
        self.P = 1.0
        self.initialized = False

    def reset(self):
        self.initialized = False
        self.x = 0.0
        self.P = 1.0

    def update(self, measurement):
        if not self.initialized:
            self.x = float(measurement)
            self.P = 1.0
            self.initialized = True
            return self.x

        # Predict
        x_pred = self.x
        P_pred = self.P + self.Q
        # Update
        K = P_pred / (P_pred + self.R)
        self.x = x_pred + K * (measurement - x_pred)
        self.P = (1.0 - K) * P_pred
        return self.x


# ==============================================================================
# BATCH KALMAN SMOOTHER (Forward-Backward RTS)
# ==============================================================================
class OneDKalmanSmoother:
    """Non-causal RTS smoother for offline post-processing."""
    def __init__(self, process_noise=None, measurement_noise=None, Q=None, R=None):
        if Q is not None:
            process_noise = Q
        if R is not None:
            measurement_noise = R
        self.Q = float(process_noise) if process_noise is not None else float(KALMAN_Q)
        self.R = float(measurement_noise) if measurement_noise is not None else float(KALMAN_R)

    def smooth(self, measurements):
        n = len(measurements)
        if n == 0:
            return []
        # Forward pass
        x = np.zeros(n)
        P = np.zeros(n)
        x[0] = float(measurements[0])
        P[0] = 1.0
        for t in range(1, n):
            x_pred = x[t-1]
            P_pred = P[t-1] + self.Q
            K = P_pred / (P_pred + self.R)
            x[t] = x_pred + K * (measurements[t] - x_pred)
            P[t] = (1.0 - K) * P_pred

        # Backward RTS pass
        x_s = x.copy()
        for t in range(n - 2, -1, -1):
            denom = P[t] + self.Q
            C = P[t] / denom if denom > 1e-9 else 0.0
            x_s[t] = x[t] + C * (x_s[t+1] - x[t])
        return x_s.tolist()


# ==============================================================================
# PER-TRAJECTORY MAD OUTLIER REJECTION
# ==============================================================================
class MADOutlierRejector:
    """Median-Absolute-Deviation outlier rejector for 1-D trajectories."""
    def __init__(self, window=10, z_thresh=3.0):
        self.window = window
        self.z_thresh = z_thresh
        self.history = []

    def reset(self):
        self.history = []

    def update(self, value):
        self.history.append(float(value))
        if len(self.history) > self.window:
            self.history.pop(0)
        if len(self.history) < 3:
            return value

        arr = np.array(self.history)
        med = np.median(arr)
        mad = np.median(np.abs(arr - med))
        if mad < 1e-6:
            return value

        z = np.abs(value - med) / (1.4826 * mad)
        if z > self.z_thresh:
            # Reject: return previous accepted value (or median if unavailable)
            return self.history[-2] if len(self.history) >= 2 else med
        return value

def apply_mad_per_trajectory(y_pred, test_seq, z_thresh=3.0):
    """Batch helper: apply MAD cleaning per fingerprint trajectory."""
    fp_data = defaultdict(list)
    for i, s in enumerate(test_seq):
        key = (s['session'], s['fingerprintId'])
        fp_data[key].append(i)

    y_clean = y_pred.copy()
    for indices in fp_data.values():
        if len(indices) < 5:
            continue
        vals = y_pred[indices]
        med = np.median(vals)
        mad = np.median(np.abs(vals - med))
        if mad < 1e-6:
            continue
        for idx in indices:
            z = np.abs(y_pred[idx] - med) / (1.4826 * mad)
            if z > z_thresh:
                y_clean[idx] = med
    return y_clean


# ==============================================================================
# REAL-TIME INFERENCE ENGINE
# ==============================================================================
class RealTimeLocalizer:
    """
    Production-grade real-time localizer.
    Processes single windows causally: no future data, no RTS backward pass.
    """
    def __init__(self, model, scaler, scaler_g, y_stats,
                 beacon_median_map, floor_median_map, sensor_medians,
                 use_tta=False, use_knn=False, knn_db=None,
                 kalman_q=KALMAN_Q, kalman_r=KALMAN_R,
                 outlier_z=3.0, outlier_window=10):
        self.model = model
        self.scaler = scaler
        self.scaler_g = scaler_g
        self.y_mean, self.y_std = y_stats
        self.y_stats = y_stats
        self.beacon_median_map = beacon_median_map
        self.floor_median_map = floor_median_map
        self.sensor_medians = sensor_medians
        self.use_tta = use_tta
        self.use_knn = use_knn
        self.knn_db = knn_db

        self.kalman = OneDKalmanFilter(Q=kalman_q, R=kalman_r)
        self.outlier_rejector = MADOutlierRejector(window=outlier_window, z_thresh=outlier_z)
        self.last_floor = None
        self.last_y = None

    def reset(self):
        self.kalman.reset()
        self.outlier_rejector.reset()
        self.last_floor = None
        self.last_y = None

    def _infer_floor_from_beacons(self, beacon_ids):
        floors = [BEACON_FLOOR.get(int(b), 0) for b in beacon_ids if int(b) in BEACON_FLOOR]
        if not floors:
            return self.last_floor if self.last_floor is not None else 3
        return int(max(set(floors), key=floors.count))

    def _prepare_window_df(self, window_df, assumed_floor=None):
        df = window_df.copy()

        # Ensure beaconId exists
        if 'beaconId' not in df.columns and 'beaconUid' in df.columns:
            df['beaconId'] = df['beaconUid'].map(BEACON_MAP)
        df = df.dropna(subset=['beaconId'])
        if len(df) == 0:
            return df
        df['beaconId'] = df['beaconId'].astype(int)

        # Infer floor if missing (needed for floor-dependent features)
        if 'floorLevel' not in df.columns or df['floorLevel'].isna().all():
            if assumed_floor is None:
                assumed_floor = self._infer_floor_from_beacons(df['beaconId'].values)
            df['floorLevel'] = assumed_floor

        # Ensure metadata columns for feature engineering
        if 'session_name' not in df.columns:
            df['session_name'] = 'live'
        if 'fingerprintId' not in df.columns:
            df['fingerprintId'] = 0
        if 'windowIndex' not in df.columns:
            df['windowIndex'] = 0
        if 'capturedAt' not in df.columns:
            df['capturedAt'] = pd.Timestamp.now()
        if 'y' not in df.columns:
            df['y'] = 0.0
        if 'x' not in df.columns:
            df['x'] = 0.0

        df = preprocess_data(df, sensor_medians=self.sensor_medians)
        df = engineer_features(df, beacon_median_map=self.beacon_median_map,
                               floor_median_map=self.floor_median_map)
        return df

    def predict_window(self, window_df, assumed_floor=None, return_floor=False):
        """Run neural network (and optional TTA/kNN) on a single live window."""
        df = self._prepare_window_df(window_df, assumed_floor)
        if len(df) == 0:
            return (None, None) if return_floor else None

        seq = create_single_sequence(df)
        if seq is None:
            return (None, None) if return_floor else None

        normalize_single_sequence(seq, self.scaler, self.scaler_g, self.y_mean, self.y_std)

        X_seq = np.expand_dims(seq['X'], 0)
        X_graph = np.expand_dims(seq['X_graph'], 0)

        # Optional kNN fallback (uses raw RSSI signature; disabled by default)
        y_knn = None
        if self.use_knn and self.knn_db is not None:
            sig = extract_rssi_signature(seq['X'], seq['beacon_ids'])
            sigs = np.expand_dims(sig, 0)
            k_eff = min(self.knn_db.k, len(self.knn_db.X_sigs))
            if k_eff > 0:
                dist, idx = self.knn_db.nn.kneighbors(sigs, n_neighbors=k_eff)
                w = 1.0 / (dist + 1e-6)
                w = w / w.sum(axis=1, keepdims=True)
                y_knn = np.sum(self.knn_db.y_vals[idx] * w, axis=1)[0]

        # Neural network prediction
        if self.use_tta:
            y_pred, fl_prob = predict_with_tta(self.model, X_seq, X_graph, self.y_stats, n_tta=10)
            y_pred = float(y_pred[0])
            fl_prob = float(fl_prob[0])
        else:
            out = self.model.predict([X_seq, X_graph], batch_size=1, verbose=0)
            y_pred = float(out['y_output'][0, 0] * self.y_std + self.y_mean)
            fl_prob = float(out['floor_output'][0, 0])

        y_pred = float(np.clip(y_pred, Y_MIN, Y_MAX))
        fl_pred = 1 if fl_prob >= 0.5 else 0
        fl_label = FLOOR_CLASSES_INV[fl_pred]

        # Sanity-check blend with kNN only if wildly divergent
        if y_knn is not None and abs(y_pred - y_knn) > 10.0:
            y_pred = y_knn

        self.last_floor = fl_label
        self.last_y = y_pred
        return (y_pred, fl_label) if return_floor else y_pred

    def update(self, window_df, assumed_floor=None):
        """
        Full real-time update: NN → outlier rejection → causal Kalman filter.
        Returns (y_smooth, floor_label).
        """
        y_raw, fl = self.predict_window(window_df, assumed_floor, return_floor=True)
        if y_raw is None:
            return None, None

        # Outlier rejection before Kalman (protects filter from spikes)
        y_clean = self.outlier_rejector.update(y_raw)

        # Causal Kalman filter
        y_smooth = self.kalman.update(y_clean)

        self.last_y = y_smooth
        return float(y_smooth), int(fl)


# ==============================================================================
# STREAMING LOCALIZER (sliding buffer for continuous sensor feeds)
# ==============================================================================
class StreamingLocalizer:
    """
    Wraps RealTimeLocalizer with a rolling buffer for continuous sensor streams.
    Accepts individual beacon observations and emits a prediction every N rows.
    """
    def __init__(self, localizer, window_size=WINDOW_SIZE, min_rows=4):
        self.localizer = localizer
        self.window_size = window_size
        self.min_rows = min_rows
        self.buffer = []
        self.obs_count = 0
        self.session_name = 'live_stream'
        self.fingerprint_id = 0
        self.window_index = 0

    def reset(self):
        self.buffer = []
        self.obs_count = 0
        self.window_index = 0
        self.localizer.reset()

    def add_observation(self, beacon_uid, rssi, timestamp=None,
                        gyro=None, accel=None, user_accel=None,
                        mag=None, pitch=None, roll=None, yaw=None,
                        floor_level=None):
        """Add a single beacon observation to the rolling buffer."""
        row = {
            'beaconUid': beacon_uid,
            'rssi': rssi,
            'capturedAt': pd.Timestamp(timestamp) if timestamp else pd.Timestamp.now(),
            'session_name': self.session_name,
            'fingerprintId': self.fingerprint_id,
            'windowIndex': self.window_index,
        }
        if floor_level is not None:
            row['floorLevel'] = floor_level
        if gyro is not None:
            row['gyroX'], row['gyroY'], row['gyroZ'] = gyro
        if accel is not None:
            row['accelX'], row['accelY'], row['accelZ'] = accel
        if user_accel is not None:
            row['userAccelX'], row['userAccelY'], row['userAccelZ'] = user_accel
        if mag is not None:
            row['magX'], row['magY'], row['magZ'] = mag
        if pitch is not None:
            row['pitch'] = pitch
        if roll is not None:
            row['roll'] = roll
        if yaw is not None:
            row['yaw'] = yaw

        self.buffer.append(row)
        self.obs_count += 1

        # Auto-advance window if buffer exceeds size
        if len(self.buffer) > self.window_size * 2:
            self.advance_window()

    def advance_window(self):
        """Advance to next window index; old buffer rows are discarded."""
        self.window_index += 1
        self.buffer = []
        self.fingerprint_id += 1

    def predict(self, force=False):
        """
        Emit prediction if buffer has enough rows.
        force=True forces prediction even with fewer rows.
        """
        if len(self.buffer) < self.min_rows and not force:
            return None, None
        if len(self.buffer) < self.min_rows:
            return None, None

        df = pd.DataFrame(self.buffer)
        y, fl = self.localizer.update(df)
        return y, fl


# ==============================================================================
# TEST-TIME AUGMENTATION (dual input)
# ==============================================================================
def predict_with_tta(model, X_seq, X_graph, y_stats, n_tta=20):
    y_mean, y_std = y_stats
    preds_y = []
    preds_fl = []

    rssi_idx = FEATURE_COLS.index('rssi') if 'rssi' in FEATURE_COLS else 1

    for i in range(n_tta):
        Xs_aug = X_seq.copy()
        Xg_aug = X_graph.copy()
        if i > 0:
            # Protect beaconId (column 0) from corruption during TTA
            Xs_aug[:, :, rssi_idx] += np.random.normal(0, 0.12, Xs_aug[:, :, rssi_idx].shape)

            mask = np.random.random((Xs_aug.shape[0], Xs_aug.shape[1])) > 0.15
            mask = mask[:, :, np.newaxis]          # (B, T, 1)

            # Apply mask only to non-beaconId columns (broadcasts correctly)
            Xs_aug[:, :, 1:] = Xs_aug[:, :, 1:] * mask

            noise = np.random.normal(0, 0.02, Xs_aug.shape)
            noise[:, :, 0] = 0.0
            Xs_aug += noise
            Xs_aug[:, :, 0] = np.round(np.clip(Xs_aug[:, :, 0], 0, NUM_BEACONS - 1))

            Xg_aug[:, :, 1] += np.random.normal(0, 0.08, Xg_aug[:, :, 1].shape)
            Xg_aug[:, :, 3] *= np.random.uniform(0.9, 1.1, Xg_aug[:, :, 3].shape)
            drop_mask = np.random.random((Xg_aug.shape[0], NUM_BEACONS)) > 0.08
            Xg_aug[:, :, 0] *= drop_mask.astype(np.float32)
            Xg_aug += np.random.normal(0, 0.015, Xg_aug.shape)

        out = model.predict([Xs_aug, Xg_aug], batch_size=BATCH_SIZE, verbose=0)
        preds_y.append(out['y_output'])
        preds_fl.append(out['floor_output'])

    y_med = np.median(preds_y, axis=0)
    fl_med = np.median(preds_fl, axis=0)

    y_pred = y_med[:, 0] * y_std + y_mean
    y_pred = np.clip(y_pred, Y_MIN, Y_MAX)

    return y_pred, fl_med


# ==============================================================================
# POST-PROCESSING & EVALUATION
# ==============================================================================
def geometric_postprocess(y_pred, test_seq, train_seq, y_stats):
    train_ys = np.array(sorted(set(s['y'] for s in train_seq)))
    train_min, train_max = train_ys.min(), train_ys.max()

    snapped = y_pred.copy()
    n_clamped = 0
    for i, y in enumerate(y_pred):
        if y < train_min - 3.0 or y > train_max + 3.0:
            snapped[i] = np.clip(y, train_min - 2.0, train_max + 2.0)
            n_clamped += 1

    if n_clamped > 0:
        print(f"    Corridor clamp: fixed {n_clamped} out-of-bounds predictions")
    return snapped

def fingerprint_kalman_smooth(y_pred, test_seq, process_noise=KALMAN_Q, measurement_noise=KALMAN_R):
    fp_data = defaultdict(lambda: {'indices': [], 'y_preds': [], 'timestamp': None, 'y_true': None})
    for i, s in enumerate(test_seq):
        key = (s['session'], s['fingerprintId'])
        fp_data[key]['indices'].append(i)
        fp_data[key]['y_preds'].append(y_pred[i])
        if fp_data[key]['timestamp'] is None:
            fp_data[key]['timestamp'] = s.get('timestamp', 0)
        fp_data[key]['y_true'] = s['y']

    fp_y = {k: np.median(v['y_preds']) for k, v in fp_data.items()}

    sessions = defaultdict(list)
    for key, data in fp_data.items():
        sessions[key[0]].append({
            'key': key,
            'ts': data['timestamp'],
            'indices': data['indices'],
        })

    y_smooth = y_pred.copy()
    smoother = OneDKalmanSmoother(process_noise=process_noise, measurement_noise=measurement_noise)

    for session, items in sessions.items():
        if len(items) < 3:
            continue
        items.sort(key=lambda x: x['ts'])
        y_seq = [fp_y[it['key']] for it in items]
        y_smooth_seq = smoother.smooth(y_seq)
        for it, ys in zip(items, y_smooth_seq):
            for idx in it['indices']:
                y_smooth[idx] = ys

    return y_smooth

def evaluate_model(model_wrapper, test_seq, y_stats, train_seq, val_seq=None):
    y_mean, y_std = y_stats

    X_seq_test   = np.array([s['X'] for s in test_seq], dtype=np.float32)
    X_graph_test = np.array([s['X_graph'] for s in test_seq], dtype=np.float32)
    y_test       = np.array([s['y'] for s in test_seq], dtype=np.float32)
    fl_test      = np.array([s['floor_class'] for s in test_seq], dtype=np.float32)

    # -------------------------------------------------------------------------
    # 1. Build train-only kNN database
    # -------------------------------------------------------------------------
    knn_db = FingerprintKNN(k=KNN_K)
    knn_db.fit(train_seq)

    # -------------------------------------------------------------------------
    # 2. Raw NN prediction
    # -------------------------------------------------------------------------
    out_raw = model_wrapper.model.predict([X_seq_test, X_graph_test], batch_size=BATCH_SIZE, verbose=0)
    y_nn_raw = out_raw['y_output'][:, 0] * y_std + y_mean
    y_nn_raw = np.clip(y_nn_raw, Y_MIN, Y_MAX)

    # -------------------------------------------------------------------------
    # 3. TTA NN prediction
    # -------------------------------------------------------------------------
    y_nn_tta, fl_tta = predict_with_tta(model_wrapper.model, X_seq_test, X_graph_test, y_stats, n_tta=20)

    # -------------------------------------------------------------------------
    # 4. kNN retrieval prediction
    # -------------------------------------------------------------------------
    y_knn = knn_db.predict(test_seq)

    # -------------------------------------------------------------------------
    # 5. Tune blend weight on validation set (leakage-free)
    # -------------------------------------------------------------------------
    if val_seq is not None and len(val_seq) > 0:
        X_seq_val   = np.array([s['X'] for s in val_seq], dtype=np.float32)
        X_graph_val = np.array([s['X_graph'] for s in val_seq], dtype=np.float32)
        y_val       = np.array([s['y'] for s in val_seq], dtype=np.float32)

        out_val = model_wrapper.model.predict([X_seq_val, X_graph_val], batch_size=BATCH_SIZE, verbose=0)
        y_nn_val = out_val['y_output'][:, 0] * y_std + y_mean

        y_knn_val = knn_db.predict(val_seq)

        best_w = 0.5
        best_mae = float('inf')
        for w in np.arange(0.0, 1.01, 0.05):
            mae = np.abs(y_val - (w * y_nn_val + (1 - w) * y_knn_val)).mean()
            if mae < best_mae:
                best_mae = mae
                best_w = w
        print(f"  Tuned NN/kNN blend weight: {best_w:.2f} (val MAE={best_mae:.3f}m)")
    else:
        best_w = 0.70

    # -------------------------------------------------------------------------
    # 6. Hybrid prediction
    # -------------------------------------------------------------------------
    y_hybrid = best_w * y_nn_tta + (1.0 - best_w) * y_knn

    # -------------------------------------------------------------------------
    # 7. Geometric clamp
    # -------------------------------------------------------------------------
    y_geom = geometric_postprocess(y_hybrid, test_seq, train_seq, y_stats)

    # -------------------------------------------------------------------------
    # 8. Kalman trajectory smoothing (tune Q/R on val if available)
    # -------------------------------------------------------------------------
    if val_seq is not None and len(val_seq) > 0:
        y_knn_val = knn_db.predict(val_seq)
        y_hybrid_val = best_w * y_nn_val + (1.0 - best_w) * y_knn_val
        y_geom_val = geometric_postprocess(y_hybrid_val, val_seq, train_seq, y_stats)

        best_q, best_r = KALMAN_Q, KALMAN_R
        best_mae = float('inf')
        for Q in [0.01, 0.05, 0.1, 0.2, 0.5]:
            for R in [0.5, 1.0, 2.0, 4.0]:
                y_sm_val = fingerprint_kalman_smooth(y_geom_val, val_seq, process_noise=Q, measurement_noise=R)
                mae = np.abs(y_val - y_sm_val).mean()
                if mae < best_mae:
                    best_mae = mae
                    best_q, best_r = Q, R
        print(f"  Tuned Kalman Q={best_q}, R={best_r} (val MAE={best_mae:.3f}m)")
    else:
        best_q, best_r = KALMAN_Q, KALMAN_R

    y_final = fingerprint_kalman_smooth(y_geom, test_seq, process_noise=best_q, measurement_noise=best_r)

    # -------------------------------------------------------------------------
    # Metrics
    # -------------------------------------------------------------------------
    def report(y_pred, label):
        err = np.abs(y_test - y_pred)
        mae = err.mean()
        acc_1m = (err <= 1.0).mean() * 100
        acc_2m = (err <= 2.0).mean() * 100
        acc_5m = (err <= 5.0).mean() * 100
        fl_cls = (fl_tta.flatten() >= 0.5).astype(int)
        fl_acc = (fl_cls == fl_test).mean() * 100
        print(f"\n  [{label}]")
        print(f"    Y MAE : {mae:.3f}m")
        print(f"    Acc@1m: {acc_1m:.1f}%  |  Acc@2m: {acc_2m:.1f}%  |  Acc@5m: {acc_5m:.1f}%")
        print(f"    Floor Accuracy : {fl_acc:.1f}%")
        return mae, acc_1m, acc_2m, acc_5m, fl_acc, err

    _, _, _, _, _, _ = report(y_nn_raw,  'NN raw')
    _, _, _, _, _, _ = report(y_nn_tta, 'NN + TTA')
    _, _, _, _, _, _ = report(y_hybrid,  'NN + TTA + kNN')
    mae_final, acc1, acc2, acc5, fl_acc, err_final = report(y_final, 'NN + TTA + kNN + Kalman')

    # Error analysis
    print(f"\n  ── Error Analysis ──")
    print(f"    Median Y Error : {np.median(err_final):.3f}m")
    print(f"    90th percentile: {np.percentile(err_final, 90):.3f}m")
    print(f"    95th percentile: {np.percentile(err_final, 95):.3f}m")
    print(f"    Max Y Error    : {err_final.max():.3f}m")

    for floor in [3, 4]:
        mask = np.array([s['floor_class'] for s in test_seq]) == FLOOR_CLASSES[floor]
        if mask.sum() > 0:
            print(f"    Floor {floor} MAE  : {err_final[mask].mean():.3f}m (n={mask.sum()})")

    # Save CSV
    eval_df = pd.DataFrame([{
        'model': 'hybrid_gat_kalman',
        'mae_final': mae_final,
        'acc_1m': acc1,
        'acc_2m': acc2,
        'acc_5m': acc5,
        'floor_acc': fl_acc,
        'kalman_Q': best_q,
        'kalman_R': best_r,
        'nn_weight': best_w,
    }])
    eval_df.to_csv(EVAL_FILE_PATH, index=False)

    return {
        'model': 'hybrid_gat_kalman',
        'mae_final': mae_final,
        'acc_1m': acc1,
        'acc_2m': acc2,
        'acc_5m': acc5,
        'floor_acc': fl_acc,
        'y_pred': y_final,
        'y_err': err_final,
    }


# ==============================================================================
# REAL-TIME (CAUSAL) EVALUATION — NO FUTURE DATA
# ==============================================================================
def evaluate_realtime(model_wrapper, test_seq, y_stats, train_seq, val_seq=None,
                      kalman_q=KALMAN_Q, kalman_r=KALMAN_R,
                      outlier_z=3.0, outlier_window=10):
    """
    Evaluates the model in a strictly causal, real-time mode:
      • Forward-only Kalman filter (no RTS backward pass)
      • Per-trajectory MAD outlier rejection
      • No TTA, no kNN (fast single-pass)
    """
    y_mean, y_std = y_stats

    X_seq_test   = np.array([s['X'] for s in test_seq], dtype=np.float32)
    X_graph_test = np.array([s['X_graph'] for s in test_seq], dtype=np.float32)
    y_test       = np.array([s['y'] for s in test_seq], dtype=np.float32)
    fl_test      = np.array([s['floor_class'] for s in test_seq], dtype=np.float32)

    # Single-pass NN prediction (fast, no TTA)
    out = model_wrapper.model.predict([X_seq_test, X_graph_test], batch_size=BATCH_SIZE, verbose=0)
    y_nn = out['y_output'][:, 0] * y_std + y_mean
    y_nn = np.clip(y_nn, Y_MIN, Y_MAX)
    fl_prob = out['floor_output'][:, 0]
    fl_cls = (fl_prob >= 0.5).astype(int)

    # --- Step 1: MAD outlier rejection per trajectory ---
    y_mad = apply_mad_per_trajectory(y_nn, test_seq, z_thresh=outlier_z)

    # --- Step 2: Causal forward-only Kalman per session ---
    fp_data = defaultdict(list)
    for i, s in enumerate(test_seq):
        key = (s['session'], s['fingerprintId'])
        fp_data[key].append(i)

    sessions = defaultdict(list)
    for key, indices in fp_data.items():
        sessions[key[0]].append({
            'indices': indices,
            'ts': test_seq[indices[0]]['timestamp'],
        })

    y_causal = y_mad.copy()
    for session, items in sessions.items():
        if len(items) < 2:
            continue
        items.sort(key=lambda x: x['ts'])
        # FIX: Q/R kwargs now accepted by OneDKalmanFilter
        kf = OneDKalmanFilter(Q=kalman_q, R=kalman_r)
        for it in items:
            for idx in it['indices']:
                y_causal[idx] = kf.update(y_mad[idx])

    # --- Step 3: Geometric clamp (train-derived bounds) ---
    y_final = geometric_postprocess(y_causal, test_seq, train_seq, y_stats)

    # --- Metrics ---
    def report(y_pred, label):
        err = np.abs(y_test - y_pred)
        mae = err.mean()
        acc_1m = (err <= 1.0).mean() * 100
        acc_2m = (err <= 2.0).mean() * 100
        acc_5m = (err <= 5.0).mean() * 100
        fl_acc = (fl_cls == fl_test).mean() * 100
        print(f"\n  [{label}]")
        print(f"    Y MAE : {mae:.3f}m")
        print(f"    Acc@1m: {acc_1m:.1f}%  |  Acc@2m: {acc_2m:.1f}%  |  Acc@5m: {acc_5m:.1f}%")
        print(f"    Floor Accuracy : {fl_acc:.1f}%")
        return mae, acc_1m, acc_2m, acc_5m, fl_acc, err

    mae_final, acc1, acc2, acc5, fl_acc, err_final = report(y_final, 'REAL-TIME (Causal + MAD + KF)')

    print(f"\n  ── Real-Time Error Analysis ──")
    print(f"    Median Y Error : {np.median(err_final):.3f}m")
    print(f"    90th percentile: {np.percentile(err_final, 90):.3f}m")
    print(f"    95th percentile: {np.percentile(err_final, 95):.3f}m")
    print(f"    Max Y Error    : {err_final.max():.3f}m")

    for floor in [3, 4]:
        mask = np.array([s['floor_class'] for s in test_seq]) == FLOOR_CLASSES[floor]
        if mask.sum() > 0:
            print(f"    Floor {floor} MAE  : {err_final[mask].mean():.3f}m (n={mask.sum()})")

    return {
        'model': 'realtime_causal',
        'mae_final': mae_final,
        'acc_1m': acc1,
        'acc_2m': acc2,
        'acc_5m': acc5,
        'floor_acc': fl_acc,
        'y_pred': y_final,
        'y_err': err_final,
    }


# ==============================================================================
# MAIN
# ==============================================================================
def main(realtime_mode=False):
    print("\n" + "=" * 70)
    print("BEACON INDOOR LOCALIZATION — GRAPH + kNN + KALMAN PIPELINE")
    print("=" * 70)
    if realtime_mode:
        print("  [REAL-TIME MODE: causal Kalman + MAD outlier rejection]")

    print("\n── STEP 1: LOAD ──────────────────────────────────────────────────")
    df = load_all_datasets(data_dir='/content/drive/MyDrive/beacons_chaos/data')

    print("\n── STEP 2: MINIMAL CLEAN ─────────────────────────────────────────")
    df = minimal_clean(df)
    print(f"  NUM_BEACONS = {NUM_BEACONS} (fixed)")

    print("\n── STEP 3: SPLIT ─────────────────────────────────────────────────")
    unique_sessions = df[['session_name', 'fingerprintId']].drop_duplicates()
    shuffled = unique_sessions.sample(frac=1, random_state=42).reset_index(drop=True)

    n = len(shuffled)
    n_test = int(n * TEST_SPLIT)
    n_val  = int(n * VAL_SPLIT)

    test_keys_df  = shuffled.iloc[:n_test]
    val_keys_df   = shuffled.iloc[n_test:n_test + n_val]
    train_keys_df = shuffled.iloc[n_test + n_val:]

    train_keys_set = set(zip(train_keys_df['session_name'], train_keys_df['fingerprintId']))
    val_keys_set   = set(zip(val_keys_df['session_name'], val_keys_df['fingerprintId']))
    test_keys_set  = set(zip(test_keys_df['session_name'], test_keys_df['fingerprintId']))

    train_df = df[df.apply(lambda row: (row['session_name'], row['fingerprintId']) in train_keys_set, axis=1)].copy()
    val_df   = df[df.apply(lambda row: (row['session_name'], row['fingerprintId']) in val_keys_set, axis=1)].copy()
    test_df  = df[df.apply(lambda row: (row['session_name'], row['fingerprintId']) in test_keys_set, axis=1)].copy()

    print(f"  Rows: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

    print("\n── STEP 4: TRAIN-ONLY STATISTICS ─────────────────────────────────")
    beacon_median_train = train_df.groupby('beaconId')['rssi'].median().to_dict()

    floor_median_train = {}
    for (bid, flvl), grp in train_df.groupby(['beaconId', 'floorLevel'])['rssi']:
        floor_median_train[(bid, flvl)] = grp.median()

    sensor_cols = [
        'gyroX', 'gyroY', 'gyroZ', 'accelX', 'accelY', 'accelZ',
        'userAccelX', 'userAccelY', 'userAccelZ',
        'magX', 'magY', 'magZ', 'pitch', 'roll', 'yaw'
    ]
    sensor_medians = {
        col: float(train_df[col].median())
        for col in sensor_cols
        if col in train_df.columns
    }

    print(f"  Beacon medians: {len(beacon_median_train)}, Floor-beacon medians: {len(floor_median_train)}")

    print("\n── STEP 5: PREPROCESS ────────────────────────────────────────────")
    train_df = preprocess_data(train_df, sensor_medians=sensor_medians)
    val_df   = preprocess_data(val_df, sensor_medians=sensor_medians)
    test_df  = preprocess_data(test_df, sensor_medians=sensor_medians)

    print("\n── STEP 6: ENGINEER FEATURES ─────────────────────────────────────")
    train_df = engineer_features(train_df, beacon_median_map=beacon_median_train, floor_median_map=floor_median_train)
    val_df   = engineer_features(val_df,   beacon_median_map=beacon_median_train, floor_median_map=floor_median_train)
    test_df  = engineer_features(test_df,  beacon_median_map=beacon_median_train, floor_median_map=floor_median_train)

    df_final = pd.concat([train_df, val_df, test_df], ignore_index=True)
    finalise_feature_cols(df_final)

    # FIX: Hard guard against NaN/Inf entering the model (catches engineering bugs early)
    assert not df_final[FEATURE_COLS].isna().any().any(), "NaN detected in engineered features!"
    assert not np.isinf(df_final[FEATURE_COLS].values).any(), "Inf detected in engineered features!"

    all_sequences = create_sequences(df_final, window_size=WINDOW_SIZE, stride=WINDOW_STRIDE)

    train_seq, val_seq, test_seq = [], [], []
    for seq in all_sequences:
        key = (seq['session'], seq['fingerprintId'])
        if key in test_keys_set:
            test_seq.append(seq)
        elif key in val_keys_set:
            val_seq.append(seq)
        elif key in train_keys_set:
            train_seq.append(seq)

    print(f"\n  Sequences: train={len(train_seq)}, val={len(val_seq)}, test={len(test_seq)}")

    print("\n── STEP 7: NORMALIZE ─────────────────────────────────────────────")
    train_seq, val_seq, test_seq, y_stats, scaler, scaler_g = normalize_sequences(train_seq, val_seq, test_seq)

    print("\n── STEP 8: GENERATORS ────────────────────────────────────────────")
    train_gen = MultiTaskGenerator(train_seq, batch_size=BATCH_SIZE, shuffle=True, augment=True)
    val_gen   = MultiTaskGenerator(val_seq,   batch_size=BATCH_SIZE, shuffle=False, augment=False)

    (Bx_seq, Bx_graph), By = train_gen[0]
    print(f"  Batch: seq{Bx_seq.shape} | graph{Bx_graph.shape} | y{By['y_output'].shape} | fl{By['floor_output'].shape}")

    # ── STEP 8.5: SELF-SUPERVISED PRETRAINING (optional) ──────────────────────
    # Trajectory JSON -> shared preprocessing -> masked-reconstruction pretraining of the
    # encoder. Runs ONLY if PRETRAIN=True; the labeled-train scaler/medians are reused so
    # fine-tuning stays leakage-free. Encoder weights are written to ENCODER_WEIGHTS_PATH.
    if PRETRAIN:
        print("\n── STEP 8.5: SSL PRETRAINING ─────────────────────────────────────")
        pretrain_encoder(scaler, scaler_g, y_stats,
                         beacon_median_train, floor_median_train, sensor_medians)

    print("\n── STEP 9: BUILD MODEL ───────────────────────────────────────────")
    clear_stale_cache(N_FEATURES)
    tf.keras.backend.clear_session()

    model = build_hybrid_localizer(name='hybrid_gat')
    model.summary()

    # Transfer pretrained encoder weights BEFORE CustomModel loads any cached fine-tuned
    # checkpoint. Only 'enc_*' layers are copied; y/floor heads keep their fresh init.
    if PRETRAIN and os.path.exists(ENCODER_WEIGHTS_PATH):
        transfer_encoder_weights(model, ENCODER_WEIGHTS_PATH)

    custom_model = CustomModel(name='hybrid_gat', model=model)

    print("\n── STEP 10: TRAIN ────────────────────────────────────────────────")
    custom_model.fit(train_gen, val_gen, epochs=400)

    print("\n── STEP 11: EVALUATE ─────────────────────────────────────────────")
    if realtime_mode:
        result = evaluate_realtime(custom_model, test_seq, y_stats, train_seq, val_seq=val_seq)
    else:
        result = evaluate_model(custom_model, test_seq, y_stats, train_seq, val_seq=val_seq)

    print("\n" + "=" * 70)
    print("DONE")
    print(f"  Final Y MAE : {result['mae_final']:.3f} m")
    print(f"  Acc@1m      : {result['acc_1m']:.1f} %")
    print(f"  Acc@2m      : {result['acc_2m']:.1f} %")
    print(f"  Floor       : {result['floor_acc']:.1f} %")
    print("=" * 70)

    # -------------------------------------------------------------------------
    # Return training artifacts for real-time deployment
    # -------------------------------------------------------------------------
    artifacts = {
        'model': custom_model,
        'scaler': scaler,
        'scaler_g': scaler_g,
        'y_stats': y_stats,
        'beacon_median_map': beacon_median_train,
        'floor_median_map': floor_median_train,
        'sensor_medians': sensor_medians,
        'result': result,
    }
    return artifacts


# ==============================================================================
# DEMO / USAGE EXAMPLE
# ==============================================================================
if __name__ == '__main__':
    # --- Option A: Standard batch evaluation (RTS smoother, TTA, kNN) ---
    # artifacts = main(realtime_mode=False)

    # --- Option B: Real-time causal evaluation (fast, no future data) ---
    artifacts = main(realtime_mode=True)

    # --- Option C: Deploy RealTimeLocalizer for live streaming ---
    #
    # localizer = RealTimeLocalizer(
    #     model=artifacts['model'].model,
    #     scaler=artifacts['scaler'],
    #     scaler_g=artifacts['scaler_g'],
    #     y_stats=artifacts['y_stats'],
    #     beacon_median_map=artifacts['beacon_median_map'],
    #     floor_median_map=artifacts['floor_median_map'],
    #     sensor_medians=artifacts['sensor_medians'],
    #     use_tta=False,          # False for real-time speed
    #     use_knn=False,          # False for real-time speed
    #     kalman_q=0.5,           # tuned on val set
    #     kalman_r=4.0,           # tuned on val set
    #     outlier_z=3.0,
    #     outlier_window=10,
    # )
    #
    # # Simulate live window from test data
    # test_df = ...  # your live DataFrame window
    # y_smooth, floor = localizer.update(test_df)
    # print(f"Live position: y={y_smooth:.2f}m, floor={floor}")
    #
    # # Or use StreamingLocalizer for continuous feed
    # stream = StreamingLocalizer(localizer, window_size=WINDOW_SIZE, min_rows=4)
    # for obs in live_sensor_feed:
    #     stream.add_observation(**obs)
    #     if stream.obs_count % 5 == 0:
    #         y, fl = stream.predict()
    #         if y is not None:
    #             print(f"Stream: y={y:.2f}m, floor={fl}")


BEACON INDOOR LOCALIZATION — GRAPH + kNN + KALMAN PIPELINE
  [REAL-TIME MODE: causal Kalman + MAD outlier rejection]

── STEP 1: LOAD ──────────────────────────────────────────────────

  adhamFloor3.csv
  Loaded 18166 rows × 26 cols

  adhamFloor4.csv
  Loaded 12180 rows × 26 cols

  baselFloor3.csv
  Loaded 18082 rows × 26 cols

  baselFloor4.csv
  Loaded 15730 rows × 26 cols

  remonFloor3.csv
  Loaded 22020 rows × 26 cols

  remonFloor4.csv
  Loaded 21038 rows × 26 cols

COMBINED: 107216 rows

── STEP 2: MINIMAL CLEAN ─────────────────────────────────────────
  NUM_BEACONS = 7 (fixed)

── STEP 3: SPLIT ─────────────────────────────────────────────────
  Rows: train=74862, val=15793, test=16561

── STEP 4: TRAIN-ONLY STATISTICS ─────────────────────────────────
  Beacon medians: 6, Floor-beacon medians: 12

── STEP 5: PREPROCESS ────────────────────────────────────────────

── STEP 6: ENGINEER FEATURES ─────────────────────────────────────

  Final feature set (41): ['beaconId', 'r

Model: "hybrid_gat"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 5, 41)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 5)         │          0 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_t_beacon_emb    │ (None, 5, 16)     │        112 │ lambda[0][0]      │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 5, 40)     │          0 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 5, 56)     │          0 │ enc_t_beacon_emb… │
│ (Concatenate)       │                   │            │ lambda_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_t_proj (Dense)  │ (None, 5, 128)    │      7,296 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d   │ (None, 5, 128)    │          0 │ enc_t_proj[0][0]  │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_t_mha_0         │ (None, 5, 128)    │     66,048 │ spatial_dropout1… │
│ (MultiHeadAttentio… │                   │            │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 5, 128)    │          0 │ spatial_dropout1… │
│                     │                   │            │ enc_t_mha_0[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_t_ln_a_0        │ (None, 5, 128)    │        256 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_t_ffn0_0        │ (None, 5, 256)    │     33,024 │ enc_t_ln_a_0[0][… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 5, 256)    │          0 │ enc_t_ffn0_0[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_t_ffn1_0        │ (None, 5, 128)    │     32,896 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 5, 128)    │          0 │ enc_t_ln_a_0[0][… │
│                     │                   │            │ enc_t_ffn1_0[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_t_ln_b_0        │ (None, 5, 128)    │        256 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_t_mha_1         │ (None, 5, 128)    │     66,048 │ enc_t_ln_b_0[0][… │
│ (MultiHeadAttentio… │                   │            │ enc_t_ln_b_0[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 5, 128)    │          0 │ enc_t_ln_b_0[0][… │
│                     │                   │            │ enc_t_mha_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_t_ln_a_1        │ (None, 5, 128)    │        256 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                 

 Total params: 822,946 (3.14 MB)

 Trainable params: 821,410 (3.13 MB)

 Non-trainable params: 1,536 (6.00 KB)

  Transferred 38 pretrained encoder layers (by name) into the supervised model

── STEP 10: TRAIN ────────────────────────────────────────────────
Epoch 1/400
448/448 ━━━━━━━━━━━━━━━━━━━━ 91s 94ms/step - floor_output_accuracy: 0.5391 - floor_output_loss: 0.7813 - loss: 0.5595 - y_output_loss: 0.5205 - y_output_mae_euclidean: 0.9734 - val_floor_output_accuracy: 0.6590 - val_floor_output_loss: 0.6300 - val_loss: 0.2207 - val_y_output_loss: 0.1894 - val_y_output_mae_euclidean: 0.5123 - learning_rate: 1.5000e-05
Epoch 2/400
448/448 ━━━━━━━━━━━━━━━━━━━━ 9s 21ms/step - floor_output_accuracy: 0.7642 - floor_output_loss: 0.5071 - loss: 0.3406 - y_output_loss: 0.3152 - y_output_mae_euclidean: 0.7043 - val_floor_output_accuracy: 0.9094 - val_floor_output_loss: 0.3331 - val_loss: 0.1419 - val_y_output_loss: 0.1261 - val_y_output_mae_euclidean: 0.4088 - learning_rate: 3.0000e-05
Epoch 3/400
448/448 ━━━━━━━━━━━━━━━━━━━━ 8s 18ms/step - floor_output_accuracy: 0.8969 - floor_output_loss: 0.2998 - loss

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
================================================================================
BEACON INDOOR LOCALIZATION — GRAPH ATTENTION + kNN HYBRID + KALMAN SMOOTHING
Leakage-free, train-only statistics, session/fingerprint split.

REAL-TIME UPDATE (v2):
  • Causal forward-only Kalman filter for streaming inference
  • RealTimeLocalizer + StreamingLocalizer classes for live deployment
  • Per-trajectory MAD outlier rejection (fixes 29 m jumps)
  • Fast prediction path: TTA & kNN optional/disabled by default
  • Batch vs. Real-Time evaluation modes in main()
================================================================================
"""

import os
import re
import sys
import json
import math
import shutil
import pickle
import warnings
from collections import defaultdict

# Windows consoles default to cp1252 and crash on the Unicode box-drawing chars used in the
# step banners. Force UTF-8 on stdout/stderr so the script runs without PYTHONUTF8=1.
for _stream in (sys.stdout, sys.stderr):
    try:
        _stream.reconfigure(encoding='utf-8')
    except Exception:
        pass

import chardet
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model, optimizers, callbacks
from tensorflow.keras.utils import Sequence
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
tf.keras.backend.clear_session()
tf.keras.backend.set_floatx('float32')


# ==============================================================================
# CONFIG
# ==============================================================================
BASE_DIR         = r'/content/drive/MyDrive/beacons_chaos'
CACHE_MODELS_DIR = os.path.join(BASE_DIR, 'train_cache_gat')
EVAL_FILE_PATH   = os.path.join(BASE_DIR, 'eval_df_gat.csv')
NORM_FILE_PATH   = os.path.join(BASE_DIR, 'normalization_gat.json')

os.makedirs(CACHE_MODELS_DIR, exist_ok=True)

BATCH_SIZE    = 128
VAL_SPLIT     = 0.15
TEST_SPLIT    = 0.15
WINDOW_SIZE   = 10
WINDOW_STRIDE = 2

N_FLOORS      = 2
FLOOR_CLASSES = {3: 0, 4: 1}
FLOOR_CLASSES_INV = {0: 3, 1: 4}

W_Y     = 1.0
W_FLOOR = 0.05

Y_MIN, Y_MAX = 0.0, 93.0
MIN_BEACONS_PER_WINDOW = 2   # lowered 4->2: heuristic informativeness filter, not a
                             # trilateration requirement (1-D regressor). >=4 rejects
                             # 5-28% of walking windows; >=2 holds ~99-100% (measured).

# Beacon graph parameters
NUM_BEACONS    = 7      # 0 = padding/unknown, 1..6 = real beacons
EMBED_DIM      = 16
N_BEACON_FEATS = 6      # presence, mean_rssi_norm, std_rssi_norm,
                        # mean_rssi_distance, count_ratio, mean_motion_magnitude

# kNN / Kalman defaults (tuned automatically on val set)
KNN_K       = 5
KALMAN_Q    = 0.1
KALMAN_R    = 2.0

# ------------------------------------------------------------------------------
# SELF-SUPERVISED PRETRAINING (masked reconstruction on trajectory walks)
# ------------------------------------------------------------------------------
# PRETRAIN=False -> supervised-only path (from-scratch baseline / ablation control):
#   no trajectory or SSL code runs, pipeline behaves exactly as the labeled-only version.
# PRETRAIN=True  -> pretrain the shared encoder on unlabeled trajectory JSON via masked
#   signal reconstruction, then fine-tune the full supervised model on the labeled CSVs.
PRETRAIN                = True
TRAJ_DATA_DIR           = r'/content/drive/MyDrive/beacons_chaos/trajectory_data'   # folder with the 15 trajectory JSON files
PRETRAIN_EPOCHS         = 100
PRETRAIN_WINDOW_SECONDS = 1.5     # time-bucket size for synthetic walk windows (~WINDOW_SIZE readings)
SSL_MASK_RATE           = 0.5     # fraction of (timestep x channel) entries masked for reconstruction
IMU_JOIN_TOLERANCE_MS   = 40      # as-of merge tolerance BLE<-nearest IMU sample
ENCODER_WEIGHTS_PATH    = os.path.join(CACHE_MODELS_DIR, 'ssl_encoder.weights.h5')

# ------------------------------------------------------------------------------
# RSSI-ONLY FINE-TUNING  (SSL pretrains on ALL features; labeled model uses RSSI only)
# ------------------------------------------------------------------------------
# When True, the non-RSSI channels (IMU + motion) are ZEROED in the LABELED sequences
# (train/val/test + live inference) AFTER normalization. The SSL/trajectory path is left
# untouched, so pretraining still sees real IMU. Because the input tensor SHAPE is
# unchanged, the pretrained encoder still transfers 100%. Net effect: the fine-tuned /
# deployed model depends only on RSSI -> the mobile app needs NO IMU (pass zeros).
RSSI_ONLY_FINETUNE = True
# Channels neutralised when RSSI_ONLY_FINETUNE=True (raw IMU + motion-derived features).
# rel_time / window_progress / all RSSI & beacon-geometry features are KEPT.
NON_RSSI_FEATURE_COLS = [
    'gyroX', 'gyroY', 'gyroZ', 'accelX', 'accelY', 'accelZ',
    'userAccelX', 'userAccelY', 'userAccelZ',
    'magX', 'magY', 'magZ', 'pitch', 'roll', 'yaw',
    'motion_magnitude', 'motion_yaw_rate',
]

# Optional: only if known from architectural drawings, NOT estimated from data
BEACON_Y_POS = {
    # 1: 0.0,
    # 2: 18.0,
    # 3: 37.0,
    # 4: 56.0,
    # 5: 74.0,
    # 6: 93.0,
}

# ==============================================================================
# BEACON MAPPING
# ==============================================================================
BEACON_MAP = {
    'fda50693-a4e2-4fb1-afcf-c6eb07647825:1:1':         1,
    'fda50693-a4e2-4fb1-afcf-c6eb07647825:0:2':         2,
    'aaaaaaaa-aaaa-aaaa-aaaa-aaaaaaaaaaaa:0:3':          3,
    'fda50693-a4e2-4fb1-afcf-c6eb07647825:10065:26049': 4,
    'fda50693-a4e2-4fb1-afcf-c6eb07647825:1:5':         5,
    'fda50693-a4e2-4fb1-afcf-c6eb07647825:1:6':         6,
}
BEACON_FLOOR = {1: 3, 2: 3, 3: 3, 4: 4, 5: 4, 6: 4}


# ==============================================================================
# HELPERS
# ==============================================================================
def pkl_save(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)

def pkl_load(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

def json_save(obj, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=4)

def json_load(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def clear_stale_cache(expected_features):
    marker = os.path.join(CACHE_MODELS_DIR, '.feature_count')
    if os.path.exists(marker):
        with open(marker) as f:
            cached = int(f.read().strip())
        if cached != expected_features:
            print(f"  Cache stale ({cached} → {expected_features}). Clearing...")
            shutil.rmtree(CACHE_MODELS_DIR)
            os.makedirs(CACHE_MODELS_DIR, exist_ok=True)
    with open(marker, 'w') as f:
        f.write(str(expected_features))


# ==============================================================================
# DATA LOADING
# ==============================================================================
def inspect_raw_file(filepath, n_bytes=500):
    if not os.path.exists(filepath):
        return None, None
    with open(filepath, 'rb') as f:
        raw = f.read(n_bytes)
    enc = chardet.detect(raw)['encoding'] or 'utf-8'
    with open(filepath, 'r', encoding=enc, errors='replace') as f:
        first = f.readline()
    seps = {s: first.count(s) for s in ['\t', ',', ';', '|']}
    sep = max(seps, key=seps.get) if max(seps.values()) > 0 else '\t'
    return enc, sep

def robust_load_csv(filepath):
    encoding, sep = inspect_raw_file(filepath)
    df = None
    for kw in [
        {'encoding': encoding, 'sep': sep},
        {'encoding': 'utf-8-sig', 'sep': sep},
        {'encoding': 'utf-8-sig', 'sep': sep, 'engine': 'python'},
        {'encoding': 'latin-1', 'sep': sep},
    ]:
        try:
            df = pd.read_csv(filepath, **kw)
            print(f"  Loaded {len(df)} rows × {len(df.columns)} cols")
            break
        except Exception:
            continue
    if df is None:
        raise RuntimeError(f"Cannot load {filepath}")

    def clean(c):
        c = str(c).replace('\ufeff', '').strip()
        return ''.join(ch for ch in c if ch.isprintable() or ch.isspace()).strip()

    df.columns = [clean(c) for c in df.columns]

    seen, new_cols = {}, []
    for c in df.columns:
        seen[c] = seen.get(c, 0) + 1
        new_cols.append(c if seen[c] == 1 else f"{c}_dup{seen[c]}")
    df.columns = new_cols

    remap = {}
    for c in df.columns:
        cl = c.lower().replace(' ', '').replace('_', '')
        if cl in ('fingerprintid', 'fingerprint', 'fprintid'):
            remap[c] = 'fingerprintId'
        elif cl in ('sessionid', 'session'):
            remap[c] = 'sessionId'
        elif cl in ('floorlevel', 'floor', 'floorlvl'):
            remap[c] = 'floorLevel'
        elif cl in ('windowindex', 'windowidx', 'winindex'):
            remap[c] = 'windowIndex'
        elif cl in ('beaconuid', 'beaconid', 'beaconuuid'):
            remap[c] = 'beaconUid'
        elif cl in ('capturedat', 'timestamp', 'time', 'datetime'):
            remap[c] = 'capturedAt'
    if remap:
        df.rename(columns=remap, inplace=True)
    return df

def load_all_datasets(data_dir='labelled_data'):
    # Filenames match the actual files in labelled_data/ (note the source typos
    # "remonFloro4" and the " (2)" suffix — kept as-is to avoid renaming raw data).
    file_mapping = {
        'adhamFloor3.csv':     {'person': 'adham', 'floor': 3},
        'adhamFloor4.csv': {'person': 'adham', 'floor': 4},
        'baselFloor3.csv':     {'person': 'basel', 'floor': 3},
        'baselFloor4.csv':     {'person': 'basel', 'floor': 4},
        'remonFloor3.csv':     {'person': 'remon', 'floor': 3},
        'remonFloor4.csv':     {'person': 'remon', 'floor': 4},
    }
    all_dfs = []
    for fname, info in file_mapping.items():
        fpath = os.path.join(data_dir, fname)
        if not os.path.exists(fpath):
            print(f"  WARNING: {fname} not found")
            continue
        print(f"\n  {fname}")
        df = robust_load_csv(fpath)
        df['person'] = info['person']
        df['floor'] = info['floor']
        df['session_name'] = f"{info['person']}_floor{info['floor']}"
        all_dfs.append(df)

    if not all_dfs:
        raise RuntimeError("No input files found.")

    combined = pd.concat(all_dfs, ignore_index=True)
    print(f"\nCOMBINED: {len(combined)} rows")
    return combined


# ==============================================================================
# TRAJECTORY LOADING  (unlabeled walks -> CSV-schema rows for SSL pretraining)
# ==============================================================================
# The trajectory JSON is structured completely differently from the labeled CSV:
#   session -> walks[] -> {steps[], imuSamples[], bleReadings[], checkpoints[]}
# bleReadings and imuSamples are SEPARATE asynchronous streams and there is no
# fingerprintId / windowIndex / sessionId on the readings. This loader flattens each
# walk into the SAME flat schema the labeled CSV uses, so every downstream function
# (minimal_clean / preprocess_data / engineer_features / create_sequences) is reused
# verbatim. Position labels are NOT used (SSL is label-free); y is a dummy.
IMU_COLS_JSON = [
    'gyroX', 'gyroY', 'gyroZ', 'accelX', 'accelY', 'accelZ',
    'userAccelX', 'userAccelY', 'userAccelZ',
    'magX', 'magY', 'magZ', 'pitch', 'roll', 'yaw',
]

def load_trajectory_json(filepath):
    """Flatten one trajectory JSON file into a CSV-schema DataFrame (one row per BLE reading)."""
    data = json_load(filepath)
    stem = os.path.splitext(os.path.basename(filepath))[0]
    session = data.get('session', {}) or {}
    floor_level = session.get('floorLevel')

    walk_frames = []
    for w_ord, walk in enumerate(data.get('walks', [])):
        ble = walk.get('bleReadings', []) or []
        imu = walk.get('imuSamples', []) or []
        if not ble:
            continue

        ble_df = pd.DataFrame(ble)[['capturedAt', 'beaconUid', 'rssi']].copy()
        ble_df['capturedAt'] = pd.to_datetime(ble_df['capturedAt'], errors='coerce', utc=True)
        ble_df = ble_df.dropna(subset=['capturedAt']).sort_values('capturedAt').reset_index(drop=True)
        if ble_df.empty:
            continue

        # As-of merge: attach the nearest-in-time IMU sample to each BLE reading.
        if imu:
            imu_df = pd.DataFrame(imu)
            keep = ['capturedAt'] + [c for c in IMU_COLS_JSON if c in imu_df.columns]
            imu_df = imu_df[keep].copy()
            imu_df['capturedAt'] = pd.to_datetime(imu_df['capturedAt'], errors='coerce', utc=True)
            imu_df = imu_df.dropna(subset=['capturedAt']).sort_values('capturedAt').reset_index(drop=True)
            merged = pd.merge_asof(
                ble_df, imu_df, on='capturedAt', direction='nearest',
                tolerance=pd.Timedelta(milliseconds=IMU_JOIN_TOLERANCE_MS),
            )
        else:
            merged = ble_df.copy()

        # Ensure all IMU columns exist (missing -> NaN; existing imputation handles it).
        for col in IMU_COLS_JSON:
            if col not in merged.columns:
                merged[col] = np.nan

        # Reconstruct window boundaries from timestamps (no windowIndex exists in the JSON).
        t0 = merged['capturedAt'].min()
        rel_s = (merged['capturedAt'] - t0).dt.total_seconds()
        merged['windowIndex'] = np.floor(rel_s / PRETRAIN_WINDOW_SECONDS).astype(int)

        # Synthetic metadata mirroring the labeled CSV schema.
        merged['fingerprintId'] = f"{stem}_w{w_ord}"        # walk ordinal within the file
        merged['sessionId'] = stem
        merged['session_name'] = f"traj_{stem}"             # never collides with labeled sessions
        merged['floorLevel'] = floor_level
        merged['x'] = 1.0                                   # constant corridor coordinate
        merged['y'] = 0.0                                   # dummy label (unused by SSL)
        walk_frames.append(merged)

    if not walk_frames:
        return pd.DataFrame()
    return pd.concat(walk_frames, ignore_index=True)

def load_all_trajectories(data_dir=TRAJ_DATA_DIR):
    if not os.path.isdir(data_dir):
        raise RuntimeError(f"Trajectory dir not found: {data_dir}")
    files = sorted(f for f in os.listdir(data_dir) if f.lower().endswith('.json'))
    if not files:
        raise RuntimeError(f"No trajectory JSON files in {data_dir}")
    all_dfs = []
    for fname in files:
        df = load_trajectory_json(os.path.join(data_dir, fname))
        if not df.empty:
            print(f"  {fname}: {len(df)} BLE rows, "
                  f"{df['windowIndex'].nunique()} windows/walk-avg")
            all_dfs.append(df)
    combined = pd.concat(all_dfs, ignore_index=True)
    print(f"\nTRAJECTORY COMBINED: {len(combined)} rows from {len(all_dfs)} files")
    return combined


# ==============================================================================
# CLEANING  —  drop unknown beacons so NUM_BEACONS stays fixed
# ==============================================================================
def minimal_clean(df):
    df = df.copy()

    for col in ['pressure', 'relativeAltitude']:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)

    df['beaconId'] = df['beaconUid'].map(BEACON_MAP)
    unknown = df['beaconId'].isna()
    n_unknown = unknown.sum()
    if n_unknown:
        print(f"  WARNING: dropping {n_unknown} rows with unknown beacons")
        df = df[~unknown].copy()

    df['beaconId'] = df['beaconId'].astype(int)

    df['capturedAt'] = pd.to_datetime(df['capturedAt'], errors='coerce')
    df = df.dropna(subset=['capturedAt'])

    df.sort_values(['session_name', 'fingerprintId', 'windowIndex', 'capturedAt'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df


# ==============================================================================
# PREPROCESSING — TRAIN-ONLY IMPUTATION
# ==============================================================================
def preprocess_data(df, sensor_medians=None):
    df = df.copy()

    df['rel_time'] = df.groupby(['session_name', 'fingerprintId', 'windowIndex'])['capturedAt'].transform(
        lambda x: (x - x.min()).dt.total_seconds()
    )
    df['window_row_idx'] = df.groupby(['session_name', 'fingerprintId', 'windowIndex']).cumcount()

    df.sort_values(['session_name', 'fingerprintId', 'windowIndex', 'capturedAt'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    impute_cols = [
        'gyroX', 'gyroY', 'gyroZ', 'accelX', 'accelY', 'accelZ',
        'userAccelX', 'userAccelY', 'userAccelZ',
        'magX', 'magY', 'magZ', 'pitch', 'roll', 'yaw'
    ]

    for col in impute_cols:
        if col not in df.columns:
            continue
        if df[col].isna().any():
            df[col] = df.groupby(['session_name', 'fingerprintId', 'windowIndex'])[col].transform(
                lambda x: x.ffill().bfill()
            )
            if sensor_medians is not None and col in sensor_medians:
                df[col] = df[col].fillna(sensor_medians[col])
            else:
                df[col] = df[col].fillna(0)

    df['floor_class'] = df['floorLevel'].map(FLOOR_CLASSES).fillna(0).astype(int)
    return df


# ==============================================================================
# FEATURE ENGINEERING — NO FINGERPRINT MEMORIZATION
# ==============================================================================
BASE_SENSOR_COLS = [
    'beaconId', 'rssi', 'rel_time',
    'gyroX', 'gyroY', 'gyroZ', 'accelX', 'accelY', 'accelZ',
    'userAccelX', 'userAccelY', 'userAccelZ',
    'magX', 'magY', 'magZ', 'pitch', 'roll', 'yaw',
    'rssi_norm', 'rssi_inv', 'rssi_distance',
    'beacon_floor_match', 'window_progress',
    'rssi_diff_from_beacon_median',
    'beacon_signal_rank',
    'window_rssi_mean', 'window_rssi_std',
    'min_beacon_distance', 'beacon_weight',
    'rssi_vs_floor_median',
    'beacon_on_this_floor',
    'dominant_beacon_id',
    'rssi_diff_to_max', 'rssi_diff_to_2nd_max',
    'beacon_count_in_window',
    'motion_magnitude', 'motion_yaw_rate',
    'beacon_y_pos',
    'unique_beacon_count',
    'weighted_beacon_y',
    'strongest_beacon_y',
]

FEATURE_COLS = None
N_FEATURES = None

def engineer_features(df, beacon_median_map=None, floor_median_map=None):
    df = df.copy()

    df['rssi_norm'] = (df['rssi'] + 100) / 60
    df['rssi_inv'] = -df['rssi']

    TX_POWER, PATH_LOSS = -59, 2.0
    df['rssi_distance'] = 10 ** ((TX_POWER - df['rssi']) / (10 * PATH_LOSS))
    # FIX: Cap rssi_distance to prevent gradient explosions from anomalous RSSI
    df['rssi_distance'] = df['rssi_distance'].clip(upper=500)

    df['beacon_floor'] = df['beaconId'].map(BEACON_FLOOR).fillna(0).astype(int)
    df['beacon_on_this_floor'] = (df['beacon_floor'] == df['floorLevel']).astype(float)
    df['beacon_floor_match'] = df['beacon_on_this_floor']

    df['window_progress'] = df.groupby(['session_name', 'fingerprintId', 'windowIndex'])['window_row_idx'].transform(
        lambda x: x / max(int(x.max()), 1)
    )

    group_cols = ['session_name', 'fingerprintId', 'windowIndex']

    # Train-only beacon median reference
    if beacon_median_map is not None:
        df['beacon_median_rssi'] = df['beaconId'].map(beacon_median_map).fillna(df['rssi'])
        df['rssi_diff_from_beacon_median'] = df['rssi'] - df['beacon_median_rssi']
        df.drop(columns=['beacon_median_rssi'], inplace=True)
    else:
        df['rssi_diff_from_beacon_median'] = 0.0

    # Per-observation ranks within a timestamp group
    df['beacon_signal_rank'] = df.groupby(
        ['session_name', 'fingerprintId', 'windowIndex', 'capturedAt']
    )['rssi'].rank(ascending=False, method='dense')

    df['window_rssi_mean'] = df.groupby(group_cols)['rssi'].transform('mean')
    df['window_rssi_std'] = df.groupby(group_cols)['rssi'].transform('std').fillna(0)

    df['min_beacon_distance'] = df.groupby(
        ['session_name', 'fingerprintId', 'windowIndex', 'capturedAt']
    )['rssi_distance'].transform('min')

    df['beacon_weight'] = 1.0 / (df['rssi_distance'] + 0.1)

    # Train-only floor-beacon median reference
    if floor_median_map is not None:
        df['floor_beacon_median_rssi'] = df.apply(
            lambda row: floor_median_map.get((row['beaconId'], row['floorLevel']), row['rssi']),
            axis=1
        )
        df['rssi_vs_floor_median'] = df['rssi'] - df['floor_beacon_median_rssi']
        df.drop(columns=['floor_beacon_median_rssi'], inplace=True)
    else:
        df['rssi_vs_floor_median'] = 0.0

    # Dominant / second strongest beacons inside each timestamp group
    group_time = ['session_name', 'fingerprintId', 'windowIndex', 'capturedAt']
    idxmax = df.groupby(group_time)['rssi'].idxmax()
    dominant_map = df.loc[idxmax].set_index(group_time)['beaconId']
    df['dominant_beacon_id'] = df.set_index(group_time).index.map(dominant_map).fillna(0).astype(int)

    max_rssi_per_group = df.groupby(group_time)['rssi'].transform('max')
    df['rssi_diff_to_max'] = df['rssi'] - max_rssi_per_group

    df_sorted = df.sort_values(group_time + ['rssi'], ascending=[True, True, True, True, False])
    df_sorted['rssi_rank'] = df_sorted.groupby(group_time).cumcount()
    second_max_map = df_sorted[df_sorted['rssi_rank'] == 1].set_index(group_time)['rssi']
    df['second_max_rssi'] = df.set_index(group_time).index.map(second_max_map)
    df['second_max_rssi'] = df['second_max_rssi'].fillna(max_rssi_per_group)
    df['rssi_diff_to_2nd_max'] = df['rssi'] - df['second_max_rssi']
    df.drop(columns=['second_max_rssi'], inplace=True)

    df['beacon_count_in_window'] = df.groupby(group_cols)['beaconId'].transform('count')
    df['motion_magnitude'] = np.sqrt(df['accelX']**2 + df['accelY']**2 + df['accelZ']**2)
    df['motion_yaw_rate'] = df['gyroZ'].abs()

    if BEACON_Y_POS:
        df['beacon_y_pos'] = df['beaconId'].map(BEACON_Y_POS).fillna(-1)
    else:
        df['beacon_y_pos'] = -1

    df['unique_beacon_count'] = df.groupby(group_cols)['beaconId'].transform('nunique')

    # Optional geometric features from external beacon map only
    if BEACON_Y_POS:
        df['rssi_lin'] = 10 ** (df['rssi'] / 10.0)
        df['weighted_y_num'] = df['beacon_y_pos'] * df['rssi_lin']

        wy_num = df.groupby(group_cols)['weighted_y_num'].transform('sum')
        wy_den = df.groupby(group_cols)['rssi_lin'].transform('sum')
        df['weighted_beacon_y'] = wy_num / (wy_den + 1e-6)

        idxmax_rssi = df.groupby(group_time)['rssi'].idxmax()
        strongest_y_map = df.loc[idxmax_rssi].set_index(group_time)['beacon_y_pos']
        df['strongest_beacon_y'] = df.set_index(group_time).index.map(strongest_y_map).fillna(-1)

        df.drop(columns=['rssi_lin', 'weighted_y_num'], inplace=True)
    else:
        df['weighted_beacon_y'] = -1
        df['strongest_beacon_y'] = -1

    return df

def finalise_feature_cols(df):
    global FEATURE_COLS, N_FEATURES
    FEATURE_COLS = [c for c in BASE_SENSOR_COLS if c in df.columns]
    N_FEATURES = len(FEATURE_COLS)
    print(f"\n  Final feature set ({N_FEATURES}): {FEATURE_COLS[:6]} ... {FEATURE_COLS[-4:]}")
    return FEATURE_COLS


# ==============================================================================
# BEACON GRAPH FEATURE EXTRACTOR  (per window → fixed-size beacon nodes)
# ==============================================================================
def extract_beacon_graph_features(X, beacon_ids, num_beacons=NUM_BEACONS):
    """
    X          : (window_size, n_features)  — pre-normalisation sequence matrix
    beacon_ids : (window_size,)              — raw int beacon IDs (1..6)
    Returns    : (num_beacons, N_BEACON_FEATS)
    """
    feats = np.zeros((num_beacons, N_BEACON_FEATS), dtype=np.float32)

    rssi_norm_idx = FEATURE_COLS.index('rssi_norm')
    rssi_dist_idx = FEATURE_COLS.index('rssi_distance')
    motion_idx    = FEATURE_COLS.index('motion_magnitude') if 'motion_magnitude' in FEATURE_COLS else None

    b_ids = np.clip(beacon_ids.astype(int), 0, num_beacons - 1)

    for b in range(num_beacons):
        mask = b_ids == b
        if not mask.any():
            continue
        sub = X[mask]
        feats[b, 0] = 1.0                                           # presence
        feats[b, 1] = sub[:, rssi_norm_idx].mean()                  # mean rssi_norm
        feats[b, 2] = sub[:, rssi_norm_idx].std() if len(sub) > 1 else 0.0  # std
        feats[b, 3] = sub[:, rssi_dist_idx].mean()                  # mean distance
        feats[b, 4] = len(sub) / len(X)                           # count ratio
        if motion_idx is not None:
            feats[b, 5] = sub[:, motion_idx].mean()                 # mean motion

    return feats

def extract_rssi_signature(X, beacon_ids, num_beacons=NUM_BEACONS, fill=-2.0):
    """Single-vector RSSI signature for kNN lookup."""
    sig = np.full(num_beacons, fill, dtype=np.float32)
    rssi_norm_idx = FEATURE_COLS.index('rssi_norm')
    b_ids = np.clip(beacon_ids.astype(int), 0, num_beacons - 1)
    for b in range(num_beacons):
        mask = b_ids == b
        if mask.any():
            sig[b] = X[mask, rssi_norm_idx].mean()
    return sig

# ==============================================================================
# SEQUENCE CREATION  (batch + single-window for real-time)
# ==============================================================================
def create_sequences(df, window_size=WINDOW_SIZE, stride=WINDOW_STRIDE):
    feature_cols = FEATURE_COLS
    sequences = []
    grouped = df.groupby(['session_name', 'fingerprintId', 'windowIndex'])
    print(f"\n  Creating sequences from {len(grouped)} windows...")

    rejected = 0
    for (session, fp_id, w_idx), wdf in grouped:
        wdf = wdf.sort_values('capturedAt').reset_index(drop=True)
        n = len(wdf)

        if wdf['beaconId'].nunique() < MIN_BEACONS_PER_WINDOW:
            rejected += 1
            continue

        base = {
            'y': float(wdf['y'].iloc[0]),
            'x': float(wdf['x'].iloc[0]) if 'x' in wdf.columns else 0.0,
            'floor_class': int(wdf['floor_class'].iloc[0]),
            'session': session,
            'fingerprintId': fp_id,
            'windowIndex': w_idx,
            'timestamp': wdf['capturedAt'].min().timestamp() if 'capturedAt' in wdf.columns else 0.0,
        }

        feat = wdf[feature_cols].values.astype(np.float32)
        b_ids = wdf['beaconId'].values.astype(np.int32)

        if n < window_size:
            feat_pad = np.pad(feat, ((0, window_size - n), (0, 0)), mode='edge')
            b_ids_pad = np.pad(b_ids, (0, window_size - n), mode='edge')
            X_graph = extract_beacon_graph_features(feat_pad, b_ids_pad)
            sequences.append({**base, 'X': feat_pad, 'X_graph': X_graph, 'beacon_ids': b_ids_pad})
        else:
            for start in range(0, n - window_size + 1, stride):
                sub_feat = feat[start:start + window_size]
                sub_bids = b_ids[start:start + window_size]
                if len(np.unique(sub_bids)) < MIN_BEACONS_PER_WINDOW:
                    rejected += 1
                    continue
                X_graph = extract_beacon_graph_features(sub_feat, sub_bids)
                sequences.append({
                    **base,
                    'X': sub_feat,
                    'X_graph': X_graph,
                    'beacon_ids': sub_bids,
                })

    print(f"  Total sequences: {len(sequences)} (rejected {rejected} under-determined)")
    return sequences

def create_single_sequence(window_df):
    """Lightweight sequence extraction for a single live window (real-time)."""
    global FEATURE_COLS, N_FEATURES
    if FEATURE_COLS is None:
        finalise_feature_cols(window_df)

    feature_cols = FEATURE_COLS
    wdf = window_df.sort_values('capturedAt').reset_index(drop=True)
    n = len(wdf)

    if wdf['beaconId'].nunique() < MIN_BEACONS_PER_WINDOW:
        return None

    base = {
        'y': float(wdf['y'].iloc[0]) if 'y' in wdf.columns else 0.0,
        'x': float(wdf['x'].iloc[0]) if 'x' in wdf.columns else 0.0,
        'floor_class': int(wdf['floor_class'].iloc[0]) if 'floor_class' in wdf.columns else 0,
        'session': str(wdf['session_name'].iloc[0]) if 'session_name' in wdf.columns else 'live',
        'fingerprintId': int(wdf['fingerprintId'].iloc[0]) if 'fingerprintId' in wdf.columns else 0,
        'windowIndex': int(wdf['windowIndex'].iloc[0]) if 'windowIndex' in wdf.columns else 0,
        'timestamp': wdf['capturedAt'].min().timestamp() if 'capturedAt' in wdf.columns else 0.0,
    }

    feat = wdf[feature_cols].values.astype(np.float32)
    b_ids = wdf['beaconId'].values.astype(np.int32)

    if n < WINDOW_SIZE:
        feat = np.pad(feat, ((0, WINDOW_SIZE - n), (0, 0)), mode='edge')
        b_ids = np.pad(b_ids, (0, WINDOW_SIZE - n), mode='edge')

    X_graph = extract_beacon_graph_features(feat, b_ids)
    return {**base, 'X': feat, 'X_graph': X_graph, 'beacon_ids': b_ids}


# ==============================================================================
# SPLIT — BY SESSION + FINGERPRINT ONLY
# ==============================================================================
def split_by_session(sequences, val_ratio=0.15, test_ratio=0.15, seed=42):
    np.random.seed(seed)
    groups = defaultdict(list)

    for s in sequences:
        groups[(s['session'], s['fingerprintId'])].append(s)

    keys = list(groups.keys())
    np.random.shuffle(keys)

    n = len(keys)
    n_test = int(n * test_ratio)
    n_val = int(n * val_ratio)

    test_keys = keys[:n_test]
    val_keys = keys[n_test:n_test + n_val]
    train_keys = keys[n_test + n_val:]

    train_seq = [s for k in train_keys for s in groups[k]]
    val_seq   = [s for k in val_keys   for s in groups[k]]
    test_seq  = [s for k in test_keys  for s in groups[k]]

    print(f"\n  Split: train={len(train_seq)}, val={len(val_seq)}, test={len(test_seq)}")
    return train_seq, val_seq, test_seq


# ==============================================================================
# NORMALIZATION — TRAIN ONLY  (protect beaconId from scaling!)
# ==============================================================================
def normalize_sequences(train_seq, val_seq, test_seq):
    n_feat = N_FEATURES

    # --- temporal branch ---
    train_X = np.array([s['X'] for s in train_seq], dtype=np.float32)
    flat = train_X.reshape(-1, n_feat)
    scaler = StandardScaler()
    scaler.fit(flat)

    # CRITICAL: keep beaconId as integer index for Embedding layer
    bidx = FEATURE_COLS.index('beaconId')
    scaler.mean_[bidx] = 0.0
    scaler.scale_[bidx] = 1.0

    # --- graph branch ---
    train_Xg = np.array([s['X_graph'] for s in train_seq], dtype=np.float32)
    flat_g = train_Xg.reshape(-1, N_BEACON_FEATS)
    scaler_g = StandardScaler()
    scaler_g.fit(flat_g)

    # --- y ---
    train_y = np.array([s['y'] for s in train_seq], dtype=np.float32)
    y_mean = float(train_y.mean())
    y_std  = float(train_y.std())
    y_std  = max(y_std, 1e-6)

    for seq_list in [train_seq, val_seq, test_seq]:
        for s in seq_list:
            s['X'] = scaler.transform(s['X'].reshape(-1, n_feat)).reshape(WINDOW_SIZE, n_feat).astype(np.float32)
            s['X_graph'] = scaler_g.transform(s['X_graph'].reshape(-1, N_BEACON_FEATS)).reshape(NUM_BEACONS, N_BEACON_FEATS).astype(np.float32)
            s['y_norm'] = (s['y'] - y_mean) / y_std

    json_save({
        'feature_mean': scaler.mean_.tolist(),
        'feature_scale': scaler.scale_.tolist(),
        'feature_cols': FEATURE_COLS,
        'graph_mean': scaler_g.mean_.tolist(),
        'graph_scale': scaler_g.scale_.tolist(),
        'y_mean': y_mean,
        'y_std': y_std,
        'y_min': Y_MIN,
        'y_max': Y_MAX,
    }, NORM_FILE_PATH)

    print(f"  y∈[{train_y.min():.1f},{train_y.max():.1f}] μ={y_mean:.2f} σ={y_std:.2f}")
    return train_seq, val_seq, test_seq, (y_mean, y_std), scaler, scaler_g

def normalize_single_sequence(seq, scaler, scaler_g, y_mean, y_std):
    """Normalize a single live-window sequence in-place."""
    n_feat = N_FEATURES
    seq['X'] = scaler.transform(seq['X'].reshape(-1, n_feat)).reshape(WINDOW_SIZE, n_feat).astype(np.float32)
    seq['X_graph'] = scaler_g.transform(seq['X_graph'].reshape(-1, N_BEACON_FEATS)).reshape(NUM_BEACONS, N_BEACON_FEATS).astype(np.float32)
    seq['y_norm'] = (seq['y'] - y_mean) / max(y_std, 1e-6)
    return seq


# ------------------------------------------------------------------------------
# RSSI-ONLY MASKING (labeled / deployed path only — never applied to SSL trajectory data)
# ------------------------------------------------------------------------------
_NON_RSSI_IDX_CACHE = None

def _non_rssi_indices():
    """Column indices in FEATURE_COLS that must be zeroed under RSSI_ONLY_FINETUNE (cached)."""
    global _NON_RSSI_IDX_CACHE
    if _NON_RSSI_IDX_CACHE is None:
        _NON_RSSI_IDX_CACHE = [FEATURE_COLS.index(c) for c in NON_RSSI_FEATURE_COLS if c in FEATURE_COLS]
    return _NON_RSSI_IDX_CACHE

def mask_non_rssi(seq):
    """
    Zero the IMU/motion channels of a NORMALISED sequence in place (RSSI_ONLY_FINETUNE).
    Applied to LABELED / live sequences only; the SSL trajectory path is never masked so
    pretraining keeps real IMU. Input SHAPE is unchanged -> pretrained encoder still transfers.
    """
    idx = _non_rssi_indices()
    if idx:
        seq['X'][:, idx] = 0.0
    # beacon-graph motion channel (mean_motion_magnitude) is the last N_BEACON_FEATS column
    if N_BEACON_FEATS >= 6:
        seq['X_graph'][:, 5] = 0.0
    return seq

def mask_non_rssi_batch(X_seq, X_graph):
    """Vectorised zero-mask for a batch inside the generator (post-augmentation)."""
    idx = _non_rssi_indices()
    if idx:
        X_seq[:, :, idx] = 0.0
    if N_BEACON_FEATS >= 6:
        X_graph[:, :, 5] = 0.0
    return X_seq, X_graph


# ==============================================================================
# GENERATOR  (dual input — tuple return required by TF)
# ==============================================================================
class MultiTaskGenerator(Sequence):
    def __init__(self, sequences, batch_size=BATCH_SIZE, shuffle=True, augment=False):
        self.sequences = sequences
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment
        self.indices = np.arange(len(sequences))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.sequences) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        X_seq   = np.array([self.sequences[i]['X'] for i in batch_idx], dtype=np.float32)
        X_graph = np.array([self.sequences[i]['X_graph'] for i in batch_idx], dtype=np.float32)
        y  = np.array([[self.sequences[i]['y_norm']] for i in batch_idx], dtype=np.float32)
        fl = np.array([self.sequences[i]['floor_class'] for i in batch_idx], dtype=np.float32)

        if self.augment:
            rssi_idx = FEATURE_COLS.index('rssi') if 'rssi' in FEATURE_COLS else 1
            # FIX: Protect beaconId (column 0) from corruption during augmentation
            X_seq[:, :, rssi_idx] += np.random.normal(0, 0.10, X_seq[:, :, rssi_idx].shape)

            # Dropout mask: shape (B,T,1) so it broadcasts to all feature channels
            mask = np.random.random((X_seq.shape[0], X_seq.shape[1])) > 0.12
            mask = mask[:, :, np.newaxis]          # (B, T, 1)

            # Apply mask only to non-beaconId columns (broadcasts correctly)
            X_seq[:, :, 1:] = X_seq[:, :, 1:] * mask

            # Gaussian noise: zero out channel 0 (beaconId)
            noise = np.random.normal(0, 0.015, X_seq.shape)
            noise[:, :, 0] = 0.0
            X_seq += noise
            # Ensure beaconId stays valid integer indices after any float drift
            X_seq[:, :, 0] = np.round(np.clip(X_seq[:, :, 0], 0, NUM_BEACONS - 1))

            # graph augment
            X_graph[:, :, 1] += np.random.normal(0, 0.08, X_graph[:, :, 1].shape)   # rssi_norm
            X_graph[:, :, 3] *= np.random.uniform(0.9, 1.1, X_graph[:, :, 3].shape)  # rssi_distance
            drop_mask = np.random.random((X_graph.shape[0], NUM_BEACONS)) > 0.05
            X_graph[:, :, 0] *= drop_mask.astype(np.float32)
            g_noise = np.random.normal(0, 0.01, X_graph.shape)
            X_graph += g_noise

        # RSSI-only fine-tuning: re-zero IMU/motion channels AFTER augmentation so the
        # model never sees non-RSSI signal (augmentation would otherwise add tiny noise).
        if RSSI_ONLY_FINETUNE:
            mask_non_rssi_batch(X_seq, X_graph)

        # TUPLE return is required for multi-input models in TF 2.x
        return (X_seq, X_graph), {'y_output': y, 'floor_output': fl}

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


# ==============================================================================
# LOSSES
# ==============================================================================
def logcosh_euclidean(y_true, y_pred):
    # FIX: Clip distance to prevent cosh() overflow → NaN weights
    dist = tf.norm(y_true - y_pred, axis=-1)
    dist = tf.clip_by_value(dist, 0.0, 50.0)
    return tf.reduce_mean(tf.math.log(tf.cosh(dist)))

def mae_euclidean(y_true, y_pred):
    return tf.reduce_mean(tf.norm(y_true - y_pred, axis=-1))


# ==============================================================================
# MODEL  —  Dual-branch: Temporal Transformer + Beacon Graph Attention
# ==============================================================================
def build_encoder(seq_inp, graph_inp):
    """
    Shared dual-branch encoder (temporal Transformer + beacon graph attention -> 128-dim
    `shared` embedding). Extracted so the SAME encoder can be reused by both the supervised
    localizer and the self-supervised pretrainer. Every weighted layer has an EXPLICIT name
    so pretrained weights transfer via `load_weights(by_name=True)`. The layer topology is
    byte-identical to the original inline version (order-based weight loading still works).
    """
    # -------------------------------------------------------------------------
    # TEMPORAL BRANCH  (window_size time steps)
    # -------------------------------------------------------------------------
    beacon_ids = layers.Lambda(lambda x: tf.cast(x[:, :, 0], tf.int32))(seq_inp)
    beacon_emb = layers.Embedding(NUM_BEACONS, EMBED_DIM, name='enc_t_beacon_emb')(beacon_ids)

    rest = layers.Lambda(lambda x: x[:, :, 1:])(seq_inp)
    x_t = layers.Concatenate(axis=-1)([beacon_emb, rest])

    x_t = layers.Dense(128, activation='gelu', name='enc_t_proj')(x_t)
    x_t = layers.SpatialDropout1D(0.15)(x_t)

    for i in range(4):
        attn = layers.MultiHeadAttention(num_heads=8, key_dim=16, dropout=0.15,
                                         name=f'enc_t_mha_{i}')(x_t, x_t)
        x_t = layers.LayerNormalization(name=f'enc_t_ln_a_{i}')(x_t + attn)

        ffn = layers.Dense(256, activation='gelu', name=f'enc_t_ffn0_{i}')(x_t)
        ffn = layers.Dropout(0.15)(ffn)
        ffn = layers.Dense(128, name=f'enc_t_ffn1_{i}')(ffn)
        x_t = layers.LayerNormalization(name=f'enc_t_ln_b_{i}')(x_t + ffn)

    avg_t = layers.GlobalAveragePooling1D()(x_t)
    max_t = layers.GlobalMaxPooling1D()(x_t)
    temporal_emb = layers.Concatenate()([avg_t, max_t])          # 256 dim

    # -------------------------------------------------------------------------
    # BEACON GRAPH BRANCH  (NUM_BEACONS nodes)
    # -------------------------------------------------------------------------
    def tile_beacon_indices(x):
        return tf.tile(tf.expand_dims(tf.range(NUM_BEACONS, dtype=tf.int32), 0),
                       [tf.shape(x)[0], 1])

    beacon_idx_layer = layers.Lambda(tile_beacon_indices)(graph_inp)
    graph_beacon_emb = layers.Embedding(NUM_BEACONS, EMBED_DIM, name='enc_g_beacon_emb')(beacon_idx_layer)

    x_g = layers.Concatenate(axis=-1)([graph_beacon_emb, graph_inp])   # (B, 7, 16+6)
    x_g = layers.Dense(64, activation='gelu', name='enc_g_proj')(x_g)

    for i in range(2):
        attn_g = layers.MultiHeadAttention(num_heads=4, key_dim=16, dropout=0.1,
                                           name=f'enc_g_mha_{i}')(x_g, x_g)
        x_g = layers.LayerNormalization(name=f'enc_g_ln_a_{i}')(x_g + attn_g)

        ffn_g = layers.Dense(128, activation='gelu', name=f'enc_g_ffn0_{i}')(x_g)
        ffn_g = layers.Dropout(0.1)(ffn_g)
        ffn_g = layers.Dense(64, name=f'enc_g_ffn1_{i}')(ffn_g)
        x_g = layers.LayerNormalization(name=f'enc_g_ln_b_{i}')(x_g + ffn_g)

    avg_g = layers.GlobalAveragePooling1D()(x_g)
    max_g = layers.GlobalMaxPooling1D()(x_g)
    graph_emb = layers.Concatenate()([avg_g, max_g])                   # 128 dim

    # -------------------------------------------------------------------------
    # FUSION
    # -------------------------------------------------------------------------
    fused = layers.Concatenate()([temporal_emb, graph_emb])            # 384 dim
    shared = layers.Dense(256, activation='gelu', name='enc_shared0')(fused)
    shared = layers.BatchNormalization(epsilon=1e-4, name='enc_shared_bn0')(shared)
    shared = layers.Dropout(0.35)(shared)
    shared = layers.Dense(128, activation='gelu', name='enc_shared1')(shared)
    shared = layers.BatchNormalization(epsilon=1e-4, name='enc_shared_bn1')(shared)
    shared = layers.Dropout(0.25)(shared)
    return shared


def build_hybrid_localizer(name='hybrid_gat'):
    # -------------------------------------------------------------------------
    # Inputs
    # -------------------------------------------------------------------------
    seq_inp   = layers.Input(shape=(WINDOW_SIZE, N_FEATURES), name='sequence_input')
    graph_inp = layers.Input(shape=(NUM_BEACONS, N_BEACON_FEATS), name='beacon_graph_input')

    # Shared encoder (pretrainable) -> 128-dim embedding
    shared = build_encoder(seq_inp, graph_inp)

    # --- Y branch ---
    y_branch = layers.Dense(256, activation='gelu')(shared)
    y_branch = layers.BatchNormalization(epsilon=1e-4)(y_branch)
    y_branch = layers.Dropout(0.25)(y_branch)
    y_branch = layers.Dense(128, activation='gelu')(y_branch)
    y_branch = layers.BatchNormalization(epsilon=1e-4)(y_branch)
    y_branch = layers.Dropout(0.15)(y_branch)
    y_branch = layers.Dense(64, activation='gelu')(y_branch)
    y_out = layers.Dense(1, activation='linear', name='y_output')(y_branch)

    # --- Floor branch ---
    fl = layers.Dense(64, activation='gelu')(shared)
    fl = layers.Dropout(0.1)(fl)
    fl_out = layers.Dense(1, activation='sigmoid', name='floor_output')(fl)

    model = Model(inputs=[seq_inp, graph_inp],
                  outputs={'y_output': y_out, 'floor_output': fl_out},
                  name=name)

    # FIX: Use clipnorm only (Keras forbids combining clipnorm + clipvalue)
    model.compile(
        optimizer=optimizers.Adam(1.5e-4, clipnorm=1.0),
        loss={
            'y_output': logcosh_euclidean,
            'floor_output': 'binary_crossentropy',
        },
        loss_weights={
            'y_output': W_Y,
            'floor_output': W_FLOOR,
        },
        metrics={
            'y_output': [mae_euclidean],
            'floor_output': ['accuracy'],
        }
    )
    return model


# ==============================================================================
# SELF-SUPERVISED PRETRAINING  (masked reconstruction on trajectory walks)
# ==============================================================================
def build_ssl_model(name='ssl_pretrainer'):
    """
    Same shared encoder as build_hybrid_localizer, with a lightweight reconstruction
    decoder instead of the y/floor heads. The encoder layer NAMES match, so after
    pretraining the encoder weights transfer to the supervised model via
    `load_weights(by_name=True, skip_mismatch=True)`. Pretext: reconstruct the (clean)
    temporal `X` and beacon-graph `X_graph` from their masked versions -> the encoder
    must learn RSSI / IMU / beacon structure without any position labels.
    """
    seq_inp   = layers.Input(shape=(WINDOW_SIZE, N_FEATURES), name='sequence_input')
    graph_inp = layers.Input(shape=(NUM_BEACONS, N_BEACON_FEATS), name='beacon_graph_input')

    shared = build_encoder(seq_inp, graph_inp)                  # 128-dim (encoder, shared names)

    # --- decoder (distinct names -> skipped by by_name transfer) ---
    h = layers.Dense(256, activation='gelu', name='dec_hidden')(shared)
    h = layers.Dropout(0.1)(h)

    seq_rec = layers.Dense(WINDOW_SIZE * N_FEATURES, name='dec_seq_dense')(h)
    seq_rec = layers.Reshape((WINDOW_SIZE, N_FEATURES), name='recon_seq')(seq_rec)

    graph_rec = layers.Dense(NUM_BEACONS * N_BEACON_FEATS, name='dec_graph_dense')(h)
    graph_rec = layers.Reshape((NUM_BEACONS, N_BEACON_FEATS), name='recon_graph')(graph_rec)

    model = Model(inputs=[seq_inp, graph_inp],
                  outputs={'recon_seq': seq_rec, 'recon_graph': graph_rec},
                  name=name)
    model.compile(
        optimizer=optimizers.Adam(1.5e-4, clipnorm=1.0),
        loss={'recon_seq': 'mse', 'recon_graph': 'mse'},
        loss_weights={'recon_seq': 1.0, 'recon_graph': 1.0},
    )
    return model


class SSLMaskGenerator(Sequence):
    """
    Yields ((X_masked, Xg_masked), {'recon_seq': X, 'recon_graph': Xg}).
    A random SSL_MASK_RATE fraction of (timestep x channel) entries is zeroed in the
    inputs; the target is the clean (unmasked) tensor. Column 0 of X (beaconId) is
    NEVER masked and is re-quantised to a valid embedding index, mirroring the existing
    augmenters so the Embedding lookup stays valid.
    """
    def __init__(self, sequences, batch_size=BATCH_SIZE, mask_rate=SSL_MASK_RATE, shuffle=True):
        self.sequences = sequences
        self.batch_size = batch_size
        self.mask_rate = mask_rate
        self.shuffle = shuffle
        self.indices = np.arange(len(sequences))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.sequences) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        X  = np.array([self.sequences[i]['X'] for i in batch_idx], dtype=np.float32)
        Xg = np.array([self.sequences[i]['X_graph'] for i in batch_idx], dtype=np.float32)

        Xm  = X.copy()
        Xgm = Xg.copy()

        # Temporal masking (protect beaconId at channel 0).
        seq_mask = (np.random.random(Xm.shape) > self.mask_rate).astype(np.float32)
        seq_mask[:, :, 0] = 1.0
        Xm = Xm * seq_mask
        Xm[:, :, 0] = np.round(np.clip(Xm[:, :, 0], 0, NUM_BEACONS - 1))

        # Graph masking (all channels of a node may be dropped).
        g_mask = (np.random.random(Xgm.shape) > self.mask_rate).astype(np.float32)
        Xgm = Xgm * g_mask

        return (Xm, Xgm), {'recon_seq': X, 'recon_graph': Xg}

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


def transfer_encoder_weights(dst_model, weights_path):
    """
    Copy pretrained ENCODER weights (layers named 'enc_*') from a saved SSL checkpoint into
    `dst_model` (the supervised localizer). Version-agnostic: works on both Keras 2 and Keras 3
    (Keras 3 removed `load_weights(by_name=...)`, so we rebuild the SSL model, full-load its
    weights, and copy matching encoder layers by name via set_weights).
    """
    src = build_ssl_model()
    src.load_weights(weights_path)
    src_layers = {l.name: l for l in src.layers}
    n = 0
    for l in dst_model.layers:
        if l.name.startswith('enc_') and l.name in src_layers:
            sw = src_layers[l.name].get_weights()
            if sw:
                l.set_weights(sw)
                n += 1
    print(f"  Transferred {n} pretrained encoder layers (by name) into the supervised model")
    return n


def transform_sequences(seqs, scaler, scaler_g, y_mean, y_std):
    """
    Apply the LABELED-TRAIN scaler to a list of raw sequences in place (batched version of
    normalize_single_sequence). The trajectory scaler is never fitted -> fine-tuning stays
    leakage-free. Reuses the exact per-sequence transform used by the supervised path.
    """
    for s in seqs:
        normalize_single_sequence(s, scaler, scaler_g, y_mean, y_std)
    return seqs


def pretrain_encoder(scaler, scaler_g, y_stats,
                     beacon_median_map, floor_median_map, sensor_medians):
    """
    Full SSL pretraining stage: trajectory JSON -> CSV-schema rows -> shared preprocessing
    -> masked-reconstruction training -> save encoder weights to ENCODER_WEIGHTS_PATH.
    Uses the same train-only statistics/scaler as the labeled pipeline (no leakage).
    """
    print("\n  Loading trajectory data for SSL pretraining...")
    traj_df = load_all_trajectories(TRAJ_DATA_DIR)
    traj_df = minimal_clean(traj_df)
    traj_df = preprocess_data(traj_df, sensor_medians=sensor_medians)
    traj_df = engineer_features(traj_df, beacon_median_map=beacon_median_map,
                                floor_median_map=floor_median_map)

    # Guard against NaN/Inf leaking into the encoder (same check as the supervised path).
    assert not traj_df[FEATURE_COLS].isna().any().any(), "NaN in trajectory features!"
    assert not np.isinf(traj_df[FEATURE_COLS].values).any(), "Inf in trajectory features!"

    traj_seq = create_sequences(traj_df, window_size=WINDOW_SIZE, stride=WINDOW_STRIDE)
    if len(traj_seq) == 0:
        print("  WARNING: no trajectory sequences produced; skipping pretraining.")
        return False

    transform_sequences(traj_seq, scaler, scaler_g, *y_stats)

    print(f"  Pretraining encoder on {len(traj_seq)} trajectory windows "
          f"({PRETRAIN_EPOCHS} epochs, mask_rate={SSL_MASK_RATE})...")
    ssl_gen = SSLMaskGenerator(traj_seq, batch_size=BATCH_SIZE, shuffle=True)
    ssl_model = build_ssl_model()
    ssl_cb = [
        callbacks.TerminateOnNaN(),
        callbacks.EarlyStopping(monitor='loss', patience=15, restore_best_weights=True),
    ]
    ssl_model.fit(ssl_gen, epochs=PRETRAIN_EPOCHS, callbacks=ssl_cb, verbose=1)
    ssl_model.save_weights(ENCODER_WEIGHTS_PATH)
    print(f"  Saved pretrained encoder -> {ENCODER_WEIGHTS_PATH}")
    return True


# ==============================================================================
# CALLBACKS
# ==============================================================================
class HistoryCheckpoint(callbacks.Callback):
    def __init__(self, name):
        super().__init__()
        self.path = os.path.join(CACHE_MODELS_DIR, f'{name}.history.json')
        self.history = json_load(self.path) if os.path.exists(self.path) else {}

    def on_epoch_end(self, epoch, logs=None):
        for k, v in (logs or {}).items():
            self.history.setdefault(k, []).append(float(v))
        json_save(self.history, self.path)

    def get(self):
        return self.history

def cosine_warmup(epoch, lr, warmup=10, total=400, base_lr=1.5e-4, min_lr=1e-6):
    if epoch < warmup:
        return base_lr * (epoch + 1) / warmup
    progress = (epoch - warmup) / max(total - warmup, 1)
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * progress))

class CustomModel:
    def __init__(self, name, model):
        self.name = re.sub(r'[^A-Za-z0-9]', '_', name).lower()
        self.weights = os.path.join(CACHE_MODELS_DIR, f'{self.name}.weights.h5')
        self.model = model
        self.history = {}
        self._load()

    def _load(self):
        if os.path.exists(self.weights):
            self.model.load_weights(self.weights)
            self.trained = True
            print(f'  [{self.name}] weights loaded')
        else:
            self.trained = False
        hcp = HistoryCheckpoint(self.name)
        self.history = hcp.get()

    def fit(self, train_gen, val_gen, epochs=400, force=False):
        if self.trained and not force:
            print(f'  [{self.name}] already trained, skipping')
            return
        hcp = HistoryCheckpoint(self.name)
        # FIX: Added TerminateOnNaN to stop immediately on first corrupted batch
        cb = [
            callbacks.TerminateOnNaN(),
            callbacks.EarlyStopping(monitor='val_loss', patience=35, restore_best_weights=True),
            callbacks.ModelCheckpoint(self.weights, monitor='val_loss', save_best_only=True, save_weights_only=True),
            callbacks.LearningRateScheduler(lambda e, lr: cosine_warmup(e, lr, total=epochs)),
            hcp,
        ]
        self.model.fit(train_gen, validation_data=val_gen, epochs=epochs, callbacks=cb, verbose=1)
        self.trained = True
        self.history = hcp.get()


# ==============================================================================
# kNN FINGERPRINT RETRIEVAL  (train-only database)
# ==============================================================================
class FingerprintKNN:
    def __init__(self, k=KNN_K):
        self.k = k
        self.nn = None
        self.X_sigs = None
        self.y_vals = None

    def fit(self, train_seq):
        fp_data = defaultdict(lambda: {'sigs': [], 'y': None, 'floor': None})
        for s in train_seq:
            key = (s['session'], s['fingerprintId'])
            sig = extract_rssi_signature(s['X'], s['beacon_ids'])
            fp_data[key]['sigs'].append(sig)
            fp_data[key]['y'] = s['y']
            fp_data[key]['floor'] = s['floor_class']

        keys = []
        X_sigs = []
        y_vals = []
        for key, data in fp_data.items():
            keys.append(key)
            X_sigs.append(np.mean(data['sigs'], axis=0))
            y_vals.append(data['y'])

        self.keys = keys
        self.X_sigs = np.array(X_sigs)
        self.y_vals = np.array(y_vals)

        k_eff = min(self.k, len(X_sigs))
        self.nn = NearestNeighbors(n_neighbors=k_eff, metric='euclidean')
        self.nn.fit(self.X_sigs)
        print(f"  kNN DB: {len(X_sigs)} fingerprints, k={k_eff}")

    def predict(self, test_seq):
        sigs = np.array([extract_rssi_signature(s['X'], s['beacon_ids']) for s in test_seq])
        k_eff = min(self.k, len(self.X_sigs))
        dist, idx = self.nn.kneighbors(sigs, n_neighbors=k_eff)

        w = 1.0 / (dist + 1e-6)
        w = w / w.sum(axis=1, keepdims=True)
        y_pred = np.sum(self.y_vals[idx] * w, axis=1)
        return y_pred


# ==============================================================================
# REAL-TIME KALMAN FILTER (Causal, Forward-Only)
# ==============================================================================
class OneDKalmanFilter:
    """Forward-only Kalman filter suitable for streaming / real-time use."""
    # FIX: Accept both Q/R and process_noise/measurement_noise to match all call sites
    def __init__(self, Q=None, R=None, process_noise=None, measurement_noise=None):
        if Q is not None:
            process_noise = Q
        if R is not None:
            measurement_noise = R
        self.Q = float(process_noise) if process_noise is not None else float(KALMAN_Q)
        self.R = float(measurement_noise) if measurement_noise is not None else float(KALMAN_R)
        self.x = 0.0
        self.P = 1.0
        self.initialized = False

    def reset(self):
        self.initialized = False
        self.x = 0.0
        self.P = 1.0

    def update(self, measurement):
        if not self.initialized:
            self.x = float(measurement)
            self.P = 1.0
            self.initialized = True
            return self.x

        # Predict
        x_pred = self.x
        P_pred = self.P + self.Q
        # Update
        K = P_pred / (P_pred + self.R)
        self.x = x_pred + K * (measurement - x_pred)
        self.P = (1.0 - K) * P_pred
        return self.x


# ==============================================================================
# BATCH KALMAN SMOOTHER (Forward-Backward RTS)
# ==============================================================================
class OneDKalmanSmoother:
    """Non-causal RTS smoother for offline post-processing."""
    def __init__(self, process_noise=None, measurement_noise=None, Q=None, R=None):
        if Q is not None:
            process_noise = Q
        if R is not None:
            measurement_noise = R
        self.Q = float(process_noise) if process_noise is not None else float(KALMAN_Q)
        self.R = float(measurement_noise) if measurement_noise is not None else float(KALMAN_R)

    def smooth(self, measurements):
        n = len(measurements)
        if n == 0:
            return []
        # Forward pass
        x = np.zeros(n)
        P = np.zeros(n)
        x[0] = float(measurements[0])
        P[0] = 1.0
        for t in range(1, n):
            x_pred = x[t-1]
            P_pred = P[t-1] + self.Q
            K = P_pred / (P_pred + self.R)
            x[t] = x_pred + K * (measurements[t] - x_pred)
            P[t] = (1.0 - K) * P_pred

        # Backward RTS pass
        x_s = x.copy()
        for t in range(n - 2, -1, -1):
            denom = P[t] + self.Q
            C = P[t] / denom if denom > 1e-9 else 0.0
            x_s[t] = x[t] + C * (x_s[t+1] - x[t])
        return x_s.tolist()


# ==============================================================================
# PER-TRAJECTORY MAD OUTLIER REJECTION
# ==============================================================================
class MADOutlierRejector:
    """Median-Absolute-Deviation outlier rejector for 1-D trajectories."""
    def __init__(self, window=10, z_thresh=3.0):
        self.window = window
        self.z_thresh = z_thresh
        self.history = []

    def reset(self):
        self.history = []

    def update(self, value):
        self.history.append(float(value))
        if len(self.history) > self.window:
            self.history.pop(0)
        if len(self.history) < 3:
            return value

        arr = np.array(self.history)
        med = np.median(arr)
        mad = np.median(np.abs(arr - med))
        if mad < 1e-6:
            return value

        z = np.abs(value - med) / (1.4826 * mad)
        if z > self.z_thresh:
            # Reject: return previous accepted value (or median if unavailable)
            return self.history[-2] if len(self.history) >= 2 else med
        return value

def apply_mad_per_trajectory(y_pred, test_seq, z_thresh=3.0):
    """Batch helper: apply MAD cleaning per fingerprint trajectory."""
    fp_data = defaultdict(list)
    for i, s in enumerate(test_seq):
        key = (s['session'], s['fingerprintId'])
        fp_data[key].append(i)

    y_clean = y_pred.copy()
    for indices in fp_data.values():
        if len(indices) < 5:
            continue
        vals = y_pred[indices]
        med = np.median(vals)
        mad = np.median(np.abs(vals - med))
        if mad < 1e-6:
            continue
        for idx in indices:
            z = np.abs(y_pred[idx] - med) / (1.4826 * mad)
            if z > z_thresh:
                y_clean[idx] = med
    return y_clean


# ==============================================================================
# REAL-TIME INFERENCE ENGINE
# ==============================================================================
class RealTimeLocalizer:
    """
    Production-grade real-time localizer.
    Processes single windows causally: no future data, no RTS backward pass.
    """
    def __init__(self, model, scaler, scaler_g, y_stats,
                 beacon_median_map, floor_median_map, sensor_medians,
                 use_tta=False, use_knn=False, knn_db=None,
                 kalman_q=KALMAN_Q, kalman_r=KALMAN_R,
                 outlier_z=3.0, outlier_window=10):
        self.model = model
        self.scaler = scaler
        self.scaler_g = scaler_g
        self.y_mean, self.y_std = y_stats
        self.y_stats = y_stats
        self.beacon_median_map = beacon_median_map
        self.floor_median_map = floor_median_map
        self.sensor_medians = sensor_medians
        self.use_tta = use_tta
        self.use_knn = use_knn
        self.knn_db = knn_db

        self.kalman = OneDKalmanFilter(Q=kalman_q, R=kalman_r)
        self.outlier_rejector = MADOutlierRejector(window=outlier_window, z_thresh=outlier_z)
        self.last_floor = None
        self.last_y = None

    def reset(self):
        self.kalman.reset()
        self.outlier_rejector.reset()
        self.last_floor = None
        self.last_y = None

    def _infer_floor_from_beacons(self, beacon_ids):
        floors = [BEACON_FLOOR.get(int(b), 0) for b in beacon_ids if int(b) in BEACON_FLOOR]
        if not floors:
            return self.last_floor if self.last_floor is not None else 3
        return int(max(set(floors), key=floors.count))

    def _prepare_window_df(self, window_df, assumed_floor=None):
        df = window_df.copy()

        # Ensure beaconId exists
        if 'beaconId' not in df.columns and 'beaconUid' in df.columns:
            df['beaconId'] = df['beaconUid'].map(BEACON_MAP)
        df = df.dropna(subset=['beaconId'])
        if len(df) == 0:
            return df
        df['beaconId'] = df['beaconId'].astype(int)

        # Infer floor if missing (needed for floor-dependent features)
        if 'floorLevel' not in df.columns or df['floorLevel'].isna().all():
            if assumed_floor is None:
                assumed_floor = self._infer_floor_from_beacons(df['beaconId'].values)
            df['floorLevel'] = assumed_floor

        # Ensure metadata columns for feature engineering
        if 'session_name' not in df.columns:
            df['session_name'] = 'live'
        if 'fingerprintId' not in df.columns:
            df['fingerprintId'] = 0
        if 'windowIndex' not in df.columns:
            df['windowIndex'] = 0
        if 'capturedAt' not in df.columns:
            df['capturedAt'] = pd.Timestamp.now()
        if 'y' not in df.columns:
            df['y'] = 0.0
        if 'x' not in df.columns:
            df['x'] = 0.0

        df = preprocess_data(df, sensor_medians=self.sensor_medians)
        df = engineer_features(df, beacon_median_map=self.beacon_median_map,
                               floor_median_map=self.floor_median_map)
        return df

    def predict_window(self, window_df, assumed_floor=None, return_floor=False):
        """Run neural network (and optional TTA/kNN) on a single live window."""
        df = self._prepare_window_df(window_df, assumed_floor)
        if len(df) == 0:
            return (None, None) if return_floor else None

        seq = create_single_sequence(df)
        if seq is None:
            return (None, None) if return_floor else None

        normalize_single_sequence(seq, self.scaler, self.scaler_g, self.y_mean, self.y_std)
        # Match fine-tuning: deployed model uses RSSI only (phone can pass zeros for IMU).
        if RSSI_ONLY_FINETUNE:
            mask_non_rssi(seq)

        X_seq = np.expand_dims(seq['X'], 0)
        X_graph = np.expand_dims(seq['X_graph'], 0)

        # Optional kNN fallback (uses raw RSSI signature; disabled by default)
        y_knn = None
        if self.use_knn and self.knn_db is not None:
            sig = extract_rssi_signature(seq['X'], seq['beacon_ids'])
            sigs = np.expand_dims(sig, 0)
            k_eff = min(self.knn_db.k, len(self.knn_db.X_sigs))
            if k_eff > 0:
                dist, idx = self.knn_db.nn.kneighbors(sigs, n_neighbors=k_eff)
                w = 1.0 / (dist + 1e-6)
                w = w / w.sum(axis=1, keepdims=True)
                y_knn = np.sum(self.knn_db.y_vals[idx] * w, axis=1)[0]

        # Neural network prediction
        if self.use_tta:
            y_pred, fl_prob = predict_with_tta(self.model, X_seq, X_graph, self.y_stats, n_tta=10)
            y_pred = float(y_pred[0])
            fl_prob = float(fl_prob[0])
        else:
            out = self.model.predict([X_seq, X_graph], batch_size=1, verbose=0)
            y_pred = float(out['y_output'][0, 0] * self.y_std + self.y_mean)
            fl_prob = float(out['floor_output'][0, 0])

        y_pred = float(np.clip(y_pred, Y_MIN, Y_MAX))
        fl_pred = 1 if fl_prob >= 0.5 else 0
        fl_label = FLOOR_CLASSES_INV[fl_pred]

        # Sanity-check blend with kNN only if wildly divergent
        if y_knn is not None and abs(y_pred - y_knn) > 10.0:
            y_pred = y_knn

        self.last_floor = fl_label
        self.last_y = y_pred
        return (y_pred, fl_label) if return_floor else y_pred

    def update(self, window_df, assumed_floor=None):
        """
        Full real-time update: NN → outlier rejection → causal Kalman filter.
        Returns (y_smooth, floor_label).
        """
        y_raw, fl = self.predict_window(window_df, assumed_floor, return_floor=True)
        if y_raw is None:
            return None, None

        # Outlier rejection before Kalman (protects filter from spikes)
        y_clean = self.outlier_rejector.update(y_raw)

        # Causal Kalman filter
        y_smooth = self.kalman.update(y_clean)

        self.last_y = y_smooth
        return float(y_smooth), int(fl)


# ==============================================================================
# STREAMING LOCALIZER (sliding buffer for continuous sensor feeds)
# ==============================================================================
class StreamingLocalizer:
    """
    Wraps RealTimeLocalizer with a rolling buffer for continuous sensor streams.
    Accepts individual beacon observations and emits a prediction every N rows.
    """
    def __init__(self, localizer, window_size=WINDOW_SIZE, min_rows=4):
        self.localizer = localizer
        self.window_size = window_size
        self.min_rows = min_rows
        self.buffer = []
        self.obs_count = 0
        self.session_name = 'live_stream'
        self.fingerprint_id = 0
        self.window_index = 0

    def reset(self):
        self.buffer = []
        self.obs_count = 0
        self.window_index = 0
        self.localizer.reset()

    def add_observation(self, beacon_uid, rssi, timestamp=None,
                        gyro=None, accel=None, user_accel=None,
                        mag=None, pitch=None, roll=None, yaw=None,
                        floor_level=None):
        """Add a single beacon observation to the rolling buffer."""
        row = {
            'beaconUid': beacon_uid,
            'rssi': rssi,
            'capturedAt': pd.Timestamp(timestamp) if timestamp else pd.Timestamp.now(),
            'session_name': self.session_name,
            'fingerprintId': self.fingerprint_id,
            'windowIndex': self.window_index,
        }
        if floor_level is not None:
            row['floorLevel'] = floor_level
        if gyro is not None:
            row['gyroX'], row['gyroY'], row['gyroZ'] = gyro
        if accel is not None:
            row['accelX'], row['accelY'], row['accelZ'] = accel
        if user_accel is not None:
            row['userAccelX'], row['userAccelY'], row['userAccelZ'] = user_accel
        if mag is not None:
            row['magX'], row['magY'], row['magZ'] = mag
        if pitch is not None:
            row['pitch'] = pitch
        if roll is not None:
            row['roll'] = roll
        if yaw is not None:
            row['yaw'] = yaw

        self.buffer.append(row)
        self.obs_count += 1

        # Auto-advance window if buffer exceeds size
        if len(self.buffer) > self.window_size * 2:
            self.advance_window()

    def advance_window(self):
        """Advance to next window index; old buffer rows are discarded."""
        self.window_index += 1
        self.buffer = []
        self.fingerprint_id += 1

    def predict(self, force=False):
        """
        Emit prediction if buffer has enough rows.
        force=True forces prediction even with fewer rows.
        """
        if len(self.buffer) < self.min_rows and not force:
            return None, None
        if len(self.buffer) < self.min_rows:
            return None, None

        df = pd.DataFrame(self.buffer)
        y, fl = self.localizer.update(df)
        return y, fl


# ==============================================================================
# TEST-TIME AUGMENTATION (dual input)
# ==============================================================================
def predict_with_tta(model, X_seq, X_graph, y_stats, n_tta=20):
    y_mean, y_std = y_stats
    preds_y = []
    preds_fl = []

    rssi_idx = FEATURE_COLS.index('rssi') if 'rssi' in FEATURE_COLS else 1

    for i in range(n_tta):
        Xs_aug = X_seq.copy()
        Xg_aug = X_graph.copy()
        if i > 0:
            # Protect beaconId (column 0) from corruption during TTA
            Xs_aug[:, :, rssi_idx] += np.random.normal(0, 0.12, Xs_aug[:, :, rssi_idx].shape)

            mask = np.random.random((Xs_aug.shape[0], Xs_aug.shape[1])) > 0.15
            mask = mask[:, :, np.newaxis]          # (B, T, 1)

            # Apply mask only to non-beaconId columns (broadcasts correctly)
            Xs_aug[:, :, 1:] = Xs_aug[:, :, 1:] * mask

            noise = np.random.normal(0, 0.02, Xs_aug.shape)
            noise[:, :, 0] = 0.0
            Xs_aug += noise
            Xs_aug[:, :, 0] = np.round(np.clip(Xs_aug[:, :, 0], 0, NUM_BEACONS - 1))

            Xg_aug[:, :, 1] += np.random.normal(0, 0.08, Xg_aug[:, :, 1].shape)
            Xg_aug[:, :, 3] *= np.random.uniform(0.9, 1.1, Xg_aug[:, :, 3].shape)
            drop_mask = np.random.random((Xg_aug.shape[0], NUM_BEACONS)) > 0.08
            Xg_aug[:, :, 0] *= drop_mask.astype(np.float32)
            Xg_aug += np.random.normal(0, 0.015, Xg_aug.shape)

            # Keep IMU/motion channels zeroed under RSSI-only mode (TTA noise would un-zero them)
            if RSSI_ONLY_FINETUNE:
                mask_non_rssi_batch(Xs_aug, Xg_aug)

        out = model.predict([Xs_aug, Xg_aug], batch_size=BATCH_SIZE, verbose=0)
        preds_y.append(out['y_output'])
        preds_fl.append(out['floor_output'])

    y_med = np.median(preds_y, axis=0)
    fl_med = np.median(preds_fl, axis=0)

    y_pred = y_med[:, 0] * y_std + y_mean
    y_pred = np.clip(y_pred, Y_MIN, Y_MAX)

    return y_pred, fl_med


# ==============================================================================
# POST-PROCESSING & EVALUATION
# ==============================================================================
def geometric_postprocess(y_pred, test_seq, train_seq, y_stats):
    train_ys = np.array(sorted(set(s['y'] for s in train_seq)))
    train_min, train_max = train_ys.min(), train_ys.max()

    snapped = y_pred.copy()
    n_clamped = 0
    for i, y in enumerate(y_pred):
        if y < train_min - 3.0 or y > train_max + 3.0:
            snapped[i] = np.clip(y, train_min - 2.0, train_max + 2.0)
            n_clamped += 1

    if n_clamped > 0:
        print(f"    Corridor clamp: fixed {n_clamped} out-of-bounds predictions")
    return snapped

def fingerprint_kalman_smooth(y_pred, test_seq, process_noise=KALMAN_Q, measurement_noise=KALMAN_R):
    fp_data = defaultdict(lambda: {'indices': [], 'y_preds': [], 'timestamp': None, 'y_true': None})
    for i, s in enumerate(test_seq):
        key = (s['session'], s['fingerprintId'])
        fp_data[key]['indices'].append(i)
        fp_data[key]['y_preds'].append(y_pred[i])
        if fp_data[key]['timestamp'] is None:
            fp_data[key]['timestamp'] = s.get('timestamp', 0)
        fp_data[key]['y_true'] = s['y']

    fp_y = {k: np.median(v['y_preds']) for k, v in fp_data.items()}

    sessions = defaultdict(list)
    for key, data in fp_data.items():
        sessions[key[0]].append({
            'key': key,
            'ts': data['timestamp'],
            'indices': data['indices'],
        })

    y_smooth = y_pred.copy()
    smoother = OneDKalmanSmoother(process_noise=process_noise, measurement_noise=measurement_noise)

    for session, items in sessions.items():
        if len(items) < 3:
            continue
        items.sort(key=lambda x: x['ts'])
        y_seq = [fp_y[it['key']] for it in items]
        y_smooth_seq = smoother.smooth(y_seq)
        for it, ys in zip(items, y_smooth_seq):
            for idx in it['indices']:
                y_smooth[idx] = ys

    return y_smooth

def evaluate_model(model_wrapper, test_seq, y_stats, train_seq, val_seq=None):
    y_mean, y_std = y_stats

    X_seq_test   = np.array([s['X'] for s in test_seq], dtype=np.float32)
    X_graph_test = np.array([s['X_graph'] for s in test_seq], dtype=np.float32)
    y_test       = np.array([s['y'] for s in test_seq], dtype=np.float32)
    fl_test      = np.array([s['floor_class'] for s in test_seq], dtype=np.float32)

    # -------------------------------------------------------------------------
    # 1. Build train-only kNN database
    # -------------------------------------------------------------------------
    knn_db = FingerprintKNN(k=KNN_K)
    knn_db.fit(train_seq)

    # -------------------------------------------------------------------------
    # 2. Raw NN prediction
    # -------------------------------------------------------------------------
    out_raw = model_wrapper.model.predict([X_seq_test, X_graph_test], batch_size=BATCH_SIZE, verbose=0)
    y_nn_raw = out_raw['y_output'][:, 0] * y_std + y_mean
    y_nn_raw = np.clip(y_nn_raw, Y_MIN, Y_MAX)

    # -------------------------------------------------------------------------
    # 3. TTA NN prediction
    # -------------------------------------------------------------------------
    y_nn_tta, fl_tta = predict_with_tta(model_wrapper.model, X_seq_test, X_graph_test, y_stats, n_tta=20)

    # -------------------------------------------------------------------------
    # 4. kNN retrieval prediction
    # -------------------------------------------------------------------------
    y_knn = knn_db.predict(test_seq)

    # -------------------------------------------------------------------------
    # 5. Tune blend weight on validation set (leakage-free)
    # -------------------------------------------------------------------------
    if val_seq is not None and len(val_seq) > 0:
        X_seq_val   = np.array([s['X'] for s in val_seq], dtype=np.float32)
        X_graph_val = np.array([s['X_graph'] for s in val_seq], dtype=np.float32)
        y_val       = np.array([s['y'] for s in val_seq], dtype=np.float32)

        out_val = model_wrapper.model.predict([X_seq_val, X_graph_val], batch_size=BATCH_SIZE, verbose=0)
        y_nn_val = out_val['y_output'][:, 0] * y_std + y_mean

        y_knn_val = knn_db.predict(val_seq)

        best_w = 0.5
        best_mae = float('inf')
        for w in np.arange(0.0, 1.01, 0.05):
            mae = np.abs(y_val - (w * y_nn_val + (1 - w) * y_knn_val)).mean()
            if mae < best_mae:
                best_mae = mae
                best_w = w
        print(f"  Tuned NN/kNN blend weight: {best_w:.2f} (val MAE={best_mae:.3f}m)")
    else:
        best_w = 0.70

    # -------------------------------------------------------------------------
    # 6. Hybrid prediction
    # -------------------------------------------------------------------------
    y_hybrid = best_w * y_nn_tta + (1.0 - best_w) * y_knn

    # -------------------------------------------------------------------------
    # 7. Geometric clamp
    # -------------------------------------------------------------------------
    y_geom = geometric_postprocess(y_hybrid, test_seq, train_seq, y_stats)

    # -------------------------------------------------------------------------
    # 8. Kalman trajectory smoothing (tune Q/R on val if available)
    # -------------------------------------------------------------------------
    if val_seq is not None and len(val_seq) > 0:
        y_knn_val = knn_db.predict(val_seq)
        y_hybrid_val = best_w * y_nn_val + (1.0 - best_w) * y_knn_val
        y_geom_val = geometric_postprocess(y_hybrid_val, val_seq, train_seq, y_stats)

        best_q, best_r = KALMAN_Q, KALMAN_R
        best_mae = float('inf')
        for Q in [0.01, 0.05, 0.1, 0.2, 0.5]:
            for R in [0.5, 1.0, 2.0, 4.0]:
                y_sm_val = fingerprint_kalman_smooth(y_geom_val, val_seq, process_noise=Q, measurement_noise=R)
                mae = np.abs(y_val - y_sm_val).mean()
                if mae < best_mae:
                    best_mae = mae
                    best_q, best_r = Q, R
        print(f"  Tuned Kalman Q={best_q}, R={best_r} (val MAE={best_mae:.3f}m)")
    else:
        best_q, best_r = KALMAN_Q, KALMAN_R

    y_final = fingerprint_kalman_smooth(y_geom, test_seq, process_noise=best_q, measurement_noise=best_r)

    # -------------------------------------------------------------------------
    # Metrics
    # -------------------------------------------------------------------------
    def report(y_pred, label):
        err = np.abs(y_test - y_pred)
        mae = err.mean()
        acc_1m = (err <= 1.0).mean() * 100
        acc_2m = (err <= 2.0).mean() * 100
        acc_5m = (err <= 5.0).mean() * 100
        fl_cls = (fl_tta.flatten() >= 0.5).astype(int)
        fl_acc = (fl_cls == fl_test).mean() * 100
        print(f"\n  [{label}]")
        print(f"    Y MAE : {mae:.3f}m")
        print(f"    Acc@1m: {acc_1m:.1f}%  |  Acc@2m: {acc_2m:.1f}%  |  Acc@5m: {acc_5m:.1f}%")
        print(f"    Floor Accuracy : {fl_acc:.1f}%")
        return mae, acc_1m, acc_2m, acc_5m, fl_acc, err

    _, _, _, _, _, _ = report(y_nn_raw,  'NN raw')
    _, _, _, _, _, _ = report(y_nn_tta, 'NN + TTA')
    _, _, _, _, _, _ = report(y_hybrid,  'NN + TTA + kNN')
    mae_final, acc1, acc2, acc5, fl_acc, err_final = report(y_final, 'NN + TTA + kNN + Kalman')

    # Error analysis
    print(f"\n  ── Error Analysis ──")
    print(f"    Median Y Error : {np.median(err_final):.3f}m")
    print(f"    90th percentile: {np.percentile(err_final, 90):.3f}m")
    print(f"    95th percentile: {np.percentile(err_final, 95):.3f}m")
    print(f"    Max Y Error    : {err_final.max():.3f}m")

    for floor in [3, 4]:
        mask = np.array([s['floor_class'] for s in test_seq]) == FLOOR_CLASSES[floor]
        if mask.sum() > 0:
            print(f"    Floor {floor} MAE  : {err_final[mask].mean():.3f}m (n={mask.sum()})")

    # Save CSV
    eval_df = pd.DataFrame([{
        'model': 'hybrid_gat_kalman',
        'mae_final': mae_final,
        'acc_1m': acc1,
        'acc_2m': acc2,
        'acc_5m': acc5,
        'floor_acc': fl_acc,
        'kalman_Q': best_q,
        'kalman_R': best_r,
        'nn_weight': best_w,
    }])
    eval_df.to_csv(EVAL_FILE_PATH, index=False)

    return {
        'model': 'hybrid_gat_kalman',
        'mae_final': mae_final,
        'acc_1m': acc1,
        'acc_2m': acc2,
        'acc_5m': acc5,
        'floor_acc': fl_acc,
        'y_pred': y_final,
        'y_err': err_final,
    }


# ==============================================================================
# REAL-TIME (CAUSAL) EVALUATION — NO FUTURE DATA
# ==============================================================================
def evaluate_realtime(model_wrapper, test_seq, y_stats, train_seq, val_seq=None,
                      kalman_q=KALMAN_Q, kalman_r=KALMAN_R,
                      outlier_z=3.0, outlier_window=10):
    """
    Evaluates the model in a strictly causal, real-time mode:
      • Forward-only Kalman filter (no RTS backward pass)
      • Per-trajectory MAD outlier rejection
      • No TTA, no kNN (fast single-pass)
    """
    y_mean, y_std = y_stats

    X_seq_test   = np.array([s['X'] for s in test_seq], dtype=np.float32)
    X_graph_test = np.array([s['X_graph'] for s in test_seq], dtype=np.float32)
    y_test       = np.array([s['y'] for s in test_seq], dtype=np.float32)
    fl_test      = np.array([s['floor_class'] for s in test_seq], dtype=np.float32)

    # Single-pass NN prediction (fast, no TTA)
    out = model_wrapper.model.predict([X_seq_test, X_graph_test], batch_size=BATCH_SIZE, verbose=0)
    y_nn = out['y_output'][:, 0] * y_std + y_mean
    y_nn = np.clip(y_nn, Y_MIN, Y_MAX)
    fl_prob = out['floor_output'][:, 0]
    fl_cls = (fl_prob >= 0.5).astype(int)

    # --- Step 1: MAD outlier rejection per trajectory ---
    y_mad = apply_mad_per_trajectory(y_nn, test_seq, z_thresh=outlier_z)

    # --- Step 2: Causal forward-only Kalman per session ---
    fp_data = defaultdict(list)
    for i, s in enumerate(test_seq):
        key = (s['session'], s['fingerprintId'])
        fp_data[key].append(i)

    sessions = defaultdict(list)
    for key, indices in fp_data.items():
        sessions[key[0]].append({
            'indices': indices,
            'ts': test_seq[indices[0]]['timestamp'],
        })

    y_causal = y_mad.copy()
    for session, items in sessions.items():
        if len(items) < 2:
            continue
        items.sort(key=lambda x: x['ts'])
        # FIX: Q/R kwargs now accepted by OneDKalmanFilter
        kf = OneDKalmanFilter(Q=kalman_q, R=kalman_r)
        for it in items:
            for idx in it['indices']:
                y_causal[idx] = kf.update(y_mad[idx])

    # --- Step 3: Geometric clamp (train-derived bounds) ---
    y_final = geometric_postprocess(y_causal, test_seq, train_seq, y_stats)

    # --- Metrics ---
    def report(y_pred, label):
        err = np.abs(y_test - y_pred)
        mae = err.mean()
        acc_1m = (err <= 1.0).mean() * 100
        acc_2m = (err <= 2.0).mean() * 100
        acc_5m = (err <= 5.0).mean() * 100
        fl_acc = (fl_cls == fl_test).mean() * 100
        print(f"\n  [{label}]")
        print(f"    Y MAE : {mae:.3f}m")
        print(f"    Acc@1m: {acc_1m:.1f}%  |  Acc@2m: {acc_2m:.1f}%  |  Acc@5m: {acc_5m:.1f}%")
        print(f"    Floor Accuracy : {fl_acc:.1f}%")
        return mae, acc_1m, acc_2m, acc_5m, fl_acc, err

    mae_final, acc1, acc2, acc5, fl_acc, err_final = report(y_final, 'REAL-TIME (Causal + MAD + KF)')

    print(f"\n  ── Real-Time Error Analysis ──")
    print(f"    Median Y Error : {np.median(err_final):.3f}m")
    print(f"    90th percentile: {np.percentile(err_final, 90):.3f}m")
    print(f"    95th percentile: {np.percentile(err_final, 95):.3f}m")
    print(f"    Max Y Error    : {err_final.max():.3f}m")

    for floor in [3, 4]:
        mask = np.array([s['floor_class'] for s in test_seq]) == FLOOR_CLASSES[floor]
        if mask.sum() > 0:
            print(f"    Floor {floor} MAE  : {err_final[mask].mean():.3f}m (n={mask.sum()})")

    return {
        'model': 'realtime_causal',
        'mae_final': mae_final,
        'acc_1m': acc1,
        'acc_2m': acc2,
        'acc_5m': acc5,
        'floor_acc': fl_acc,
        'y_pred': y_final,
        'y_err': err_final,
    }


# ==============================================================================
# MAIN
# ==============================================================================
def main(realtime_mode=False):
    print("\n" + "=" * 70)
    print("BEACON INDOOR LOCALIZATION — GRAPH + kNN + KALMAN PIPELINE")
    print("=" * 70)
    if realtime_mode:
        print("  [REAL-TIME MODE: causal Kalman + MAD outlier rejection]")

    print("\n── STEP 1: LOAD ──────────────────────────────────────────────────")
    df = load_all_datasets(data_dir='/content/drive/MyDrive/beacons_chaos/data')

    print("\n── STEP 2: MINIMAL CLEAN ─────────────────────────────────────────")
    df = minimal_clean(df)
    print(f"  NUM_BEACONS = {NUM_BEACONS} (fixed)")

    print("\n── STEP 3: SPLIT ─────────────────────────────────────────────────")
    unique_sessions = df[['session_name', 'fingerprintId']].drop_duplicates()
    shuffled = unique_sessions.sample(frac=1, random_state=42).reset_index(drop=True)

    n = len(shuffled)
    n_test = int(n * TEST_SPLIT)
    n_val  = int(n * VAL_SPLIT)

    test_keys_df  = shuffled.iloc[:n_test]
    val_keys_df   = shuffled.iloc[n_test:n_test + n_val]
    train_keys_df = shuffled.iloc[n_test + n_val:]

    train_keys_set = set(zip(train_keys_df['session_name'], train_keys_df['fingerprintId']))
    val_keys_set   = set(zip(val_keys_df['session_name'], val_keys_df['fingerprintId']))
    test_keys_set  = set(zip(test_keys_df['session_name'], test_keys_df['fingerprintId']))

    train_df = df[df.apply(lambda row: (row['session_name'], row['fingerprintId']) in train_keys_set, axis=1)].copy()
    val_df   = df[df.apply(lambda row: (row['session_name'], row['fingerprintId']) in val_keys_set, axis=1)].copy()
    test_df  = df[df.apply(lambda row: (row['session_name'], row['fingerprintId']) in test_keys_set, axis=1)].copy()

    print(f"  Rows: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

    print("\n── STEP 4: TRAIN-ONLY STATISTICS ─────────────────────────────────")
    beacon_median_train = train_df.groupby('beaconId')['rssi'].median().to_dict()

    floor_median_train = {}
    for (bid, flvl), grp in train_df.groupby(['beaconId', 'floorLevel'])['rssi']:
        floor_median_train[(bid, flvl)] = grp.median()

    sensor_cols = [
        'gyroX', 'gyroY', 'gyroZ', 'accelX', 'accelY', 'accelZ',
        'userAccelX', 'userAccelY', 'userAccelZ',
        'magX', 'magY', 'magZ', 'pitch', 'roll', 'yaw'
    ]
    sensor_medians = {
        col: float(train_df[col].median())
        for col in sensor_cols
        if col in train_df.columns
    }

    print(f"  Beacon medians: {len(beacon_median_train)}, Floor-beacon medians: {len(floor_median_train)}")

    print("\n── STEP 5: PREPROCESS ────────────────────────────────────────────")
    train_df = preprocess_data(train_df, sensor_medians=sensor_medians)
    val_df   = preprocess_data(val_df, sensor_medians=sensor_medians)
    test_df  = preprocess_data(test_df, sensor_medians=sensor_medians)

    print("\n── STEP 6: ENGINEER FEATURES ─────────────────────────────────────")
    train_df = engineer_features(train_df, beacon_median_map=beacon_median_train, floor_median_map=floor_median_train)
    val_df   = engineer_features(val_df,   beacon_median_map=beacon_median_train, floor_median_map=floor_median_train)
    test_df  = engineer_features(test_df,  beacon_median_map=beacon_median_train, floor_median_map=floor_median_train)

    df_final = pd.concat([train_df, val_df, test_df], ignore_index=True)
    finalise_feature_cols(df_final)

    # FIX: Hard guard against NaN/Inf entering the model (catches engineering bugs early)
    assert not df_final[FEATURE_COLS].isna().any().any(), "NaN detected in engineered features!"
    assert not np.isinf(df_final[FEATURE_COLS].values).any(), "Inf detected in engineered features!"

    all_sequences = create_sequences(df_final, window_size=WINDOW_SIZE, stride=WINDOW_STRIDE)

    train_seq, val_seq, test_seq = [], [], []
    for seq in all_sequences:
        key = (seq['session'], seq['fingerprintId'])
        if key in test_keys_set:
            test_seq.append(seq)
        elif key in val_keys_set:
            val_seq.append(seq)
        elif key in train_keys_set:
            train_seq.append(seq)

    print(f"\n  Sequences: train={len(train_seq)}, val={len(val_seq)}, test={len(test_seq)}")

    print("\n── STEP 7: NORMALIZE ─────────────────────────────────────────────")
    train_seq, val_seq, test_seq, y_stats, scaler, scaler_g = normalize_sequences(train_seq, val_seq, test_seq)

    # RSSI-only fine-tuning: zero the IMU/motion channels of the LABELED sequences AFTER the
    # scaler was fit on real IMU. The SSL/trajectory path (STEP 8.5) is NOT masked, so
    # pretraining still learns from all sensors while the labeled model uses RSSI only.
    if RSSI_ONLY_FINETUNE:
        n_masked = len(_non_rssi_indices())
        for _s in train_seq + val_seq + test_seq:
            mask_non_rssi(_s)
        print(f"  RSSI_ONLY_FINETUNE: zeroed {n_masked} IMU/motion feature channels "
              f"+ graph motion channel in labeled train/val/test sequences")

    print("\n── STEP 8: GENERATORS ────────────────────────────────────────────")
    train_gen = MultiTaskGenerator(train_seq, batch_size=BATCH_SIZE, shuffle=True, augment=True)
    val_gen   = MultiTaskGenerator(val_seq,   batch_size=BATCH_SIZE, shuffle=False, augment=False)

    (Bx_seq, Bx_graph), By = train_gen[0]
    print(f"  Batch: seq{Bx_seq.shape} | graph{Bx_graph.shape} | y{By['y_output'].shape} | fl{By['floor_output'].shape}")

    # ── STEP 8.5: SELF-SUPERVISED PRETRAINING (optional) ──────────────────────
    # Trajectory JSON -> shared preprocessing -> masked-reconstruction pretraining of the
    # encoder. Runs ONLY if PRETRAIN=True; the labeled-train scaler/medians are reused so
    # fine-tuning stays leakage-free. Encoder weights are written to ENCODER_WEIGHTS_PATH.
    if PRETRAIN:
        print("\n── STEP 8.5: SSL PRETRAINING ─────────────────────────────────────")
        pretrain_encoder(scaler, scaler_g, y_stats,
                         beacon_median_train, floor_median_train, sensor_medians)

    print("\n── STEP 9: BUILD MODEL ───────────────────────────────────────────")
    clear_stale_cache(N_FEATURES)
    tf.keras.backend.clear_session()

    model = build_hybrid_localizer(name='hybrid_gat')
    model.summary()

    # Transfer pretrained encoder weights BEFORE CustomModel loads any cached fine-tuned
    # checkpoint. Only 'enc_*' layers are copied; y/floor heads keep their fresh init.
    if PRETRAIN and os.path.exists(ENCODER_WEIGHTS_PATH):
        transfer_encoder_weights(model, ENCODER_WEIGHTS_PATH)

    custom_model = CustomModel(name='hybrid_gat', model=model)

    print("\n── STEP 10: TRAIN ────────────────────────────────────────────────")
    custom_model.fit(train_gen, val_gen, epochs=400)

    print("\n── STEP 11: EVALUATE ─────────────────────────────────────────────")
    if realtime_mode:
        result = evaluate_realtime(custom_model, test_seq, y_stats, train_seq, val_seq=val_seq)
    else:
        result = evaluate_model(custom_model, test_seq, y_stats, train_seq, val_seq=val_seq)

    print("\n" + "=" * 70)
    print("DONE")
    print(f"  Final Y MAE : {result['mae_final']:.3f} m")
    print(f"  Acc@1m      : {result['acc_1m']:.1f} %")
    print(f"  Acc@2m      : {result['acc_2m']:.1f} %")
    print(f"  Floor       : {result['floor_acc']:.1f} %")
    print("=" * 70)

    # -------------------------------------------------------------------------
    # Return training artifacts for real-time deployment
    # -------------------------------------------------------------------------
    artifacts = {
        'model': custom_model,
        'scaler': scaler,
        'scaler_g': scaler_g,
        'y_stats': y_stats,
        'beacon_median_map': beacon_median_train,
        'floor_median_map': floor_median_train,
        'sensor_medians': sensor_medians,
        'result': result,
    }
    return artifacts


# ==============================================================================
# DEMO / USAGE EXAMPLE
# ==============================================================================
if __name__ == '__main__':
    # --- Option A: Standard batch evaluation (RTS smoother, TTA, kNN) ---
    # artifacts = main(realtime_mode=False)

    # --- Option B: Real-time causal evaluation (fast, no future data) ---
    artifacts = main(realtime_mode=True)

    # --- Option C: Deploy RealTimeLocalizer for live streaming ---
    #
    # localizer = RealTimeLocalizer(
    #     model=artifacts['model'].model,
    #     scaler=artifacts['scaler'],
    #     scaler_g=artifacts['scaler_g'],
    #     y_stats=artifacts['y_stats'],
    #     beacon_median_map=artifacts['beacon_median_map'],
    #     floor_median_map=artifacts['floor_median_map'],
    #     sensor_medians=artifacts['sensor_medians'],
    #     use_tta=False,          # False for real-time speed
    #     use_knn=False,          # False for real-time speed
    #     kalman_q=0.5,           # tuned on val set
    #     kalman_r=4.0,           # tuned on val set
    #     outlier_z=3.0,
    #     outlier_window=10,
    # )
    #
    # # Simulate live window from test data
    # test_df = ...  # your live DataFrame window
    # y_smooth, floor = localizer.update(test_df)
    # print(f"Live position: y={y_smooth:.2f}m, floor={floor}")
    #
    # # Or use StreamingLocalizer for continuous feed
    # stream = StreamingLocalizer(localizer, window_size=WINDOW_SIZE, min_rows=4)
    # for obs in live_sensor_feed:
    #     stream.add_observation(**obs)
    #     if stream.obs_count % 5 == 0:
    #         y, fl = stream.predict()
    #         if y is not None:
    #             print(f"Stream: y={y:.2f}m, floor={fl}")


BEACON INDOOR LOCALIZATION — GRAPH + kNN + KALMAN PIPELINE
  [REAL-TIME MODE: causal Kalman + MAD outlier rejection]

── STEP 1: LOAD ──────────────────────────────────────────────────

  adhamFloor3.csv
  Loaded 18166 rows × 26 cols

  adhamFloor4.csv
  Loaded 12180 rows × 26 cols

  baselFloor3.csv
  Loaded 18082 rows × 26 cols

  baselFloor4.csv
  Loaded 15730 rows × 26 cols

  remonFloor3.csv
  Loaded 22020 rows × 26 cols

  remonFloor4.csv
  Loaded 21038 rows × 26 cols

COMBINED: 107216 rows

── STEP 2: MINIMAL CLEAN ─────────────────────────────────────────
  NUM_BEACONS = 7 (fixed)

── STEP 3: SPLIT ─────────────────────────────────────────────────
  Rows: train=74862, val=15793, test=16561

── STEP 4: TRAIN-ONLY STATISTICS ─────────────────────────────────
  Beacon medians: 6, Floor-beacon medians: 12

── STEP 5: PREPROCESS ────────────────────────────────────────────

── STEP 6: ENGINEER FEATURES ─────────────────────────────────────

  Final feature set (41): ['beaconId', 'r

Model: "hybrid_gat"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 10, 41)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 10)        │          0 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_t_beacon_emb    │ (None, 10, 16)    │        112 │ lambda[0][0]      │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 10, 40)    │          0 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 10, 56)    │          0 │ enc_t_beacon_emb… │
│ (Concatenate)       │                   │            │ lambda_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_t_proj (Dense)  │ (None, 10, 128)   │      7,296 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d   │ (None, 10, 128)   │          0 │ enc_t_proj[0][0]  │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_t_mha_0         │ (None, 10, 128)   │     66,048 │ spatial_dropout1… │
│ (MultiHeadAttentio… │                   │            │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 10, 128)   │          0 │ spatial_dropout1… │
│                     │                   │            │ enc_t_mha_0[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_t_ln_a_0        │ (None, 10, 128)   │        256 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_t_ffn0_0        │ (None, 10, 256)   │     33,024 │ enc_t_ln_a_0[0][… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 10, 256)   │          0 │ enc_t_ffn0_0[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_t_ffn1_0        │ (None, 10, 128)   │     32,896 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 10, 128)   │          0 │ enc_t_ln_a_0[0][… │
│                     │                   │            │ enc_t_ffn1_0[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_t_ln_b_0        │ (None, 10, 128)   │        256 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_t_mha_1         │ (None, 10, 128)   │     66,048 │ enc_t_ln_b_0[0][… │
│ (MultiHeadAttentio… │                   │            │ enc_t_ln_b_0[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 10, 128)   │          0 │ enc_t_ln_b_0[0][… │
│                     │                   │            │ enc_t_mha_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_t_ln_a_1        │ (None, 10, 128)   │        256 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                 

 Total params: 822,946 (3.14 MB)

 Trainable params: 821,410 (3.13 MB)

 Non-trainable params: 1,536 (6.00 KB)

  Transferred 38 pretrained encoder layers (by name) into the supervised model

── STEP 10: TRAIN ────────────────────────────────────────────────
Epoch 1/400
158/158 ━━━━━━━━━━━━━━━━━━━━ 86s 232ms/step - floor_output_accuracy: 0.5060 - floor_output_loss: 0.7983 - loss: 0.5279 - y_output_loss: 0.4879 - y_output_mae_euclidean: 0.9359 - val_floor_output_accuracy: 0.6242 - val_floor_output_loss: 0.6582 - val_loss: 0.2360 - val_y_output_loss: 0.2018 - val_y_output_mae_euclidean: 0.5407 - learning_rate: 1.5000e-05
Epoch 2/400
158/158 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - floor_output_accuracy: 0.6712 - floor_output_loss: 0.6153 - loss: 0.3916 - y_output_loss: 0.3609 - y_output_mae_euclidean: 0.7682 - val_floor_output_accuracy: 0.8424 - val_floor_output_loss: 0.4508 - val_loss: 0.1996 - val_y_output_loss: 0.1773 - val_y_output_mae_euclidean: 0.5000 - learning_rate: 3.0000e-05
Epoch 3/400
158/158 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - floor_output_accuracy: 0.8397 - floor_output_loss: 0.4174 - los